# 🏥 The Diagnostic Odyssey Ender
### Gemma 4 Good Hackathon — Health & Sciences Track

**What this does:** Helps rare disease patients find their next diagnostic step by cross-referencing symptoms against 50 rare diseases using Semantic RAG + Gemma 4 multimodal.

**Run cells in order from top to bottom. Use A100 GPU for best performance.**

---
| Cell | Purpose | Time |
|------|---------|------|
| 1 | Install all dependencies | ~2 min |
| 2 | Load Gemma 4 (4-bit) | ~5 min |
| 3 | 50-disease database | instant |
| 4 | Semantic RAG (FAISS) | ~1 min |
| 5 | Prompt + Diagnostic engine | instant |
| 6 | ⭐ Unsloth fine-tuning (optional, A100) | ~45 min |
| 7 | Upload UI file | instant |
| 8 | FastAPI server + live tunnel | ~1 min |

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Install all dependencies
# ════════════════════════════════════════════════════════════════

!pip install -U transformers accelerate bitsandbytes -q
!pip install sentence-transformers faiss-cpu -q
!pip install fastapi uvicorn nest-asyncio pdfminer.six python-multipart -q

import torch
print("✅ All packages installed!")
print(f"   GPU : {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Load Gemma 4 E4B with 4-bit quantization
# ════════════════════════════════════════════════════════════════

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, gc

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "google/gemma-4-E4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("⏳ Loading Gemma 4 with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

print(f"\n✅ Gemma 4 E4B loaded and ready!")
print(f"   Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Rare Disease Database (50 conditions)
# ════════════════════════════════════════════════════════════════

DISEASE_DB = [
    # ── CONNECTIVE TISSUE ────────────────────────────────────────
    {"name":"Ehlers-Danlos Syndrome (hEDS)","category":"Connective Tissue",
     "key_symptoms":["joint hypermobility","chronic pain","skin hyperextensibility","easy bruising","fatigue","frequent dislocations","POTS"],
     "typical_misdiagnoses":["fibromyalgia","anxiety","growing pains","hypochondria"],
     "diagnostic_tests":["Beighton score assessment","echocardiogram","skin biopsy","genetic testing"],
     "specialist":"Clinical Geneticist or Rheumatologist","avg_diagnosis_time":"10-12 years"},

    {"name":"Marfan Syndrome","category":"Connective Tissue/Genetic",
     "key_symptoms":["tall thin stature","long arms and fingers","aortic aneurysm","lens dislocation","scoliosis","flat feet"],
     "typical_misdiagnoses":["normal tall stature","musculoskeletal pain","scoliosis only"],
     "diagnostic_tests":["echocardiogram","FBN1 gene testing","slit-lamp eye exam","chest MRI","Ghent criteria"],
     "specialist":"Clinical Geneticist, Cardiologist","avg_diagnosis_time":"3-5 years"},

    {"name":"Loeys-Dietz Syndrome","category":"Connective Tissue/Genetic",
     "key_symptoms":["aortic aneurysm","arterial tortuosity","hypertelorism (wide-set eyes)","cleft palate","joint laxity"],
     "typical_misdiagnoses":["Marfan syndrome","EDS","normal tall stature"],
     "diagnostic_tests":["CT/MR angiography full body","TGFBR1/TGFBR2 gene testing","echocardiogram"],
     "specialist":"Clinical Geneticist, Cardiologist","avg_diagnosis_time":"3-5 years"},

    # ── AUTOIMMUNE ───────────────────────────────────────────────
    {"name":"Lupus (SLE)","category":"Autoimmune",
     "key_symptoms":["butterfly facial rash","joint pain","extreme fatigue","hair loss","photosensitivity","kidney problems","oral ulcers"],
     "typical_misdiagnoses":["rheumatoid arthritis","fibromyalgia","depression","viral infection"],
     "diagnostic_tests":["ANA test","anti-dsDNA antibodies","complement C3/C4 levels","CBC","urinalysis"],
     "specialist":"Rheumatologist","avg_diagnosis_time":"6 years"},

    {"name":"Sjögren's Syndrome","category":"Autoimmune",
     "key_symptoms":["chronic dry eyes","chronic dry mouth","fatigue","joint pain","peripheral neuropathy","brain fog"],
     "typical_misdiagnoses":["depression","fibromyalgia","anxiety","dental problems only"],
     "diagnostic_tests":["anti-SSA/Ro antibodies","anti-SSB/La antibodies","Schirmer's test","lip biopsy"],
     "specialist":"Rheumatologist, Ophthalmologist","avg_diagnosis_time":"5-7 years"},

    {"name":"Antiphospholipid Syndrome (APS)","category":"Autoimmune",
     "key_symptoms":["recurrent miscarriages","blood clots in young person","stroke in young person","mottled skin","thrombocytopenia"],
     "typical_misdiagnoses":["idiopathic clotting","unexplained pregnancy loss","lupus alone"],
     "diagnostic_tests":["anticardiolipin antibodies","lupus anticoagulant","anti-beta2 glycoprotein I"],
     "specialist":"Rheumatologist, Hematologist","avg_diagnosis_time":"3-5 years"},

    {"name":"Systemic Sclerosis (Scleroderma)","category":"Autoimmune/Connective Tissue",
     "key_symptoms":["skin hardening/thickening","Raynaud phenomenon","reflux/swallowing problems","lung fibrosis","kidney crisis"],
     "typical_misdiagnoses":["Raynaud's only","GERD","fibromyalgia","lupus"],
     "diagnostic_tests":["ANA speckled pattern","anti-SCL-70 antibodies","anti-centromere antibodies","HRCT chest","echocardiogram"],
     "specialist":"Rheumatologist","avg_diagnosis_time":"3-5 years"},

    {"name":"Dermatomyositis / Polymyositis","category":"Autoimmune/Neuromuscular",
     "key_symptoms":["proximal muscle weakness","heliotrope rash (purple eyelids)","Gottron papules on knuckles","swallowing difficulty"],
     "typical_misdiagnoses":["fibromyalgia","statin side effects","MS","muscular dystrophy"],
     "diagnostic_tests":["CK level (very high)","myositis antibodies (anti-Jo-1, anti-MDA5)","MRI muscle","EMG","muscle biopsy"],
     "specialist":"Rheumatologist, Neurologist","avg_diagnosis_time":"2-4 years"},

    {"name":"Ankylosing Spondylitis","category":"Autoimmune/Inflammatory",
     "key_symptoms":["chronic low back pain in young adults","morning stiffness over 1hr","sacroiliac pain","uveitis","improves with exercise"],
     "typical_misdiagnoses":["mechanical back pain","slipped disc","sports injury","fibromyalgia"],
     "diagnostic_tests":["HLA-B27 testing","MRI sacroiliac joints","X-ray pelvis/spine","CRP/ESR"],
     "specialist":"Rheumatologist","avg_diagnosis_time":"8-10 years"},

    {"name":"Behçet's Disease","category":"Autoimmune/Vasculitis",
     "key_symptoms":["recurrent mouth ulcers","genital ulcers","eye inflammation","skin lesions","joint pain","blood clots"],
     "typical_misdiagnoses":["Crohn's disease","herpes","reactive arthritis","IBD"],
     "diagnostic_tests":["pathergy test","HLA-B51","ophthalmology exam","colonoscopy"],
     "specialist":"Rheumatologist, Ophthalmologist","avg_diagnosis_time":"5-10 years"},

    {"name":"Adult-onset Still's Disease (AOSD)","category":"Autoimmune/Inflammatory",
     "key_symptoms":["daily high spiking fevers","salmon-pink rash during fever","arthritis","sore throat","enlarged lymph nodes"],
     "typical_misdiagnoses":["infection","lymphoma","viral illness","rheumatoid arthritis"],
     "diagnostic_tests":["serum ferritin (extremely high)","CBC","LFTs","bone marrow biopsy","full body PET scan"],
     "specialist":"Rheumatologist, Hematologist","avg_diagnosis_time":"3-6 months"},

    {"name":"IgG4-Related Disease","category":"Autoimmune/Inflammatory",
     "key_symptoms":["swelling of organs/glands","autoimmune pancreatitis","salivary gland swelling","kidney involvement","orbital swelling"],
     "typical_misdiagnoses":["cancer","lymphoma","Sjögren's syndrome","sarcoidosis"],
     "diagnostic_tests":["serum IgG4 level","biopsy with IgG4 staining","CT/MRI of affected organs","PET scan"],
     "specialist":"Rheumatologist, Pathologist","avg_diagnosis_time":"2-5 years"},

    {"name":"Relapsing Polychondritis","category":"Autoimmune",
     "key_symptoms":["red painful ear (not earlobe)","nasal bridge collapse","saddle nose deformity","tracheal problems","joint pain"],
     "typical_misdiagnoses":["infection","gout","vasculitis","ear infection"],
     "diagnostic_tests":["clinical diagnosis","biopsy of cartilage","CT chest (trachea)","pulmonary function tests"],
     "specialist":"Rheumatologist, ENT","avg_diagnosis_time":"3-6 years"},

    # ── NEUROLOGICAL ─────────────────────────────────────────────
    {"name":"Stiff Person Syndrome","category":"Neurological/Autoimmune",
     "key_symptoms":["progressive muscle stiffness","painful muscle spasms","triggered by noise or touch","difficulty walking","falls"],
     "typical_misdiagnoses":["Parkinson's disease","MS","anxiety disorder","psychiatric condition"],
     "diagnostic_tests":["anti-GAD65 antibodies","EMG","MRI spine","lumbar puncture"],
     "specialist":"Neurologist, Neuroimmunologist","avg_diagnosis_time":"7-8 years"},

    {"name":"Myasthenia Gravis","category":"Neuromuscular",
     "key_symptoms":["drooping eyelids","double vision","muscle weakness worsening with activity","difficulty swallowing","breathing problems"],
     "typical_misdiagnoses":["stroke","MS","depression","conversion disorder"],
     "diagnostic_tests":["AChR antibodies","MuSK antibodies","edrophonium test","EMG","CT chest for thymoma"],
     "specialist":"Neurologist","avg_diagnosis_time":"1-3 years"},

    {"name":"Autoimmune Encephalitis","category":"Neurological/Autoimmune",
     "key_symptoms":["rapid psychiatric symptoms","seizures","memory loss","movement disorders","altered consciousness"],
     "typical_misdiagnoses":["psychiatric illness","schizophrenia","drug-induced psychosis","viral encephalitis"],
     "diagnostic_tests":["NMDA-R antibodies serum and CSF","VGKC antibodies","MRI brain","EEG","lumbar puncture"],
     "specialist":"Neurologist, Neuroimmunologist","avg_diagnosis_time":"months to years if missed"},

    {"name":"Neuromyelitis Optica (NMOSD)","category":"Neurological/Autoimmune",
     "key_symptoms":["severe optic neuritis","transverse myelitis","vision loss","paralysis","attacks worse than MS"],
     "typical_misdiagnoses":["multiple sclerosis","optic neuritis alone","spinal cord injury"],
     "diagnostic_tests":["AQP4-IgG antibodies","MOG-IgG antibodies","MRI brain and spine","visual evoked potentials"],
     "specialist":"Neurologist (MS/Neuroimmunology)","avg_diagnosis_time":"2-4 years"},

    {"name":"Chiari Malformation","category":"Neurological/Structural",
     "key_symptoms":["headache at base of skull worsening with cough or sneeze","neck pain","balance problems","hand tingling","swallowing difficulty"],
     "typical_misdiagnoses":["migraine","MS","anxiety","fibromyalgia"],
     "diagnostic_tests":["MRI brain and cervical spine sagittal view","cine MRI CSF flow study"],
     "specialist":"Neurosurgeon, Neurologist","avg_diagnosis_time":"5-8 years"},

    {"name":"Complex Regional Pain Syndrome (CRPS)","category":"Neurological/Pain",
     "key_symptoms":["severe burning pain","skin color and temperature changes","allodynia (pain from light touch)","swelling","shiny skin after minor injury"],
     "typical_misdiagnoses":["psychosomatic pain","anxiety","malingering","arthritis"],
     "diagnostic_tests":["clinical diagnosis Budapest criteria","bone scan","MRI","thermography"],
     "specialist":"Pain Specialist, Neurologist","avg_diagnosis_time":"2-4 years"},

    {"name":"Kleine-Levin Syndrome (KLS)","category":"Neurological",
     "key_symptoms":["recurring episodes of excessive sleep 18-20hrs per day","cognitive impairment during episodes","hyperphagia","episodes last days to weeks","completely normal between episodes"],
     "typical_misdiagnoses":["depression","psychiatric illness","drug use","encephalitis"],
     "diagnostic_tests":["clinical diagnosis","MRI brain","EEG","sleep study"],
     "specialist":"Neurologist, Sleep Specialist","avg_diagnosis_time":"3-7 years"},

    # ── DYSAUTONOMIA ─────────────────────────────────────────────
    {"name":"POTS (Postural Orthostatic Tachycardia Syndrome)","category":"Dysautonomia",
     "key_symptoms":["heart racing on standing","dizziness","fainting","brain fog","fatigue","nausea when upright"],
     "typical_misdiagnoses":["anxiety","panic disorder","dehydration","depression"],
     "diagnostic_tests":["tilt table test","10-minute stand test","24hr Holter monitor","autonomic function testing"],
     "specialist":"Cardiologist, Autonomic Neurologist","avg_diagnosis_time":"4-6 years"},

    {"name":"Orthostatic Hypotension / Pure Autonomic Failure","category":"Dysautonomia",
     "key_symptoms":["blood pressure drop over 20mmHg on standing","dizziness","fainting","fatigue","cognitive slowing when upright"],
     "typical_misdiagnoses":["dehydration","anxiety","medication side effects","normal aging"],
     "diagnostic_tests":["lying and standing BP measurement","tilt table test","autonomic function battery","plasma norepinephrine levels"],
     "specialist":"Autonomic Neurologist, Cardiologist","avg_diagnosis_time":"2-5 years"},

    # ── IMMUNOLOGICAL ────────────────────────────────────────────
    {"name":"Mast Cell Activation Syndrome (MCAS)","category":"Immunological",
     "key_symptoms":["anaphylaxis-like episodes","hives","flushing","GI problems","brain fog","reactions to foods/chemicals"],
     "typical_misdiagnoses":["IBS","anxiety","allergy","panic attacks"],
     "diagnostic_tests":["serum tryptase during reaction","24hr urine histamine","prostaglandin D2","bone marrow biopsy"],
     "specialist":"Allergist/Immunologist, Hematologist","avg_diagnosis_time":"6-8 years"},

    {"name":"Hereditary Angioedema (HAE)","category":"Immunological/Genetic",
     "key_symptoms":["recurrent non-itchy swelling of face lips throat abdomen","abdominal pain attacks","family history","no response to antihistamines"],
     "typical_misdiagnoses":["allergy/anaphylaxis","IBS","appendicitis"],
     "diagnostic_tests":["C1-inhibitor level and function","C4 level (low)","genetic testing SERPING1"],
     "specialist":"Allergist/Immunologist","avg_diagnosis_time":"8-10 years"},

    {"name":"Common Variable Immunodeficiency (CVID)","category":"Primary Immunodeficiency",
     "key_symptoms":["recurrent bacterial infections lungs and sinuses","low antibody levels","GI problems","autoimmune features"],
     "typical_misdiagnoses":["bad luck with infections","IBS","asthma"],
     "diagnostic_tests":["immunoglobulin levels IgG IgA IgM","specific antibody responses to vaccines","B-cell phenotyping"],
     "specialist":"Immunologist","avg_diagnosis_time":"5-7 years"},

    # ── HEMATOLOGICAL ────────────────────────────────────────────
    {"name":"Systemic Mastocytosis","category":"Hematological",
     "key_symptoms":["skin lesions (urticaria pigmentosa)","anaphylaxis","bone pain","GI problems","flushing episodes"],
     "typical_misdiagnoses":["allergy","IBS","skin conditions","carcinoid syndrome"],
     "diagnostic_tests":["serum tryptase","bone marrow biopsy","KIT D816V mutation","skin biopsy"],
     "specialist":"Hematologist, Allergist","avg_diagnosis_time":"5-10 years"},

    {"name":"Paroxysmal Nocturnal Hemoglobinuria (PNH)","category":"Hematological",
     "key_symptoms":["dark urine especially in morning","blood clots in unusual sites","severe fatigue","anemia","abdominal pain"],
     "typical_misdiagnoses":["hemolytic anemia","aplastic anemia","idiopathic thrombosis"],
     "diagnostic_tests":["flow cytometry GPI-anchored proteins","LDH level","reticulocyte count","bone marrow biopsy"],
     "specialist":"Hematologist","avg_diagnosis_time":"1-5 years"},

    # ── METABOLIC / GENETIC ──────────────────────────────────────
    {"name":"Wilson's Disease","category":"Metabolic/Genetic",
     "key_symptoms":["liver disease in young person","psychiatric symptoms","tremor","Kayser-Fleischer rings in eyes","dysarthria"],
     "typical_misdiagnoses":["hepatitis","psychiatric disorder","Parkinson's disease"],
     "diagnostic_tests":["serum ceruloplasmin","24hr urine copper","liver biopsy","slit-lamp eye exam","ATP7B gene testing"],
     "specialist":"Hepatologist, Neurologist, Clinical Geneticist","avg_diagnosis_time":"1-3 years"},

    {"name":"Hemochromatosis","category":"Metabolic/Genetic",
     "key_symptoms":["chronic fatigue","joint pain especially knuckles","liver disease","diabetes","bronze skin","heart problems"],
     "typical_misdiagnoses":["arthritis","liver disease of unknown cause","type 2 diabetes"],
     "diagnostic_tests":["serum ferritin","transferrin saturation","HFE gene mutation test","liver biopsy"],
     "specialist":"Hematologist, Hepatologist, Geneticist","avg_diagnosis_time":"5-10 years"},

    {"name":"Acute Intermittent Porphyria","category":"Metabolic/Genetic",
     "key_symptoms":["severe abdominal pain attacks","nausea and vomiting","confusion","dark urine","psychiatric symptoms","seizures"],
     "typical_misdiagnoses":["appendicitis","IBS","psychiatric disorder","MS"],
     "diagnostic_tests":["urine porphobilinogen during attack","ALA levels","HMBS gene testing"],
     "specialist":"Hematologist, Metabolic Physician","avg_diagnosis_time":"15 years"},

    {"name":"Fabry Disease","category":"Metabolic/Genetic (Lysosomal)",
     "key_symptoms":["burning pain in hands and feet","heat/cold intolerance","kidney disease","heart disease in young person","small red skin spots"],
     "typical_misdiagnoses":["MS","growing pains","erythromelalgia","anxiety"],
     "diagnostic_tests":["alpha-galactosidase A enzyme activity","GLA gene testing","plasma lyso-Gb3"],
     "specialist":"Clinical Geneticist, Nephrologist, Cardiologist","avg_diagnosis_time":"10-15 years"},

    {"name":"Gaucher Disease","category":"Metabolic/Genetic (Lysosomal)",
     "key_symptoms":["enlarged spleen and liver","bone pain and fractures","fatigue","easy bruising","low blood counts"],
     "typical_misdiagnoses":["leukemia","lymphoma","idiopathic thrombocytopenia"],
     "diagnostic_tests":["beta-glucocerebrosidase enzyme activity","GBA gene testing","bone marrow biopsy"],
     "specialist":"Hematologist, Clinical Geneticist","avg_diagnosis_time":"3-5 years"},

    {"name":"Pompe Disease","category":"Metabolic/Genetic",
     "key_symptoms":["progressive proximal muscle weakness","respiratory insufficiency","difficulty breathing lying flat","exercise intolerance"],
     "typical_misdiagnoses":["muscular dystrophy","polymyositis","limb girdle muscular dystrophy"],
     "diagnostic_tests":["acid alpha-glucosidase GAA enzyme activity","GAA gene testing","CK level","muscle biopsy"],
     "specialist":"Neurologist, Clinical Geneticist, Pulmonologist","avg_diagnosis_time":"5-10 years"},

    # ── ENDOCRINE ────────────────────────────────────────────────
    {"name":"Addison's Disease","category":"Endocrine",
     "key_symptoms":["extreme fatigue","weight loss","low blood pressure","salt craving","skin darkening","nausea"],
     "typical_misdiagnoses":["depression","anorexia","chronic fatigue","GI problems"],
     "diagnostic_tests":["morning cortisol test","ACTH stimulation test","ACTH plasma level","adrenal antibodies","CT adrenal glands"],
     "specialist":"Endocrinologist","avg_diagnosis_time":"2-3 years"},

    {"name":"Cushing's Syndrome","category":"Endocrine",
     "key_symptoms":["weight gain centrally","moon face","buffalo hump","purple stretch marks","muscle weakness","easy bruising"],
     "typical_misdiagnoses":["obesity","metabolic syndrome","depression","PCOS"],
     "diagnostic_tests":["24hr urinary free cortisol","late-night salivary cortisol","1mg dexamethasone suppression","MRI pituitary"],
     "specialist":"Endocrinologist","avg_diagnosis_time":"3-6 years"},

    {"name":"Primary Hyperaldosteronism (Conn's)","category":"Endocrine",
     "key_symptoms":["resistant hypertension","low potassium","muscle weakness and cramps","headaches","high blood pressure in young person"],
     "typical_misdiagnoses":["essential hypertension","anxiety","idiopathic hypokalemia"],
     "diagnostic_tests":["aldosterone-renin ratio","24hr urine aldosterone","CT adrenal","adrenal vein sampling"],
     "specialist":"Endocrinologist","avg_diagnosis_time":"5-8 years"},

    # ── GASTROINTESTINAL ─────────────────────────────────────────
    {"name":"Gastroparesis","category":"Gastrointestinal",
     "key_symptoms":["nausea and vomiting after eating","feeling full after small amounts","bloating","weight loss","upper abdominal pain"],
     "typical_misdiagnoses":["IBS","anxiety","eating disorder","GERD","functional dyspepsia"],
     "diagnostic_tests":["gastric emptying scintigraphy 4hr","gastric emptying breath test","endoscopy"],
     "specialist":"Gastroenterologist (motility specialist)","avg_diagnosis_time":"3-5 years"},

    {"name":"Eosinophilic Esophagitis (EoE)","category":"Gastrointestinal/Immunological",
     "key_symptoms":["difficulty swallowing","food getting stuck in throat","chest pain not cardiac","heartburn not responding to PPIs","food impaction"],
     "typical_misdiagnoses":["GERD/acid reflux","esophageal spasm","anxiety"],
     "diagnostic_tests":["upper endoscopy with biopsies","esophageal eosinophil count over 15 per hpf","allergy testing"],
     "specialist":"Gastroenterologist, Allergist","avg_diagnosis_time":"4-6 years"},

    {"name":"Celiac Disease","category":"Gastrointestinal/Autoimmune",
     "key_symptoms":["GI symptoms with gluten","fatigue","anemia","malabsorption","bone density loss","skin rash"],
     "typical_misdiagnoses":["IBS","Crohn's disease","depression","anxiety","fibromyalgia"],
     "diagnostic_tests":["anti-tTG IgA antibodies","total IgA level","duodenal biopsy","HLA-DQ2/DQ8"],
     "specialist":"Gastroenterologist","avg_diagnosis_time":"6-10 years"},

    {"name":"Median Arcuate Ligament Syndrome (MALS)","category":"Vascular/Gastrointestinal",
     "key_symptoms":["severe abdominal pain after eating","fear of eating","significant weight loss","abdominal bruit","nausea"],
     "typical_misdiagnoses":["eating disorder","IBS","anxiety","functional abdominal pain"],
     "diagnostic_tests":["Doppler ultrasound celiac artery","CT angiography","mesenteric angiography"],
     "specialist":"Vascular Surgeon, Gastroenterologist","avg_diagnosis_time":"5-10 years"},

    {"name":"Primary Sclerosing Cholangitis (PSC)","category":"Hepatobiliary",
     "key_symptoms":["jaundice","itching","fatigue","right upper abdominal pain","recurrent cholangitis","associated with IBD"],
     "typical_misdiagnoses":["gallstones","hepatitis","IBD alone"],
     "diagnostic_tests":["MRCP biliary imaging","liver biopsy","ALP/GGT levels","p-ANCA","colonoscopy for IBD"],
     "specialist":"Hepatologist, Gastroenterologist","avg_diagnosis_time":"2-4 years"},

    # ── NEUROIMMUNE ──────────────────────────────────────────────
    {"name":"ME/CFS (Myalgic Encephalomyelitis)","category":"Neuroimmune",
     "key_symptoms":["post-exertional malaise","debilitating fatigue","brain fog","unrefreshing sleep","orthostatic intolerance"],
     "typical_misdiagnoses":["depression","anxiety","deconditioning","malingering"],
     "diagnostic_tests":["rule out thyroid disease","rule out anemia","tilt table test","sleep study"],
     "specialist":"ME/CFS Specialist, Neurologist, Immunologist","avg_diagnosis_time":"5-7 years"},

    # ── VASCULITIS ───────────────────────────────────────────────
    {"name":"Takayasu's Arteritis","category":"Vasculitis",
     "key_symptoms":["absent or diminished pulse","blood pressure difference between arms","chronic fatigue","visual symptoms","young Asian women"],
     "typical_misdiagnoses":["hypertension","MS","anxiety","fibromyalgia"],
     "diagnostic_tests":["CT/MR angiography","PET scan","ESR/CRP","angiography","echocardiogram"],
     "specialist":"Rheumatologist, Vascular Surgeon","avg_diagnosis_time":"3-7 years"},

    {"name":"ANCA-associated Vasculitis (Wegener's/MPA)","category":"Vasculitis/Autoimmune",
     "key_symptoms":["chronic sinusitis and nosebleeds","coughing blood","blood in urine","kidney failure","peripheral neuropathy"],
     "typical_misdiagnoses":["recurrent sinusitis","asthma","glomerulonephritis of unknown cause","TB"],
     "diagnostic_tests":["c-ANCA/PR3 antibodies","p-ANCA/MPO antibodies","urinalysis red cell casts","chest CT","kidney biopsy"],
     "specialist":"Rheumatologist, Nephrologist","avg_diagnosis_time":"1-3 years"},

    # ── INFLAMMATORY ─────────────────────────────────────────────
    {"name":"Sarcoidosis","category":"Inflammatory",
     "key_symptoms":["persistent cough","shortness of breath","skin lesions","swollen lymph nodes","eye inflammation","fatigue"],
     "typical_misdiagnoses":["tuberculosis","lymphoma","asthma","lupus"],
     "diagnostic_tests":["chest X-ray","CT scan","tissue biopsy","serum ACE level","bronchoalveolar lavage"],
     "specialist":"Pulmonologist, Rheumatologist","avg_diagnosis_time":"2-5 years"},

    # ── CARDIOVASCULAR ───────────────────────────────────────────
    {"name":"Pulmonary Arterial Hypertension (PAH)","category":"Cardiovascular/Pulmonary",
     "key_symptoms":["progressive shortness of breath on exertion","fatigue","fainting","leg swelling","chest pain"],
     "typical_misdiagnoses":["asthma","anxiety","deconditioning","COPD"],
     "diagnostic_tests":["echocardiogram","right heart catheterization (gold standard)","CT pulmonary angiography","pulmonary function tests"],
     "specialist":"Pulmonologist, Cardiologist (PAH specialist)","avg_diagnosis_time":"2-3 years"},

    # ── CHRONIC PAIN ─────────────────────────────────────────────
    {"name":"Fibromyalgia","category":"Chronic Pain/Central Sensitization",
     "key_symptoms":["widespread musculoskeletal pain","fatigue","sleep problems","brain fog","heightened sensitivity to pain"],
     "typical_misdiagnoses":["malingering","psychosomatic","depression","arthritis"],
     "diagnostic_tests":["clinical diagnosis 2016 criteria","rule out inflammatory arthritis","thyroid function","sleep study"],
     "specialist":"Rheumatologist, Pain Specialist","avg_diagnosis_time":"5 years"},
]

print(f"✅ Disease database loaded: {len(DISEASE_DB)} rare conditions")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3b — Disease Database Expansion: 600+ ORPHA Diseases
# Run AFTER Cell 3 (which defines the original 50-disease DISEASE_DB)
# This cell adds 600 real diseases from ORPHANET/HPO database
# Total after merge: 650+ unique rare diseases
# ════════════════════════════════════════════════════════════════

ORPHA_DISEASES = [
    {"name": "Giant cell arteritis", "category": "rare disease", "symptoms": ["headache", "fatigue", "visual impairment", "anemia", "renal insufficiency", "hearing loss", "ptosis", "nystagmus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Huntington disease", "category": "rare disease", "symptoms": ["seizures", "dystonia", "rigidity", "abnormal cerebral white matter", "basal ganglia abnormality", "Babinski sign"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Homocystinuria due to cystathionine beta-synthase deficiency", "category": "rare disease", "symptoms": ["intellectual disability", "kyphosis", "osteosarcoma", "arteriovenous malformation", "strabismus", "optic atrophy", "autistic behavior", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Homocystinuria due to methylene tetrahydrofolate reductase deficiency", "category": "rare disease", "symptoms": ["seizures", "ataxia", "peripheral neuropathy", "intellectual disability", "global developmental delay", "failure to thrive", "headache", "abnormal cerebral white matter"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Holt-Oram syndrome", "category": "rare disease", "symptoms": ["ventricular septal defect", "atrial septal defect", "kyphosis", "patent ductus arteriosus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "15q13.3 microdeletion syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "global developmental delay", "microcephaly", "strabismus", "downslanted palpebral fissures", "seizures", "hypotonia", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Classic Hodgkin lymphoma", "category": "hematology", "symptoms": ["lymphoma", "fatigue", "back pain", "ataxia", "eosinophilia", "anemia", "leukocytosis", "respiratory insufficiency"], "tests": ["CBC with differential", "peripheral blood smear", "bone marrow biopsy", "flow cytometry"], "specialist": "hematologist", "relevance_score": 0},
    {"name": "Silver-Russell syndrome due to 7p11.2p13 microduplication", "category": "rare disease", "symptoms": ["delayed skeletal maturation", "growth delay", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Charlie M syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "wide nasal bridge"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Olivopontocerebellar atrophy-deafness syndrome", "category": "otolaryngology", "symptoms": ["hearing loss", "strabismus", "chorioretinal coloboma", "nystagmus", "optic atrophy", "seizures", "ataxia", "cerebral cortical atrophy"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Spastic paraplegia type 7", "category": "rare disease", "symptoms": ["nystagmus", "optic atrophy", "confusion", "somatic sensory dysfunction", "Babinski sign", "attention deficit hyperactivity disorder", "cerebral cortical atrophy", "abnormal cerebral white matter"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Severe generalized junctional epidermolysis bullosa", "category": "dermatology", "symptoms": ["failure to thrive", "growth retardation", "anemia", "renal cyst", "hydronephrosis", "squamous cell carcinoma", "seizures"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "X-linked Charcot-Marie-Tooth disease type 5", "category": "rare disease", "symptoms": ["hearing loss", "optic atrophy", "ataxia", "muscle weakness", "tremor", "kyphosis", "peripheral neuropathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Langerhans cell histiocytosis", "category": "rare disease", "symptoms": ["osteolysis", "hepatomegaly", "hearing loss", "ataxia", "growth retardation", "thrombocytopenia", "respiratory insufficiency"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Progressive microcephaly-seizures-cortical blindness-developmental delay syndrome", "category": "rare disease", "symptoms": ["seizures", "global developmental delay", "susceptibility to infections", "short stature", "optic atrophy", "hypotonia", "bronchiectasis", "lymphoma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hirschsprung disease", "category": "rare disease", "symptoms": ["short stature", "intestinal malrotation", "abdominal distension", "growth retardation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Spastic paraplegia type 2", "category": "rare disease", "symptoms": ["nystagmus", "optic atrophy", "intellectual disability", "ataxia", "spasticity", "muscle weakness", "Babinski sign"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Silver-Russell syndrome due to an imprinting defect of 11p15", "category": "rare disease", "symptoms": ["delayed skeletal maturation", "short stature", "growth delay", "intellectual disability", "seizures", "motor delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Silver-Russell syndrome due to maternal uniparental disomy of chromosome 11", "category": "rare disease", "symptoms": ["growth delay", "global developmental delay", "premature birth", "abnormal heart morphology", "delayed speech and language"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Guanidinoacetate methyltransferase deficiency", "category": "rare disease", "symptoms": ["seizures", "intellectual disability severe", "severe global developmental delay", "ataxia", "dystonia", "generalized myoclonic seizures", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "PMP22-RAI1 contiguous gene duplication syndrome", "category": "rare disease", "symptoms": ["long philtrum", "strabismus", "downslanted palpebral fissures", "ventricular septal defect", "atrial septal defect", "bicuspid aortic valve", "patent foramen ovale", "delayed speech"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive generalized dystrophic epidermolysis bullosa, severe form", "category": "dermatology", "symptoms": ["growth retardation", "anemia", "squamous cell carcinoma", "renal insufficiency", "dysphagia", "vitamin D deficiency", "urinary tract abnormality"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Recessive dystrophic epidermolysis bullosa inversa", "category": "dermatology", "symptoms": ["urinary tract abnormality", "growth retardation", "anemia"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Silver-Russell syndrome due to 11p15 microduplication", "category": "rare disease", "symptoms": ["failure to thrive", "delayed skeletal maturation", "growth delay", "short stature", "motor delay", "midface retrusion"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculocerebral hypopigmentation syndrome, Cross type", "category": "rare disease", "symptoms": ["cryptorchidism", "urinary tract abnormality", "microcephaly", "sensorineural hearing loss", "anteverted nares", "nystagmus", "intellectual disability", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Griscelli syndrome", "category": "rare disease", "symptoms": ["nystagmus", "intellectual disability", "seizures", "ataxia", "hypotonia", "spasticity", "global developmental delay", "thrombocytopenia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Greig cephalopolysyndactyly syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "wide nasal bridge", "seizures", "intellectual disability mild", "gastrointestinal bleeding"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Joubert syndrome with renal defect", "category": "nephrology", "symptoms": ["intellectual disability", "ataxia", "hypotonia", "global developmental delay", "nystagmus", "renal insufficiency", "prominent nasal bridge", "anteverted nares"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Progressive supranuclear palsy-predominant parkinsonism syndrome", "category": "neurology", "symptoms": ["rigidity", "tremor", "postural instability", "dystonia", "neurological speech impairment"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Joubert syndrome with ocular defect", "category": "rare disease", "symptoms": ["strabismus", "ptosis", "seizures", "tremor", "intestinal malrotation", "myopathy", "intellectual disability", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Blepharophimosis-intellectual disability syndrome, Ohdo type", "category": "rare disease", "symptoms": ["cryptorchidism", "proteinuria", "microcephaly", "hearing loss", "ptosis", "delayed speech", "intellectual disability mild", "motor delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculocerebral hypopigmentation syndrome, Preus type", "category": "rare disease", "symptoms": ["hearing loss", "nystagmus", "intellectual disability", "seizures", "ataxia", "global developmental delay", "hip abnormality", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Isolated thyroid-stimulating hormone deficiency", "category": "endocrinology", "symptoms": ["failure to thrive", "growth retardation", "delayed skeletal maturation", "hypercholesterolemia", "fatigue", "hypotonia", "attention deficit hyperactivity disorder"], "tests": ["hormonal panel", "thyroid function", "adrenal testing", "genetic testing"], "specialist": "endocrinologist", "relevance_score": 0},
    {"name": "Imerslund-Gräsbeck syndrome", "category": "rare disease", "symptoms": ["delayed speech", "hypotonia", "failure to thrive", "proteinuria", "neutropenia", "reticulocytosis", "developmental regression", "immunodeficiency"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hemangioblastoma", "category": "rare disease", "symptoms": ["headache", "vertigo", "distal muscle weakness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Drug-induced lupus erythematosus", "category": "rheumatology", "symptoms": ["anemia", "joint pain", "elevated creatinine kinase", "elevated creatine kinase", "autoimmune antibody", "constitutional symptom"], "tests": ["ANA panel", "RF/anti-CCP", "inflammatory markers (ESR/CRP)", "joint imaging"], "specialist": "rheumatologist", "relevance_score": 0},
    {"name": "Classic progressive supranuclear palsy syndrome", "category": "rare disease", "symptoms": ["postural instability", "tremor", "dystonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculofaciocardiodental syndrome", "category": "rare disease", "symptoms": ["long philtrum", "hearing loss", "sensorineural hearing loss", "prominent nasal bridge", "ptosis", "intellectual disability", "global developmental delay", "patent ductus arteriosus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculo-palato-cerebral syndrome", "category": "rare disease", "symptoms": ["asthma", "microcephaly", "intellectual disability", "spasticity", "global developmental delay", "short stature", "small hand"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Severe oculo-renal-cerebellar syndrome", "category": "nephrology", "symptoms": ["renal insufficiency", "proteinuria", "strabismus", "optic atrophy", "hypotonia", "visual impairment", "spasticity", "short stature"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Attenuated familial adenomatous polyposis", "category": "rare disease", "symptoms": ["multiple renal cysts", "pheochromocytoma", "breast carcinoma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive Charcot-Marie-Tooth disease with hoarseness", "category": "rare disease", "symptoms": ["skeletal muscle atrophy", "peripheral neuropathy", "motor delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculodentodigital dysplasia", "category": "rare disease", "symptoms": ["growth delay", "premature fusion of cranial sutures", "hypertelorism", "anteverted nares", "visual impairment", "optic atrophy", "seizures", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Multicentric osteolysis-nodulosis-arthropathy spectrum", "category": "rare disease", "symptoms": ["abnormal facial shape", "osteolysis", "joint hypermobility", "intellectual disability", "ventricular septal defect", "atrial septal defect", "bicuspid aortic valve"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cardiogenic shock", "category": "rare disease", "symptoms": ["abnormal ECG", "arrhythmia", "hepatomegaly", "vertigo"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Bohring-Opitz syndrome", "category": "rare disease", "symptoms": ["microcephaly", "proptosis", "optic atrophy", "sleep disturbance", "basal ganglia abnormality", "susceptibility to infections", "short stature", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "FKRP-related limb-girdle muscular dystrophy R9", "category": "neurology", "symptoms": ["elevated creatine kinase", "muscular dystrophy", "hypotonia", "motor delay"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "PURA-related severe neonatal hypotonia-seizures-encephalopathy syndrome", "category": "neurology", "symptoms": ["intellectual disability", "seizures", "motor delay", "dystonia", "drooling", "basal ganglia abnormality", "vitamin D deficiency", "strabismus"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Stickler syndrome type 1", "category": "rare disease", "symptoms": ["long philtrum", "sensorineural hearing loss", "proptosis", "osteoarthritis", "joint pain", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Noonan syndrome-like disorder with loose anagen hair", "category": "rare disease", "symptoms": ["delayed skeletal maturation", "short stature", "anteverted nares", "cryptorchidism", "hypertelorism", "hearing loss", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "PURA-related severe neonatal hypotonia-seizures-encephalopathy syndrome due to a point mutation", "category": "neurology", "symptoms": ["seizures", "global developmental delay", "severe global developmental delay", "microcephaly", "wide nasal bridge", "anteverted nares", "ataxia", "dystonia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Urofacial syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "renal insufficiency", "hydronephrosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Charcot-Marie-Tooth disease type 1E", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "postural instability", "global developmental delay", "basal ganglia abnormality", "multiple fractures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculocerebrofacial syndrome, Kaufman type", "category": "rare disease", "symptoms": ["microcephaly", "micrognathia", "abnormal optic nerve", "optic atrophy", "intellectual disability", "global developmental delay", "strabismus", "nystagmus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Charcot-Marie-Tooth disease type 1F", "category": "rare disease", "symptoms": ["skeletal muscle atrophy", "somatic sensory dysfunction", "motor delay", "multiple fractures", "sensorineural hearing loss", "hypotonia", "paresthesia", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Otopalatodigital syndrome type 1", "category": "rare disease", "symptoms": ["hypertelorism", "hearing loss", "wide nasal bridge", "downslanted palpebral fissures", "intellectual disability mild", "increased bone mineral density"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Charcot-Marie-Tooth disease type 1A", "category": "rare disease", "symptoms": ["skeletal muscle atrophy", "decreased muscle mass", "paresthesia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Otopalatodigital syndrome type 2", "category": "rare disease", "symptoms": ["hypertelorism", "hearing loss", "conductive hearing loss", "downslanted palpebral fissures", "hypospadias", "hydronephrosis", "micrognathia", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Charcot-Marie-Tooth disease type 1B", "category": "rare disease", "symptoms": ["muscle weakness", "hearing loss", "skeletal muscle atrophy", "elevated creatine kinase", "motor delay", "somatic sensory dysfunction"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "GM1 gangliosidosis", "category": "rare disease", "symptoms": ["long philtrum", "strabismus", "blindness", "nystagmus", "optic atrophy", "seizures", "ataxia", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Gamma-sarcoglycan-related limb-girdle muscular dystrophy R5", "category": "neurology", "symptoms": ["elevated creatine kinase", "proximal muscle weakness", "distal muscle weakness"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Gerstmann-Straussler-Scheinker syndrome", "category": "rare disease", "symptoms": ["cognitive impairment", "sleep disturbance", "paresthesia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Gaucher disease", "category": "metabolic/genetics", "symptoms": ["coagulation abnormality", "pulmonary hypertension", "respiratory insufficiency", "osteoarthritis", "osteolysis", "kyphosis", "short stature", "hemiplegia"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Cerebral autosomal recessive arteriopathy-subcortical infarcts-leukoencephalopathy", "category": "neurology", "symptoms": ["spasticity", "back pain", "cognitive impairment", "rigidity", "somatic sensory dysfunction"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Hyperinsulinism-hyperammonemia syndrome", "category": "rare disease", "symptoms": ["global developmental delay", "grand mal seizures", "attention deficit hyperactivity disorder"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked Charcot-Marie-Tooth disease type 2", "category": "rare disease", "symptoms": ["multiple fractures", "decreased muscle mass", "sensorineural hearing loss", "intellectual disability", "Babinski sign"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked Charcot-Marie-Tooth disease type 3", "category": "rare disease", "symptoms": ["motor delay", "decreased muscle mass", "somatic sensory dysfunction", "tremor", "basal ganglia abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked Charcot-Marie-Tooth disease type 4", "category": "rare disease", "symptoms": ["hearing loss", "intellectual disability", "ataxia", "tremor", "sleep disturbance", "kyphosis", "skeletal muscle atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Adult-onset dystonia-parkinsonism", "category": "neurology", "symptoms": ["spasticity", "tremor", "dysphagia", "rigidity", "frontotemporal dementia", "postural instability", "seizures", "global developmental delay"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Galactosialidosis", "category": "rare disease", "symptoms": ["hearing loss", "intellectual disability", "seizures", "myopathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Primary Sjögren disease", "category": "rare disease", "symptoms": ["kidney abnormality", "joint pain", "abnormal skeletal muscle", "fatigue", "renal insufficiency", "seizures", "muscle weakness", "thrombocytopenia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculo-auriculo-vertebral spectrum", "category": "rare disease", "symptoms": ["micrognathia", "hearing loss", "delayed speech", "microcephaly", "abnormal heart morphology"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked Charcot-Marie-Tooth disease type 1", "category": "rare disease", "symptoms": ["hearing loss", "ataxia", "tremor", "kyphosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fucosidosis", "category": "rare disease", "symptoms": ["hearing loss", "seizures", "hypotonia", "spasticity", "global developmental delay", "failure to thrive", "abnormal facial shape", "hepatomegaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Bilateral frontoparietal polymicrogyria", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "motor delay", "global developmental delay", "intellectual disability severe", "microcephaly", "strabismus", "generalized myoclonic seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fructose-1,6-bisphosphatase deficiency", "category": "rare disease", "symptoms": ["lactic acidosis", "intellectual disability", "seizures", "hypotonia", "hepatomegaly", "drowsiness", "elevated liver enzymes"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hyperimmunoglobulinemia D with periodic fever", "category": "rare disease", "symptoms": ["seizures", "ataxia", "global developmental delay", "growth retardation", "hepatomegaly", "joint pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Familial Mediterranean fever", "category": "rare disease", "symptoms": ["joint pain", "proteinuria", "seizures", "leukocytosis", "sleep disturbance", "fatigue", "chronic pain", "osteoarthritis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "EAST syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "ataxia", "global developmental delay", "grand mal seizures", "basal ganglia abnormality", "sensorineural hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hemorrhagic fever-renal syndrome", "category": "nephrology", "symptoms": ["proteinuria", "muscle weakness", "leukocytosis", "headache", "back pain", "fatigue", "anemia", "elevated liver enzymes"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Abnormal origin of right or left pulmonary artery from the aorta", "category": "pulmonology", "symptoms": ["failure to thrive", "atrial septal defect", "patent ductus arteriosus", "ventricular septal defect", "pulmonary hypertension"], "tests": ["pulmonary function tests", "HRCT chest", "bronchoscopy"], "specialist": "pulmonologist", "relevance_score": 0},
    {"name": "Paraneoplastic cerebellar degeneration", "category": "rare disease", "symptoms": ["ataxia", "nystagmus", "dysphagia", "headache", "neoplasm of the breast", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "MFF-related encephalopathy due to mitochondrial and peroxisomal fission defect", "category": "neurology", "symptoms": ["growth retardation", "visual impairment", "optic atrophy", "seizures", "spasticity", "motor delay", "muscle weakness", "dysphagia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Congenital tracheal stenosis", "category": "rare disease", "symptoms": ["kidney abnormality", "ventricular septal defect", "patent ductus arteriosus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fibrodysplasia ossificans progressiva", "category": "rare disease", "symptoms": ["spinal canal stenosis", "myopathy", "hearing loss", "respiratory insufficiency", "failure to thrive", "intellectual disability", "seizures", "anemia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Gordon syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hearing loss", "growth delay", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Anti-glomerular basement membrane disease", "category": "rare disease", "symptoms": ["renal insufficiency", "proteinuria", "anemia", "respiratory insufficiency", "pulmonary infiltrates", "joint pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "5q14.3 microdeletion syndrome", "category": "rare disease", "symptoms": ["anteverted nares", "strabismus", "autistic behavior", "delayed speech", "seizures", "hypotonia", "intellectual disability severe", "abnormal nervous system morphology"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Gorlin syndrome", "category": "rare disease", "symptoms": ["neoplasm", "wide nasal bridge", "ganglioneuroma", "cryptorchidism", "hypertelorism", "strabismus", "intellectual disability", "myopathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to muscle phosphofructokinase deficiency", "category": "metabolic/genetics", "symptoms": ["muscle weakness", "anemia", "skeletal muscle atrophy"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Simpson-Golabi-Behmel syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hypertelorism", "ventricular septal defect", "hepatomegaly", "hydronephrosis", "wide nasal bridge", "anteverted nares", "downslanted palpebral fissures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pseudohypoparathyroidism type 1A", "category": "endocrinology", "symptoms": ["paresthesia", "elevated CA-125", "short stature", "nystagmus", "intellectual disability", "increased bone mineral density", "sensorineural hearing loss", "strabismus"], "tests": ["hormonal panel", "thyroid function", "adrenal testing", "genetic testing"], "specialist": "endocrinologist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to liver glycogen phosphorylase deficiency", "category": "metabolic/genetics", "symptoms": ["failure to thrive", "hepatomegaly", "growth retardation", "elevated liver enzymes", "hypotonia", "motor delay", "sleep disturbance", "abdominal distension"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Pseudohypoparathyroidism type 1C", "category": "endocrinology", "symptoms": ["elevated CA-125", "nystagmus", "intellectual disability", "short stature", "paresthesia", "increased bone mineral density"], "tests": ["hormonal panel", "thyroid function", "adrenal testing", "genetic testing"], "specialist": "endocrinologist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to muscle glycogen phosphorylase deficiency", "category": "metabolic/genetics", "symptoms": ["elevated creatine kinase", "adult onset", "muscle weakness", "skeletal muscle atrophy", "fatigue", "dysphagia"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Pseudopseudohypoparathyroidism", "category": "endocrinology", "symptoms": ["short stature", "delayed speech", "intellectual disability"], "tests": ["hormonal panel", "thyroid function", "adrenal testing", "genetic testing"], "specialist": "endocrinologist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to acid maltase deficiency", "category": "metabolic/genetics", "symptoms": ["cognitive impairment", "muscle weakness", "delayed speech", "motor delay", "failure to thrive", "growth retardation", "left ventricular hypertrophy", "respiratory insufficiency"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to glucose-6-phosphatase deficiency", "category": "metabolic/genetics", "symptoms": ["seizures", "hypotonia", "susceptibility to infections", "short stature", "cognitive impairment"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to glycogen branching enzyme deficiency", "category": "metabolic/genetics", "symptoms": ["hepatomegaly", "motor delay", "hypotonia", "failure to thrive", "elevated liver enzymes", "respiratory insufficiency", "skeletal muscle atrophy"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to glycogen debranching enzyme deficiency", "category": "metabolic/genetics", "symptoms": ["intellectual disability mild", "immunodeficiency", "short stature"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Ptosis-upper ocular movement limitation-absence of lacrimal punctum syndrome", "category": "rare disease", "symptoms": ["long philtrum", "anteverted nares", "ptosis", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Waldenström macroglobulinemia", "category": "rare disease", "symptoms": ["renal insufficiency", "hearing loss", "proptosis", "ataxia", "respiratory insufficiency", "pulmonary infiltrates", "hepatomegaly", "vertigo"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Familial glucocorticoid deficiency", "category": "rare disease", "symptoms": ["intellectual disability", "spinal cord compression", "failure to thrive", "susceptibility to infections", "cryptorchidism"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Glioblastoma", "category": "rare disease", "symptoms": ["fatigue", "abnormal nervous system physiology", "seizures", "muscle weakness", "headache", "abnormal cerebral white matter"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "8q12 microduplication syndrome", "category": "rare disease", "symptoms": ["long philtrum", "sensorineural hearing loss", "wide nasal bridge", "hypotonia", "global developmental delay", "ventricular septal defect", "atrial septal defect", "attention deficit hyperactivity disorder"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Frontonasal dysplasia-alopecia-genital anomalies syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hypertelorism", "conductive hearing loss", "anteverted nares", "strabismus", "nystagmus", "intellectual disability mild"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hermansky-Pudlak syndrome", "category": "rare disease", "symptoms": ["nystagmus", "neutropenia", "immunodeficiency", "renal insufficiency", "strabismus", "abnormal optic nerve", "visual impairment", "ganglioneuroma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oculocutaneous albinism type 1A", "category": "rare disease", "symptoms": ["visual impairment", "abnormal optic nerve", "nystagmus", "ganglioneuroma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Gitelman syndrome", "category": "rare disease", "symptoms": ["muscle weakness", "failure to thrive", "proteinuria", "headache", "vertigo", "joint pain", "paresthesia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Adult-onset autosomal dominant leukodystrophy", "category": "neurology", "symptoms": ["ataxia", "spasticity", "muscle weakness", "tremor", "Babinski sign", "nystagmus", "dysphagia", "cognitive impairment"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Oculocutaneous albinism type 1B", "category": "rare disease", "symptoms": ["strabismus", "visual impairment", "abnormal optic nerve", "nystagmus", "ganglioneuroma", "melanoma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Non-acquired panhypopituitarism", "category": "rare disease", "symptoms": ["growth retardation", "elevated alkaline phosphatase", "short stature", "fatigue", "delayed skeletal maturation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Argininosuccinic aciduria", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "ataxia", "hepatomegaly", "drowsiness", "EEG abnormality", "elevated liver enzymes", "renal insufficiency"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Growth delay due to insulin-like growth factor I resistance", "category": "rare disease", "symptoms": ["microcephaly", "wide nasal bridge", "intellectual disability", "motor delay", "growth retardation", "delayed skeletal maturation", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Zebra body myopathy", "category": "rare disease", "symptoms": ["global developmental delay", "elevated creatine kinase", "axial muscle weakness", "proximal muscle weakness", "facial palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Growth delay due to insulin-like growth factor type 1 deficiency", "category": "rare disease", "symptoms": ["microcephaly", "sensorineural hearing loss", "intellectual disability", "intellectual disability mild", "failure to thrive", "abnormal facial shape", "short stature", "attention deficit hyperactivity disorder"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fumaric aciduria", "category": "rare disease", "symptoms": ["hypertelorism", "seizures", "hypotonia", "global developmental delay", "premature birth", "microcephaly", "anteverted nares", "optic atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Glutaryl-CoA dehydrogenase deficiency", "category": "rare disease", "symptoms": ["elevated CSF protein", "dystonia", "dysphagia", "headache", "seizures", "ataxia", "tremor", "rigidity"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Methylmalonic acidemia with homocystinuria", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "seizures", "hypotonia", "global developmental delay", "failure to thrive", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Rigid spine syndrome", "category": "rare disease", "symptoms": ["hypotonia", "respiratory insufficiency", "spinal canal stenosis", "abnormal skeletal morphology", "skeletal muscle atrophy", "hip stiffness", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "3-hydroxy-3-methylglutaric aciduria", "category": "rare disease", "symptoms": ["hypsarrhythmia", "intellectual disability severe", "fatigue", "microcephaly", "ataxia", "spasticity", "seizures", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "DNM1L-related encephalopathy due to mitochondrial and peroxisomal fission defect", "category": "neurology", "symptoms": ["nystagmus", "optic atrophy", "seizures", "global developmental delay", "status epilepticus", "developmental regression", "basal ganglia abnormality", "strabismus"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Succinic semialdehyde dehydrogenase deficiency", "category": "rare disease", "symptoms": ["intellectual disability", "ataxia", "hypotonia", "global developmental delay", "generalized myoclonic seizures", "status epilepticus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Vitamin B12-unresponsive methylmalonic acidemia", "category": "rare disease", "symptoms": ["renal insufficiency", "optic atrophy", "intellectual disability", "seizures", "ataxia", "hypotonia", "global developmental delay", "thrombocytopenia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Vitamin B12-responsive methylmalonic acidemia", "category": "rare disease", "symptoms": ["renal insufficiency", "intellectual disability", "hypotonia", "global developmental delay", "failure to thrive", "anemia", "respiratory insufficiency", "hepatomegaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Mevalonic aciduria", "category": "rare disease", "symptoms": ["microcephaly", "downslanted palpebral fissures", "intellectual disability", "seizures", "hypotonia", "global developmental delay", "cerebral cortical atrophy", "delayed skeletal maturation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Self-limited infantile epilepsy", "category": "neurology", "symptoms": ["seizures", "dystonia", "paroxysmal ataxia", "status epilepticus"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Short stature due to partial GHR deficiency", "category": "rare disease", "symptoms": ["delayed skeletal maturation", "growth retardation", "short stature", "midface retrusion"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital cataract-progressive muscular hypotonia-hearing loss-developmental delay syndrome", "category": "otolaryngology", "symptoms": ["ptosis", "hypotonia", "global developmental delay", "lactic acidosis"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Limited cutaneous systemic sclerosis", "category": "rare disease", "symptoms": ["dysphagia", "pulmonary hypertension", "musculoskeletal stiffness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Muscle-eye-brain disease with bilateral multicystic leucodystrophy", "category": "ophthalmology", "symptoms": ["elevated creatine kinase", "intellectual disability severe", "severe global developmental delay"], "tests": ["slit lamp exam", "visual field testing", "retinal imaging", "ERG"], "specialist": "ophthalmologist", "relevance_score": 0},
    {"name": "Foodborne botulism", "category": "rare disease", "symptoms": ["ptosis", "muscle weakness", "dysphagia", "cranial nerve palsy", "arrhythmia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Propionic acidemia", "category": "rare disease", "symptoms": ["intellectual disability", "global developmental delay", "hepatomegaly", "arrhythmia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Acrocallosal syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hypospadias", "hypertelorism", "sensorineural hearing loss", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Acrodermatitis enteropathica", "category": "rare disease", "symptoms": ["visual impairment", "failure to thrive", "cerebral cortical atrophy", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Transaldolase deficiency", "category": "rare disease", "symptoms": ["anemia", "kidney abnormality", "abnormal facial shape", "global developmental delay", "atrial septal defect"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cognitive impairment-coarse facies-heart defects-obesity-pulmonary involvement-short stature-skeletal dysplasia syndrome", "category": "orthopedics", "symptoms": ["anteverted nares", "intellectual disability", "global developmental delay", "short stature", "abnormal skeletal morphology", "microcephaly", "hearing loss", "abnormal heart morphology"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Hereditary orotic aciduria", "category": "rare disease", "symptoms": ["global developmental delay", "anemia", "hypertelorism", "wide nasal bridge", "downslanted palpebral fissures", "patent ductus arteriosus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Oxoglutaric aciduria", "category": "rare disease", "symptoms": ["ataxia", "global developmental delay", "skeletal muscle atrophy", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Glutathione synthetase deficiency", "category": "rare disease", "symptoms": ["hemolytic anemia", "intellectual disability", "seizures", "recurrent infections", "ataxia", "spasticity"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cerebellar-facial-dental syndrome", "category": "rare disease", "symptoms": ["microcephaly", "global developmental delay", "abnormal facial shape", "intellectual disability mild", "failure to thrive", "delayed skeletal maturation", "cryptorchidism", "hydronephrosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Isovaleric acidemia", "category": "rare disease", "symptoms": ["global developmental delay", "intellectual disability", "seizures", "hypotonia", "failure to thrive", "lactic acidosis", "delayed speech", "motor delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hereditary myopathy with early respiratory failure", "category": "rare disease", "symptoms": ["skeletal muscle atrophy", "elevated creatine kinase", "proximal muscle weakness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Sub-cortical nodular heterotopia", "category": "rare disease", "symptoms": ["seizures", "spasticity", "muscle weakness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "6q16 microdeletion syndrome", "category": "rare disease", "symptoms": ["delayed speech", "global developmental delay", "abnormal facial shape", "strabismus", "micrognathia", "conductive hearing loss", "anteverted nares", "autistic behavior"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cushing syndrome due to ectopic ACTH secretion", "category": "rare disease", "symptoms": ["muscle weakness", "leukocytosis", "immunodeficiency", "chronic pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Isolated permanent neonatal diabetes mellitus", "category": "endocrinology", "symptoms": ["hearing loss", "ataxia", "hypotonia", "pancreatic cyst", "intellectual disability severe", "failure to thrive", "intellectual disability", "global developmental delay"], "tests": ["hormonal panel", "thyroid function", "adrenal testing", "genetic testing"], "specialist": "endocrinologist", "relevance_score": 0},
    {"name": "Riboflavin transporter deficiency", "category": "rare disease", "symptoms": ["cranial nerve palsy", "ptosis", "hypotonia", "muscle weakness", "dysphagia", "respiratory insufficiency", "skeletal muscle atrophy", "facial palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Transient neonatal diabetes mellitus", "category": "endocrinology", "symptoms": ["kidney abnormality", "urinary tract abnormality", "hearing loss", "seizures", "hypotonia", "failure to thrive", "abnormal heart morphology"], "tests": ["hormonal panel", "thyroid function", "adrenal testing", "genetic testing"], "specialist": "endocrinologist", "relevance_score": 0},
    {"name": "Congenital intrinsic factor deficiency", "category": "rare disease", "symptoms": ["susceptibility to infections", "chronic pain", "growth retardation", "headache", "paresthesia", "peripheral neuropathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital muscular dystrophy with intellectual disability", "category": "neurology", "symptoms": ["intellectual disability", "elevated creatine kinase", "microcephaly", "global developmental delay", "motor delay", "cerebral cortical atrophy", "cryptorchidism", "strabismus"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Farber disease", "category": "rare disease", "symptoms": ["intellectual disability", "global developmental delay", "failure to thrive", "joint pain", "abnormal skeletal morphology", "seizures", "spasticity", "respiratory insufficiency"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal spastic paraplegia type 30", "category": "rare disease", "symptoms": ["confusion", "Babinski sign", "ataxia", "somatic sensory dysfunction"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "RERE-related neurodevelopmental syndrome", "category": "neurology", "symptoms": ["abnormal heart morphology", "cryptorchidism", "hypospadias", "micrognathia", "hearing loss", "anteverted nares", "strabismus", "ptosis"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Hyperparathyroidism-jaw tumor syndrome", "category": "oncology", "symptoms": ["elevated CA-125", "dysphagia", "fatigue", "renal insufficiency", "renal cyst", "muscle weakness", "headache"], "tests": ["biopsy", "PET scan", "tumor markers", "genetic testing"], "specialist": "oncologist", "relevance_score": 0},
    {"name": "Megalencephaly-severe kyphoscoliosis-overgrowth syndrome", "category": "rare disease", "symptoms": ["abnormal facial shape", "drooling", "prominent nasal bridge", "cerebral cortical atrophy", "kyphosis", "hypertelorism", "hypotonia", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Acitretin/etretinate embryopathy", "category": "rare disease", "symptoms": ["microcephaly", "micrognathia", "anteverted nares", "premature birth", "abnormal facial shape", "bilateral sensorineural hearing impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Microcephaly-intellectual disability-sensorineural hearing loss-epilepsy-abnormal muscle tone syndrome", "category": "neurology", "symptoms": ["microcephaly", "sensorineural hearing loss", "visual impairment", "global developmental delay", "EEG abnormality", "autistic behavior", "seizures", "spasticity"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Eisenmenger syndrome", "category": "rare disease", "symptoms": ["abnormal heart morphology", "pulmonary hypertension", "muscle weakness", "fatigue", "ventricular septal defect", "atrial septal defect", "patent ductus arteriosus", "hepatomegaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fabry disease", "category": "metabolic/genetics", "symptoms": ["renal insufficiency", "hearing loss", "anemia", "joint pain", "fatigue", "musculoskeletal stiffness", "proteinuria", "optic atrophy"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Visceral neuropathy-brain anomalies-facial dysmorphism-developmental delay syndrome", "category": "neurology", "symptoms": ["long philtrum", "strabismus", "downslanted palpebral fissures", "global developmental delay", "cryptorchidism", "microcephaly", "conductive hearing loss", "ptosis"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 23", "category": "rare disease", "symptoms": ["seizures", "hip dislocation", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Multiple osteochondromas", "category": "orthopedics", "symptoms": ["short stature", "osteosarcoma", "short long bones", "somatic sensory dysfunction", "dysphagia"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Subaortic stenosis-short stature syndrome", "category": "cardiology", "symptoms": ["anteverted nares", "short stature", "arrhythmia", "respiratory insufficiency", "kyphosis", "micrognathia", "nystagmus", "intellectual disability"], "tests": ["echocardiogram", "ECG/Holter", "cardiac MRI", "genetic testing"], "specialist": "cardiologist", "relevance_score": 0},
    {"name": "6-pyruvoyl-tetrahydropterin synthase deficiency", "category": "rare disease", "symptoms": ["dystonia", "hypotonia", "ptosis", "delayed speech", "intellectual disability", "seizures", "ataxia", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Apparent mineralocorticoid excess", "category": "rare disease", "symptoms": ["failure to thrive", "short stature", "renal insufficiency", "left ventricular hypertrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Abetalipoproteinemia", "category": "rare disease", "symptoms": ["failure to thrive", "anemia", "reticulocytosis", "vitamin D deficiency", "ataxia", "hepatomegaly", "elevated liver enzymes", "multiple fractures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 26", "category": "rare disease", "symptoms": ["intellectual disability", "muscle weakness", "cerebral cortical atrophy", "skeletal muscle atrophy", "Babinski sign", "urinary tract abnormality", "dystonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Achondroplasia", "category": "orthopedics", "symptoms": ["kyphosis", "colon polyp", "hearing loss", "anteverted nares", "short long bones"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Exstrophy-epispadias complex", "category": "rare disease", "symptoms": ["kidney abnormality", "renal insufficiency", "abnormality of the gastrointestinal tract", "abnormal skeletal morphology", "cryptorchidism", "microcephaly", "abnormal heart morphology", "abnormality of brain morphology"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Zygomycosis", "category": "rare disease", "symptoms": ["fatigue", "premature birth", "neutropenia", "pulmonary infiltrates", "headache", "progressive encephalopathy", "renal insufficiency", "ptosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Corneodermatoosseous syndrome", "category": "dermatology", "symptoms": ["severe short stature", "premature birth", "hearing loss", "stereotypy"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 20", "category": "rare disease", "symptoms": ["hypertelorism", "delayed speech", "spasticity", "global developmental delay", "motor delay", "hypotonia", "growth retardation", "dysphagia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "48,XXYY syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "global developmental delay", "neurological speech impairment", "hypertelorism", "strabismus", "hypotonia", "tremor", "asthma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 21", "category": "rare disease", "symptoms": ["spasticity", "global developmental delay", "dysphagia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pentasomy X syndrome", "category": "rare disease", "symptoms": ["patent ductus arteriosus", "hypotonia", "microcephaly", "hypertelorism", "micrognathia", "wide nasal bridge", "strabismus", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "20q11.2 microdeletion syndrome", "category": "rare disease", "symptoms": ["global developmental delay", "hypertelorism", "hearing loss", "hypotonia", "midface retrusion"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hereditary hyperekplexia", "category": "rare disease", "symptoms": ["ataxia", "spasticity", "rigidity", "sleep disturbance", "intellectual disability", "seizures", "hip dislocation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 27", "category": "rare disease", "symptoms": ["confusion", "Babinski sign", "sensorineural hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 28", "category": "rare disease", "symptoms": ["Babinski sign", "rigidity", "postural instability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fatal infantile lactic acidosis with methylmalonic aciduria", "category": "rare disease", "symptoms": ["global developmental delay", "intellectual disability", "failure to thrive", "growth retardation", "hepatomegaly", "elevated liver enzymes", "lactic acidosis", "skeletal muscle atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal dominant spastic paraplegia type 29", "category": "rare disease", "symptoms": ["confusion", "Babinski sign", "hearing loss", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Distal renal tubular acidosis", "category": "nephrology", "symptoms": ["muscle weakness", "failure to thrive", "growth retardation", "short stature", "renal cyst", "sensorineural hearing loss", "hemolytic anemia"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Intellectual disability-muscle weakness-short stature-facial dysmorphism syndrome", "category": "rare disease", "symptoms": ["prominent nasal bridge", "downslanted palpebral fissures", "delayed speech", "intellectual disability", "muscle weakness", "delayed skeletal maturation", "proximal muscle weakness", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Stiff person spectrum disorder", "category": "rare disease", "symptoms": ["dysphagia", "rigidity", "autoimmune antibody", "cognitive impairment", "vertigo"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital muscular dystrophy without intellectual disability", "category": "neurology", "symptoms": ["muscular dystrophy", "motor delay", "hypotonia", "abnormal cerebral white matter", "proximal muscle weakness", "microcephaly"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Thymoma", "category": "rare disease", "symptoms": ["muscle weakness", "neoplasm", "immunodeficiency", "systemic lupus erythematosus", "myositis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Erythrokeratodermia variabilis", "category": "rare disease", "symptoms": ["microcephaly", "short stature", "abnormal testis", "hearing loss", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Stimmler syndrome", "category": "rare disease", "symptoms": ["microcephaly", "ataxia", "short stature", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Lamellar ichthyosis", "category": "dermatology", "symptoms": ["renal insufficiency", "short stature", "cognitive impairment"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "16p12.1p12.3 triplication syndrome", "category": "rare disease", "symptoms": ["long philtrum", "conductive hearing loss", "failure to thrive", "abnormal heart morphology", "delayed speech and language", "strabismus", "atrial septal defect", "attention deficit hyperactivity disorder"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Central neurocytoma", "category": "neurology", "symptoms": ["ataxia", "headache", "postural instability", "paresthesia", "Babinski sign"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Carney-Stratakis syndrome", "category": "rare disease", "symptoms": ["hearing loss", "dysphagia", "cranial nerve palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Holoprosencephaly-radial heart renal anomalies syndrome", "category": "cardiology", "symptoms": ["microcephaly", "lower limb pain", "hearing loss"], "tests": ["echocardiogram", "ECG/Holter", "cardiac MRI", "genetic testing"], "specialist": "cardiologist", "relevance_score": 0},
    {"name": "Intellectual disability-seizures-macrocephaly-obesity syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "hearing loss", "prominent forehead", "strabismus", "downslanted palpebral fissures", "ptosis", "delayed speech"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Deafness-hypogonadism syndrome", "category": "otolaryngology", "symptoms": ["hypertelorism", "delayed skeletal maturation", "cognitive impairment"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Jervell and Lange-Nielsen syndrome", "category": "rare disease", "symptoms": ["bilateral sensorineural hearing impairment", "arrhythmia", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Insulinoma", "category": "rare disease", "symptoms": ["seizures", "tremor", "paresthesia", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "High altitude pulmonary edema", "category": "pulmonology", "symptoms": ["leukocytosis", "headache", "sleep disturbance", "vertigo"], "tests": ["pulmonary function tests", "HRCT chest", "bronchoscopy"], "specialist": "pulmonologist", "relevance_score": 0},
    {"name": "Lead poisoning", "category": "rare disease", "symptoms": ["abdominal distension", "fatigue", "premature birth", "anemia", "headache", "attention deficit hyperactivity disorder", "vitamin D deficiency", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "CADDS", "category": "rare disease", "symptoms": ["global developmental delay", "sensorineural hearing loss", "seizures", "abnormal cerebral white matter", "micrognathia", "strabismus", "dystonia", "elevated liver enzymes"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital muscular dystrophy with cerebellar involvement", "category": "neurology", "symptoms": ["elevated creatine kinase", "muscular dystrophy", "microcephaly", "global developmental delay", "abnormality of brain morphology", "strabismus", "optic atrophy", "seizures"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Infantile spasms-broad thumbs syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "prominent forehead", "seizures", "cerebral cortical atrophy", "microcephaly", "downslanted palpebral fissures", "EEG abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked spasticity-intellectual disability-epilepsy syndrome", "category": "neurology", "symptoms": ["intellectual disability", "spasticity", "rigidity", "status epilepticus"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "X-linked intellectual disability, Stocco Dos Santos type", "category": "rare disease", "symptoms": ["seizures", "kyphosis", "short stature", "intellectual disability severe", "strabismus", "delayed speech"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Septopreoptic holoprosencephaly", "category": "ophthalmology", "symptoms": ["intellectual disability", "hypotonia", "myopathy", "microcephaly", "dysphagia"], "tests": ["slit lamp exam", "visual field testing", "retinal imaging", "ERG"], "specialist": "ophthalmologist", "relevance_score": 0},
    {"name": "X-linked spinocerebellar ataxia type 3", "category": "neurology", "symptoms": ["sensorineural hearing loss", "optic atrophy", "ataxia", "hypotonia", "global developmental delay"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "X-linked intellectual disability, Cabezas type", "category": "rare disease", "symptoms": ["downslanted palpebral fissures", "neurological speech impairment", "growth delay", "intellectual disability severe", "small hand", "tremor", "short stature", "microcephaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Mercury poisoning", "category": "rare disease", "symptoms": ["seizures", "dystonia", "tremor", "abnormal cerebral white matter"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked intellectual disability, Wilson type", "category": "metabolic/genetics", "symptoms": ["seizures", "growth retardation", "susceptibility to infections", "intellectual disability severe", "microcephaly"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "STT3B-CDG", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "seizures", "global developmental delay", "hypotonia", "failure to thrive", "cryptorchidism", "optic atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Sézary syndrome", "category": "rare disease", "symptoms": ["lymphoma", "hepatomegaly", "immunodeficiency", "tremor", "skeletal muscle atrophy", "peripheral neuropathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "SSR4-CDG", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "global developmental delay", "hypotonia", "abnormal facial shape", "strabismus", "failure to thrive", "abnormality of the gastrointestinal tract"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "SHORT syndrome", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "severe short stature", "neurological speech impairment", "midface retrusion", "hypertelorism", "wide nasal bridge"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Omphalocele syndrome, Shprintzen-Goldberg type", "category": "rare disease", "symptoms": ["downslanted palpebral fissures", "hypotonia", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Wound botulism", "category": "rare disease", "symptoms": ["ptosis", "muscle weakness", "dysphagia", "cranial nerve palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked intellectual disability, Cantagrel type", "category": "rare disease", "symptoms": ["autistic behavior", "intellectual disability", "seizures", "cerebral cortical atrophy", "abnormal skeletal muscle", "severe global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Infant botulism", "category": "rare disease", "symptoms": ["ptosis", "hypotonia", "dysphagia", "drooling", "sleep disturbance", "cranial nerve palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Christianson syndrome", "category": "rare disease", "symptoms": ["drooling", "EEG abnormality", "sleep disturbance", "strabismus", "grand mal seizures", "developmental regression", "severe global developmental delay", "microcephaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "KDM5C-related syndromic X-linked intellectual disability", "category": "rare disease", "symptoms": ["cryptorchidism", "delayed speech", "intellectual disability severe", "seizures", "spasticity", "short stature", "microcephaly", "prominent nasal bridge"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "BRESEK syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "microcephaly", "hearing loss", "conductive hearing loss", "prominent forehead", "global developmental delay", "growth retardation", "intestinal malrotation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Progressive spondyloepimetaphyseal dysplasia-short stature-short fourth metatarsals-intellectual disability syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "proptosis", "autistic behavior", "intellectual disability", "motor delay", "delayed skeletal maturation", "hip abnormality", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Craniosynostosis-hydrocephalus-Arnold-Chiari malformation type I-radioulnar synostosis syndrome", "category": "orthopedics", "symptoms": ["cryptorchidism", "hypospadias", "hypertelorism", "long philtrum", "micrognathia", "conductive hearing loss", "anteverted nares", "seizures"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "X-linked intellectual disability, Schimke type", "category": "rare disease", "symptoms": ["hearing loss", "intellectual disability", "spasticity", "global developmental delay", "short stature", "hydronephrosis", "cerebral cortical atrophy", "hip stiffness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked intellectual disability-cubitus valgus-dysmorphism syndrome", "category": "rare disease", "symptoms": ["microcephaly", "downslanted palpebral fissures", "seizures", "abnormal facial shape", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "MEHMO syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "microcephaly", "nystagmus", "seizures", "hypotonia", "growth retardation", "EEG abnormality", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Eosinophilic fasciitis", "category": "rare disease", "symptoms": ["eosinophilia", "joint pain", "paresthesia", "fatigue", "myositis", "abnormal lymphocyte morphology"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Carnitine palmitoyl transferase II deficiency, myopathic form", "category": "rare disease", "symptoms": ["muscle weakness", "renal insufficiency", "elevated creatine kinase"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Sialuria", "category": "rare disease", "symptoms": ["hypertelorism", "conductive hearing loss", "wide nasal bridge", "seizures", "intellectual disability mild", "hypotonia", "abnormal facial shape", "hepatomegaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hyperphenylalaninemia due to tetrahydrobiopterin deficiency", "category": "rare disease", "symptoms": ["hypotonia", "microcephaly", "delayed speech", "seizures", "dystonia", "sleep disturbance", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Sillence syndrome", "category": "rare disease", "symptoms": ["short finger", "back pain", "myopathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Carnitine palmitoyl transferase II deficiency, severe infantile form", "category": "rare disease", "symptoms": ["elevated liver enzymes", "arrhythmia", "muscle weakness", "headache", "elevated creatine kinase", "seizures", "hepatomegaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "XYLT1-CDG", "category": "rare disease", "symptoms": ["proptosis", "microcephaly", "long philtrum", "growth retardation", "hepatomegaly", "short long bones", "short stature", "gastrointestinal bleeding"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Carnitine palmitoyl transferase II deficiency, neonatal form", "category": "rare disease", "symptoms": ["renal insufficiency", "seizures", "hepatomegaly", "elevated creatine kinase", "arrhythmia", "abnormality of brain morphology", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Mohr-Tranebjaerg syndrome", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "visual impairment", "optic atrophy", "dystonia", "Babinski sign", "tremor", "dysphagia", "postural instability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital tricuspid valve dysplasia", "category": "rare disease", "symptoms": ["premature birth", "hepatomegaly", "patent foramen ovale"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Multiple sclerosis-ichthyosis-factor VIII deficiency syndrome", "category": "dermatology", "symptoms": ["optic atrophy", "ataxia", "coagulation abnormality", "vertigo", "hemiplegia"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Sclerosteosis", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "ptosis", "optic atrophy", "facial palsy", "increased bone mineral density"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "FLNA-related X-linked myxomatous valvular dysplasia", "category": "rare disease", "symptoms": ["aortic insufficiency", "bicuspid aortic valve", "hypertelorism", "long philtrum", "micrognathia", "hypotonia", "patent ductus arteriosus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "MEDNIK syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "intestinal obstruction", "peripheral neuropathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Benign recurrent intrahepatic cholestasis", "category": "hepatology", "symptoms": ["hearing loss", "elevated liver enzymes", "fatigue"], "tests": ["liver function tests", "liver biopsy", "hepatic MRI"], "specialist": "hepatologist", "relevance_score": 0},
    {"name": "Isolated focal cortical dysplasia", "category": "rare disease", "symptoms": ["seizures", "intellectual disability mild", "cognitive impairment", "grand mal seizures", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Adult intestinal botulism", "category": "gastroenterology", "symptoms": ["ptosis", "muscle weakness", "cranial nerve palsy"], "tests": ["colonoscopy/endoscopy", "GI motility studies", "gastric emptying scan"], "specialist": "gastroenterologist", "relevance_score": 0},
    {"name": "X-linked intellectual disability, Abidi type", "category": "rare disease", "symptoms": ["microcephaly", "hearing loss", "prominent nasal bridge", "intellectual disability", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Syndromic X-linked intellectual disability 7", "category": "rare disease", "symptoms": ["cryptorchidism", "intellectual disability", "muscle weakness", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked intellectual disability, Armfield type", "category": "rare disease", "symptoms": ["cryptorchidism", "micrognathia", "strabismus", "downslanted palpebral fissures", "seizures", "global developmental delay", "patent ductus arteriosus", "cerebral cortical atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Intestinal botulism", "category": "gastroenterology", "symptoms": ["ptosis", "hypotonia", "dysphagia", "cranial nerve palsy"], "tests": ["colonoscopy/endoscopy", "GI motility studies", "gastric emptying scan"], "specialist": "gastroenterologist", "relevance_score": 0},
    {"name": "Short stature due to GHSR deficiency", "category": "rare disease", "symptoms": ["delayed skeletal maturation", "growth retardation", "short stature", "underweight"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Senior-Loken syndrome", "category": "rare disease", "symptoms": ["visual impairment", "ataxia", "global developmental delay", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Septo-optic dysplasia spectrum", "category": "ophthalmology", "symptoms": ["visual impairment", "cryptorchidism", "strabismus", "nystagmus", "seizures", "short stature", "hemiplegia", "sensorineural hearing loss"], "tests": ["slit lamp exam", "visual field testing", "retinal imaging", "ERG"], "specialist": "ophthalmologist", "relevance_score": 0},
    {"name": "Autism spectrum disorder-epilepsy-arthrogryposis syndrome", "category": "neurology", "symptoms": ["autistic behavior", "intellectual disability mild", "hip dislocation", "intellectual disability severe"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Familial thrombocytosis", "category": "rare disease", "symptoms": ["headache", "paresthesia", "seizures", "pulmonary hypertension", "vertigo", "myelodysplastic syndrome"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Idiopathic hypereosinophilic syndrome", "category": "rare disease", "symptoms": ["eosinophilia", "anemia", "leukocytosis", "myelodysplastic syndrome", "abdominal distension", "seizures", "muscle weakness", "failure to thrive"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autoimmune lymphoproliferative syndrome", "category": "immunology", "symptoms": ["hemolysis", "hepatomegaly", "eosinophilia", "reticulocytosis", "renal insufficiency", "seizures", "pulmonary infiltrates", "headache"], "tests": ["immunoglobulin levels", "lymphocyte subset panel", "genetic testing"], "specialist": "immunologist", "relevance_score": 0},
    {"name": "Megalencephaly-polymicrogyria-postaxial polydactyly-hydrocephalus syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "seizures", "ventricular septal defect"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Familial gestational hyperthyroidism", "category": "endocrinology", "symptoms": ["motor delay", "sleep disturbance", "proptosis"], "tests": ["hormonal panel", "thyroid function", "adrenal testing", "genetic testing"], "specialist": "endocrinologist", "relevance_score": 0},
    {"name": "LIG4 syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "microcephaly", "micrognathia", "wide nasal bridge", "intellectual disability", "global developmental delay", "growth retardation", "leukocytosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Spinocerebellar ataxia with axonal neuropathy type 1", "category": "neurology", "symptoms": ["ataxia", "hypercholesterolemia", "multiple fractures", "peripheral neuropathy", "seizures"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Stevens-Johnson syndrome/toxic epidermal necrolysis spectrum", "category": "rare disease", "symptoms": ["elevated liver enzymes", "neutropenia", "abdominal distension", "blindness", "inflammation of skin", "fatigue", "anemia", "headache"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cerebellar ataxia, Cayman type", "category": "neurology", "symptoms": ["nystagmus", "global developmental delay", "hypotonia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Radioulnar synostosis-microcephaly-scoliosis syndrome", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "global developmental delay", "premature birth", "delayed skeletal maturation", "growth delay", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Recessive mitochondrial ataxia syndrome", "category": "neurology", "symptoms": ["seizures", "ataxia", "hypotonia", "dysphagia", "headache", "sensory neuropathy", "peripheral neuropathy", "cognitive impairment"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Anterior cutaneous nerve entrapment syndrome", "category": "rare disease", "symptoms": ["somatic sensory dysfunction", "vertigo", "leukocytosis", "abdominal distension", "back pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ullrich congenital muscular dystrophy", "category": "neurology", "symptoms": ["hypotonia", "kyphosis", "elevated creatine kinase", "spinal canal stenosis", "proximal muscle weakness", "micrognathia", "muscle weakness", "hip dislocation"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Progressive myoclonic epilepsy type 3", "category": "neurology", "symptoms": ["intellectual disability", "developmental regression", "microcephaly", "optic atrophy"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "19p13.3 microduplication syndrome", "category": "rare disease", "symptoms": ["microcephaly", "delayed speech", "global developmental delay", "abnormal facial shape", "micrognathia", "conductive hearing loss", "motor delay", "growth retardation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Proximal symphalangism", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "strabismus", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Desmoplastic small round cell tumor", "category": "oncology", "symptoms": ["anemia", "hepatomegaly", "abdominal distension"], "tests": ["biopsy", "PET scan", "tumor markers", "genetic testing"], "specialist": "oncologist", "relevance_score": 0},
    {"name": "CAMOS syndrome", "category": "rare disease", "symptoms": ["renal insufficiency", "seizures", "spasticity", "brain atrophy", "microcephaly", "optic atrophy", "intellectual disability", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "T-cell immunodeficiency with thymic aplasia", "category": "immunology", "symptoms": ["susceptibility to infections", "failure to thrive", "recurrent infections", "rheumatoid arthritis"], "tests": ["immunoglobulin levels", "lymphocyte subset panel", "genetic testing"], "specialist": "immunologist", "relevance_score": 0},
    {"name": "Madras motor neuron disease", "category": "neurology", "symptoms": ["sensorineural hearing loss", "Babinski sign", "visual impairment", "optic atrophy", "dysphagia", "abnormal retinal morphology", "facial palsy"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "MORM syndrome", "category": "rare disease", "symptoms": ["visual impairment", "delayed speech", "intellectual disability mild", "intellectual disability severe", "kidney abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cleft lip/palate-ectodermal dysplasia syndrome", "category": "rare disease", "symptoms": ["micrognathia", "wide nasal bridge", "downslanted palpebral fissures", "intellectual disability", "seizures", "neurological speech impairment", "EEG abnormality", "midface retrusion"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "6q terminal deletion syndrome", "category": "rare disease", "symptoms": ["hypospadias", "hypertelorism", "micrognathia", "strabismus", "nystagmus", "delayed speech", "global developmental delay", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hemimegalencephaly", "category": "rare disease", "symptoms": ["seizures", "global developmental delay", "optic atrophy", "status epilepticus", "cranial nerve palsy", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Haddad syndrome", "category": "rare disease", "symptoms": ["strabismus", "failure to thrive", "intestinal malrotation", "intellectual disability", "seizures", "hypotonia", "sensorineural hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Filippi syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "microcephaly", "prominent nasal bridge", "wide nasal bridge", "intellectual disability", "global developmental delay", "severe short stature", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cenani-Lenz syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "downslanted palpebral fissures", "lower limb pain", "hearing loss", "prominent forehead", "ptosis", "proptosis", "nystagmus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "COG1-CDG", "category": "rare disease", "symptoms": ["intellectual disability mild", "failure to thrive", "abnormal facial shape", "pulmonary hypertension", "short long bones", "hypertelorism", "long philtrum", "micrognathia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "COG4-CDG", "category": "rare disease", "symptoms": ["microcephaly", "nystagmus", "ataxia", "global developmental delay", "growth retardation", "elevated liver enzymes", "hypercholesterolemia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Apert syndrome", "category": "rare disease", "symptoms": ["midface retrusion", "sensorineural hearing loss", "visual impairment", "optic atrophy", "respiratory insufficiency", "proptosis", "hypertelorism", "prominent forehead"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Early-onset progressive leukoencephalopathy-central nervous system calcification-deafness-visual impairment syndrome", "category": "neurology", "symptoms": ["seizures", "growth retardation", "developmental regression", "spastic tetraplegia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Deafness-craniofacial syndrome", "category": "orthopedics", "symptoms": ["sensorineural hearing loss", "wide nasal bridge", "patent ductus arteriosus"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Idiopathic aplastic anemia", "category": "hematology", "symptoms": ["neutropenia", "susceptibility to infections", "anemia"], "tests": ["CBC with differential", "peripheral blood smear", "bone marrow biopsy", "flow cytometry"], "specialist": "hematologist", "relevance_score": 0},
    {"name": "Leukoencephalopathy with brain stem and spinal cord involvement-high lactate syndrome", "category": "neurology", "symptoms": ["Babinski sign", "tremor", "neurological speech impairment", "motor delay", "hearing loss", "ptosis", "nystagmus", "optic atrophy"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Antisynthetase syndrome", "category": "rare disease", "symptoms": ["muscle weakness", "respiratory insufficiency", "myositis", "hypotonia", "joint pain", "elevated creatine kinase", "aortic insufficiency", "dysphagia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Renpenning syndrome", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "skeletal muscle atrophy", "severe short stature", "hypospadias", "growth retardation", "sensorineural hearing loss", "strabismus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive spinocerebellar ataxia-blindness-deafness syndrome", "category": "neurology", "symptoms": ["hearing loss", "blindness", "nystagmus", "optic atrophy"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Sweet syndrome", "category": "rare disease", "symptoms": ["leukocytosis", "joint pain", "myositis", "anemia", "neoplasm", "susceptibility to infections", "progressive encephalopathy", "breast carcinoma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Antley-Bixler syndrome", "category": "rare disease", "symptoms": ["anteverted nares", "proptosis", "hypertelorism", "long philtrum", "strabismus", "downslanted palpebral fissures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital tracheomalacia", "category": "rare disease", "symptoms": ["respiratory insufficiency", "failure to thrive", "premature birth", "abnormal heart morphology", "ventricular septal defect", "atrial septal defect", "patent ductus arteriosus", "pulmonary hypertension"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fanconi anemia", "category": "hematology", "symptoms": ["hypertelorism", "micrognathia", "hearing loss", "strabismus", "visual impairment", "ptosis", "proptosis", "nystagmus"], "tests": ["CBC with differential", "peripheral blood smear", "bone marrow biopsy", "flow cytometry"], "specialist": "hematologist", "relevance_score": 0},
    {"name": "Antiphospholipid syndrome", "category": "rare disease", "symptoms": ["premature birth", "fatigue", "proteinuria", "seizures", "aortic insufficiency", "pulmonary hypertension"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ataxia with vitamin E deficiency", "category": "neurology", "symptoms": ["ataxia", "muscle weakness", "Babinski sign", "peripheral neuropathy", "nystagmus", "neurological speech impairment", "visual impairment", "dystonia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Proximal myopathy with extrapyramidal signs", "category": "rare disease", "symptoms": ["global developmental delay", "dystonia", "microcephaly", "ptosis", "optic atrophy", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Familial paroxysmal ataxia", "category": "neurology", "symptoms": ["nystagmus", "ataxia", "vertigo", "dystonia", "intellectual disability"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic ataxia of Charlevoix-Saguenay", "category": "neurology", "symptoms": ["ataxia", "spasticity", "muscle weakness", "Babinski sign", "peripheral neuropathy", "dysphagia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Deafness-vitiligo-achalasia syndrome", "category": "otolaryngology", "symptoms": ["sensorineural hearing loss", "EEG abnormality", "skeletal muscle atrophy", "severe short stature"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Secondary short bowel syndrome", "category": "rare disease", "symptoms": ["failure to thrive", "growth retardation", "atherosclerosis", "abdominal distension", "intestinal malrotation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "COG8-CDG", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "ataxia", "failure to thrive", "duodenal obstruction", "developmental regression", "skeletal muscle atrophy", "severe global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "46,XY partial gonadal dysgenesis", "category": "rare disease", "symptoms": ["cryptorchidism", "delayed skeletal maturation", "hypospadias"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Auriculocondylar syndrome", "category": "rare disease", "symptoms": ["micrognathia", "hearing loss", "ptosis", "global developmental delay", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Aspartylglucosaminuria", "category": "rare disease", "symptoms": ["hypertelorism", "wide nasal bridge", "delayed speech", "intellectual disability", "abnormal facial shape", "neurological speech impairment", "seizures", "hepatomegaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Friedreich ataxia", "category": "neurology", "symptoms": ["abnormal ECG", "sensory neuropathy", "hearing loss", "Babinski sign", "nystagmus", "optic atrophy", "muscle weakness", "spasticity"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Conductive deafness-ptosis-skeletal anomalies syndrome", "category": "orthopedics", "symptoms": ["ptosis", "hip abnormality", "growth delay"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Argininemia", "category": "rare disease", "symptoms": ["seizures", "global developmental delay", "neurological speech impairment", "EEG abnormality", "hemiplegia", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Aromatase deficiency", "category": "rare disease", "symptoms": ["growth retardation", "delayed skeletal maturation", "osteosarcoma", "cryptorchidism"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Amoebiasis due to Entamoeba histolytica", "category": "rare disease", "symptoms": ["anemia", "leukocytosis", "elevated liver enzymes"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Amoebiasis due to free-living amoebae", "category": "rare disease", "symptoms": ["seizures", "headache", "abnormal cerebral white matter", "immunodeficiency", "facial palsy", "blindness", "ataxia", "arrhythmia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ossification anomalies-psychomotor developmental delay syndrome", "category": "rare disease", "symptoms": ["dysphagia", "delayed speech and language", "global developmental delay", "hypotonia", "hepatomegaly", "elevated liver enzymes", "anteverted nares", "hypertelorism"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Alport syndrome", "category": "rare disease", "symptoms": ["renal insufficiency", "proteinuria", "sensorineural hearing loss", "dysphagia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Alström syndrome", "category": "rare disease", "symptoms": ["kyphosis", "elevated liver enzymes", "somatic sensory dysfunction", "elevated gamma-glutamyltransferase", "autistic behavior", "pulmonary hypertension", "hepatomegaly", "elevated serum ferritin"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Leber congenital amaurosis", "category": "rare disease", "symptoms": ["severely reduced visual acuity", "nystagmus", "seizures", "hypotonia", "hemiplegia", "hearing loss", "autistic behavior", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Optic atrophy-intellectual disability syndrome", "category": "ophthalmology", "symptoms": ["optic atrophy", "autistic behavior", "intellectual disability", "seizures", "hypotonia", "global developmental delay", "abnormal facial shape", "hearing loss"], "tests": ["slit lamp exam", "visual field testing", "retinal imaging", "ERG"], "specialist": "ophthalmologist", "relevance_score": 0},
    {"name": "Fetal Gaucher disease", "category": "metabolic/genetics", "symptoms": ["anteverted nares", "hypotonia", "failure to thrive", "anemia", "seizures"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Sialidosis type 2", "category": "rare disease", "symptoms": ["hearing loss", "delayed speech", "seizures", "ataxia", "global developmental delay", "hypotonia", "muscle weakness", "tremor"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Deafness-enamel hypoplasia-nail defects syndrome", "category": "otolaryngology", "symptoms": ["intellectual disability", "short stature", "arrhythmia", "delayed skeletal maturation", "peripheral neuropathy", "hearing loss", "sensorineural hearing loss"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Alpha-1-antitrypsin deficiency", "category": "rare disease", "symptoms": ["emphysema", "bronchiectasis", "elevated liver enzymes", "asthma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Secondary syringomyelia", "category": "rare disease", "symptoms": ["paresthesia", "back pain", "somatic sensory dysfunction", "cranial nerve palsy", "nystagmus", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Caroli syndrome", "category": "rare disease", "symptoms": ["kidney abnormality", "hepatomegaly", "thrombocytopenia", "leukocytosis", "elevated liver enzymes"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Alpha-mannosidosis", "category": "rare disease", "symptoms": ["hearing loss", "intellectual disability", "global developmental delay", "hepatomegaly", "delayed skeletal maturation", "hypertelorism", "hypotonia", "kyphosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pituitary stalk interruption syndrome", "category": "rare disease", "symptoms": ["failure to thrive", "short stature", "cryptorchidism", "intellectual disability", "seizures", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Phosphoribosylpyrophosphate synthetase superactivity", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "ataxia", "renal insufficiency", "intellectual disability", "hypotonia", "neurological speech impairment", "strabismus", "arrhythmia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ravine syndrome", "category": "rare disease", "symptoms": ["failure to thrive", "ataxia", "spasticity"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Deafness-genital anomalies-metacarpal and metatarsal synostosis syndrome", "category": "otolaryngology", "symptoms": ["hypospadias", "conductive hearing loss", "sensorineural hearing loss", "intellectual disability", "cerebral cortical atrophy", "hypertelorism", "neurological speech impairment", "myopathy"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Global developmental delay-osteopenia-ectodermal defect syndrome", "category": "rare disease", "symptoms": ["attention deficit hyperactivity disorder", "delayed speech", "EEG abnormality", "long philtrum", "micrognathia", "wide nasal bridge"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Combined pituitary hormone deficiencies, genetic forms", "category": "rare disease", "symptoms": ["growth retardation", "elevated alkaline phosphatase", "fatigue", "delayed skeletal maturation", "seizures", "severe global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Angiostrongyliasis", "category": "rare disease", "symptoms": ["headache", "joint pain", "paresthesia", "fatigue", "seizures", "muscle weakness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Genitopatellar syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hydronephrosis", "microcephaly", "hypertelorism", "long philtrum", "micrognathia", "hearing loss", "conductive hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Keutel syndrome", "category": "rare disease", "symptoms": ["hearing loss", "optic atrophy", "seizures", "intellectual disability mild", "global developmental delay", "ventricular septal defect", "pulmonary hypertension", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Deafness-small bowel diverticulosis-neuropathy syndrome", "category": "neurology", "symptoms": ["sensorineural hearing loss", "ptosis", "neurological speech impairment"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Deafness-epiphyseal dysplasia-short stature syndrome", "category": "otolaryngology", "symptoms": ["hearing loss", "intellectual disability mild", "global developmental delay", "neurological speech impairment", "short stature"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Fountain syndrome", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "intellectual disability", "EEG abnormality", "midface retrusion", "hypertelorism", "visual impairment", "ptosis", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Proximal spinal muscular atrophy", "category": "neurology", "symptoms": ["skeletal muscle atrophy", "dysphagia", "basal ganglia abnormality", "axial muscle weakness", "fatigue", "hypotonia", "global developmental delay", "motor delay"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Genetic recurrent myoglobinuria", "category": "rare disease", "symptoms": ["adult onset", "muscle weakness", "renal insufficiency", "elevated liver enzymes", "myositis", "neurological speech impairment", "arrhythmia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Chylomicron retention disease", "category": "rare disease", "symptoms": ["elevated liver enzymes", "failure to thrive", "growth retardation", "abdominal distension", "visual impairment", "proximal muscle weakness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Angelman syndrome", "category": "rare disease", "symptoms": ["microcephaly", "autistic behavior", "delayed speech", "seizures", "ataxia", "motor delay", "tremor", "cerebral cortical atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Gorham-Stout disease", "category": "rare disease", "symptoms": ["osteolysis", "reduced bone mineral density", "hearing loss", "abnormality of the thoracic cavity"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Deaf blind hypopigmentation syndrome, Yemenite type", "category": "otolaryngology", "symptoms": ["sensorineural hearing loss", "strabismus", "nystagmus"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Leukocyte adhesion deficiency type II", "category": "rare disease", "symptoms": ["seizures", "iron deficiency anemia", "sleep disturbance", "intellectual disability severe", "microcephaly", "hypertelorism", "wide nasal bridge", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Conductive deafness-malformed external ear syndrome", "category": "otolaryngology", "symptoms": ["conductive hearing loss", "sensorineural hearing loss", "global developmental delay"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Adenylosuccinate lyase deficiency", "category": "rare disease", "symptoms": ["microcephaly", "long philtrum", "conductive hearing loss", "anteverted nares", "intellectual disability", "seizures", "hypotonia", "abnormal facial shape"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked agammaglobulinemia", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "failure to thrive", "neutropenia", "anemia", "neoplasm", "immunodeficiency", "short stature", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 59", "category": "rare disease", "symptoms": ["nystagmus", "confusion", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Medium chain acyl-CoA dehydrogenase deficiency", "category": "rare disease", "symptoms": ["hypotonia", "hepatomegaly", "delayed speech", "ataxia", "elevated liver enzymes", "skeletal muscle atrophy", "elevated creatine kinase", "arrhythmia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked adrenoleukodystrophy", "category": "neurology", "symptoms": ["visual impairment", "intellectual disability", "headache", "somatic sensory dysfunction", "attention deficit hyperactivity disorder", "cognitive impairment"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Neonatal adrenoleukodystrophy", "category": "neurology", "symptoms": ["sensorineural hearing loss", "wide nasal bridge", "anteverted nares", "strabismus", "nystagmus", "optic atrophy", "seizures", "hypotonia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Penile agenesis", "category": "rare disease", "symptoms": ["cryptorchidism", "hydronephrosis", "ventricular septal defect", "atrial septal defect"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Stüve-Wiedemann syndrome", "category": "rare disease", "symptoms": ["paresthesia", "short stature", "asthma", "osteosarcoma", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "White matter hypoplasia-corpus callosum agenesis-intellectual disability syndrome", "category": "rare disease", "symptoms": ["microcephaly", "hypertelorism", "micrognathia", "wide nasal bridge", "downslanted palpebral fissures", "intellectual disability", "hypotonia", "cerebral cortical atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Isolated succinate-CoQ reductase deficiency", "category": "rare disease", "symptoms": ["blindness", "nystagmus", "spasticity", "motor delay", "left ventricular hypertrophy", "generalized myoclonic seizures", "developmental regression", "Babinski sign"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ventricular extrasystoles with syncopal episodes-perodactyly-Robin sequence syndrome", "category": "rare disease", "symptoms": ["short stature", "arrhythmia", "microcephaly", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "11q22.2q22.3 microdeletion syndrome", "category": "rare disease", "symptoms": ["hypotonia", "global developmental delay", "micrognathia", "conductive hearing loss", "strabismus", "ptosis", "delayed speech", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Spastic tetraplegia-thin corpus callosum-progressive postnatal microcephaly syndrome", "category": "rare disease", "symptoms": ["delayed speech", "intellectual disability", "hypotonia", "motor delay", "dysphagia", "brain atrophy", "hypertelorism", "conductive hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Stormorken-Sjaastad-Langslet syndrome", "category": "rare disease", "symptoms": ["neurological speech impairment", "abnormal skeletal muscle", "short stature", "anemia", "coagulation abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Spinocerebellar ataxia type 7", "category": "neurology", "symptoms": ["ataxia", "dysphagia", "nystagmus", "global developmental delay", "motor delay", "muscle weakness", "failure to thrive", "somatic sensory dysfunction"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Resistance to thyrotropin-releasing hormone syndrome", "category": "rare disease", "symptoms": ["fatigue", "overweight", "growth retardation", "delayed skeletal maturation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Sturge-Weber syndrome", "category": "rare disease", "symptoms": ["seizures", "strabismus", "optic atrophy", "delayed speech", "intellectual disability", "headache", "attention deficit hyperactivity disorder", "abnormal cerebrovascular morphology"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Alkaptonuria", "category": "rare disease", "symptoms": ["osteoarthritis", "joint pain", "back pain", "hemolytic anemia", "atherosclerosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Glycogen storage disease due to aldolase A deficiency", "category": "metabolic/genetics", "symptoms": ["muscle weakness", "hemolytic anemia", "proximal muscle weakness", "delayed speech", "intellectual disability", "motor delay", "growth retardation", "arrhythmia"], "tests": ["enzyme assay", "lysosomal enzyme panel", "whole exome sequencing", "metabolic blood panel"], "specialist": "medical geneticist", "relevance_score": 0},
    {"name": "Alexander disease", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "spasticity", "failure to thrive", "neurological speech impairment", "EEG abnormality", "sleep disturbance", "ptosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Allan-Herndon-Dudley syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "delayed speech", "ataxia", "spasticity", "dystonia", "sleep disturbance", "skeletal muscle atrophy", "brain atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Alagille syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hypertelorism", "micrognathia", "strabismus", "downslanted palpebral fissures", "intellectual disability mild", "failure to thrive", "ventricular septal defect"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Albers-Schönberg osteopetrosis", "category": "rare disease", "symptoms": ["osteoarthritis", "cranial nerve palsy", "facial palsy", "optic atrophy", "anemia", "osteosarcoma", "short stature", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked recessive ocular albinism", "category": "rare disease", "symptoms": ["strabismus", "visual impairment", "nystagmus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Corneal dystrophy-perceptive deafness syndrome", "category": "otolaryngology", "symptoms": ["sensorineural hearing loss", "visual impairment", "nystagmus"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "X-linked complicated corpus callosum dysgenesis", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "seizures", "spasticity", "muscle weakness", "intestinal malrotation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Corpus callosum agenesis-neuronopathy syndrome", "category": "neurology", "symptoms": ["microcephaly", "strabismus", "nystagmus", "intellectual disability", "seizures", "global developmental delay", "EEG abnormality", "hemiplegia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Crimean-Congo hemorrhagic fever", "category": "rare disease", "symptoms": ["hepatomegaly", "headache", "elevated creatine kinase", "thrombocytopenia", "leukocytosis", "vertigo", "proteinuria", "pulmonary hypertension"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Intellectual disability-hypoplastic corpus callosum-preauricular tag syndrome", "category": "rare disease", "symptoms": ["microcephaly", "growth retardation", "delayed skeletal maturation", "intellectual disability severe", "micrognathia", "nystagmus", "optic atrophy", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Dengue fever", "category": "rare disease", "symptoms": ["thrombocytopenia", "hepatomegaly", "headache", "joint pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Yellow fever", "category": "rare disease", "symptoms": ["renal insufficiency", "headache", "joint pain", "elevated creatine kinase", "seizures", "leukocytosis", "status epilepticus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Vici syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "hypotonia", "global developmental delay", "EEG abnormality", "susceptibility to infections", "short stature", "nystagmus", "optic atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Lassa fever", "category": "rare disease", "symptoms": ["chronic pain", "muscle weakness", "dysphagia", "headache", "hearing loss", "seizures", "premature birth", "joint pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Aicardi syndrome", "category": "rare disease", "symptoms": ["microcephaly", "chorioretinal coloboma", "nystagmus", "optic atrophy", "hypotonia", "spasticity", "EEG abnormality", "hemiplegia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Nipah virus disease", "category": "rare disease", "symptoms": ["seizures", "tremor", "headache", "vertigo", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Aicardi-Goutières syndrome", "category": "rare disease", "symptoms": ["prominent forehead", "spasticity", "global developmental delay", "microcephaly", "seizures", "dystonia", "hypertension", "developmental regression"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Marburg hemorrhagic fever", "category": "rare disease", "symptoms": ["joint pain", "elevated liver enzymes", "lactic acidosis", "elevated creatine kinase", "back pain", "thrombocytopenia", "reticulocytosis", "headache"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Combined oxidative phosphorylation defect type 23", "category": "rare disease", "symptoms": ["lactic acidosis", "visual impairment", "seizures", "intellectual disability mild", "global developmental delay", "failure to thrive", "left ventricular hypertrophy", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cooper-Jabs syndrome", "category": "rare disease", "symptoms": ["anteverted nares", "intellectual disability", "ventricular septal defect", "strabismus", "hypotonia", "respiratory insufficiency", "hip abnormality", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Lethal congenital contracture syndrome type 1", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "skeletal muscle atrophy", "hip abnormality", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Juvenile dermatomyositis", "category": "dermatology", "symptoms": ["hypotonia", "muscle weakness", "dysphagia", "joint pain", "elevated creatine kinase", "arrhythmia", "fatigue", "musculoskeletal stiffness"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Combined oxidative phosphorylation defect type 27", "category": "rare disease", "symptoms": ["hearing loss", "intellectual disability", "global developmental delay", "generalized myoclonic seizures", "status epilepticus", "developmental regression", "abnormal cerebral white matter", "autistic behavior"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Interatrial communication", "category": "rare disease", "symptoms": ["atrial septal defect", "pulmonary hypertension", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Renal coloboma syndrome", "category": "nephrology", "symptoms": ["renal insufficiency", "visual impairment", "hearing loss", "strabismus", "nystagmus"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Uveal coloboma-cleft lip and palate-intellectual disability", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "chorioretinal coloboma", "intellectual disability", "strabismus", "visual impairment", "ptosis", "nystagmus", "optic atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Prader-Willi syndrome due to paternal deletion of 15q11q13 type 2", "category": "rare disease", "symptoms": ["small hand", "seizures", "intellectual disability mild", "global developmental delay", "abnormal facial shape", "cryptorchidism", "strabismus", "autistic behavior"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Prader-Willi syndrome due to paternal deletion of 15q11q13 type 1", "category": "rare disease", "symptoms": ["cryptorchidism", "strabismus", "autistic behavior", "hypotonia", "global developmental delay", "failure to thrive", "sleep disturbance", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Endosteal hyperostosis, Worth type", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "nystagmus", "facial palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Severe congenital hypochromic anemia with ringed sideroblasts", "category": "hematology", "symptoms": ["anemia", "elevated liver enzymes", "fatigue", "growth retardation"], "tests": ["CBC with differential", "peripheral blood smear", "bone marrow biopsy", "flow cytometry"], "specialist": "hematologist", "relevance_score": 0},
    {"name": "Cogan syndrome", "category": "rare disease", "symptoms": ["vertigo", "sensorineural hearing loss", "anemia", "leukocytosis", "blindness", "aortic insufficiency"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "COFS syndrome", "category": "rare disease", "symptoms": ["microcephaly", "micrognathia", "sensorineural hearing loss", "wide nasal bridge", "visual impairment", "optic atrophy", "seizures", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Coffin-Siris syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hearing loss", "anteverted nares", "strabismus", "visual impairment", "ptosis", "seizures", "growth retardation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Univentricular heart", "category": "cardiology", "symptoms": ["failure to thrive", "hepatomegaly", "arrhythmia"], "tests": ["echocardiogram", "ECG/Holter", "cardiac MRI", "genetic testing"], "specialist": "cardiologist", "relevance_score": 0},
    {"name": "Criss-cross heart", "category": "cardiology", "symptoms": ["ventricular septal defect", "respiratory insufficiency", "abnormal facial shape"], "tests": ["echocardiogram", "ECG/Holter", "cardiac MRI", "genetic testing"], "specialist": "cardiologist", "relevance_score": 0},
    {"name": "Autosomal recessive congenital cerebellar ataxia due to MGLUR1 deficiency", "category": "neurology", "symptoms": ["intellectual disability", "neurological speech impairment", "ptosis", "seizures"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Otofaciocervical syndrome", "category": "rare disease", "symptoms": ["anteverted nares", "intellectual disability", "global developmental delay", "neurological speech impairment", "delayed skeletal maturation", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Acquired methemoglobinemia", "category": "rare disease", "symptoms": ["headache", "vertigo", "fatigue", "drowsiness", "arrhythmia", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pachydermoperiostosis", "category": "rare disease", "symptoms": ["small hand", "ptosis", "osteolysis", "joint pain", "anemia", "hepatomegaly"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pachygyria-intellectual disability-epilepsy syndrome", "category": "neurology", "symptoms": ["seizures", "global developmental delay", "premature birth", "intellectual disability severe"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Prader-Willi syndrome due to imprinting mutation", "category": "rare disease", "symptoms": ["global developmental delay", "abnormal facial shape", "short stature", "small hand"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "CDKL5-deficiency disorder", "category": "rare disease", "symptoms": ["autistic behavior", "delayed speech", "intellectual disability", "hypotonia", "growth retardation", "delayed speech and language", "hypsarrhythmia", "severe global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fatty acyl-CoA reductase 1 deficiency", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "global developmental delay", "growth retardation", "abnormal facial shape", "short stature", "hypertelorism", "long philtrum"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Osteoporosis-pseudoglioma syndrome", "category": "rare disease", "symptoms": ["severely reduced visual acuity", "delayed speech", "global developmental delay", "delayed speech and language", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Lateral meningocele syndrome", "category": "rare disease", "symptoms": ["micrognathia", "conductive hearing loss", "downslanted palpebral fissures", "ptosis", "dural ectasia", "cryptorchidism", "hypertelorism", "sensorineural hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Joubert syndrome with hepatic defect", "category": "hepatology", "symptoms": ["renal insufficiency", "conductive hearing loss", "prominent nasal bridge", "anteverted nares", "strabismus", "visual impairment", "ptosis", "chorioretinal coloboma"], "tests": ["liver function tests", "liver biopsy", "hepatic MRI"], "specialist": "hepatologist", "relevance_score": 0},
    {"name": "Cleidocranial dysplasia", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "short stature", "hearing loss", "midface retrusion", "osteosarcoma", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "CINCA syndrome", "category": "rare disease", "symptoms": ["hearing loss", "sensorineural hearing loss", "visual impairment", "proptosis", "blindness", "intellectual disability", "global developmental delay", "growth retardation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 8 syndrome", "category": "rare disease", "symptoms": ["hydronephrosis", "anteverted nares", "intellectual disability", "cleft palate"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "3-phosphoserine phosphatase deficiency, infantile/juvenile form", "category": "rare disease", "symptoms": ["global developmental delay", "hypospadias", "microcephaly", "micrognathia", "abnormal facial shape"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Osteopathia striata-cranial sclerosis syndrome", "category": "rare disease", "symptoms": ["increased bone mineral density", "wide nasal bridge", "micrognathia", "conductive hearing loss", "intellectual disability", "global developmental delay", "severe short stature", "facial palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "3-phosphoglycerate dehydrogenase deficiency, infantile/juvenile form", "category": "rare disease", "symptoms": ["microcephaly", "spasticity", "failure to thrive", "short stature", "severe global developmental delay", "abnormal facial shape", "generalized myoclonic seizures", "spastic tetraplegia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Prader-Willi syndrome due to translocation", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "anteverted nares", "strabismus", "intellectual disability mild", "abnormal facial shape", "short stature", "small hand"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Osteopetrosis with renal tubular acidosis", "category": "nephrology", "symptoms": ["global developmental delay", "abnormal facial shape", "elevated creatine kinase", "short stature", "hydronephrosis", "micrognathia", "optic atrophy", "intellectual disability"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "CODAS syndrome", "category": "rare disease", "symptoms": ["anteverted nares", "global developmental delay", "delayed skeletal maturation", "short stature", "sensorineural hearing loss", "ptosis", "hypotonia", "strabismus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Osteoporosis-oculocutaneous hypopigmentation syndrome", "category": "rare disease", "symptoms": ["visual impairment", "nystagmus", "kyphosis", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Coarctation of aorta", "category": "rare disease", "symptoms": ["bicuspid aortic valve", "patent ductus arteriosus", "pulmonary hypertension"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Intermediate osteopetrosis", "category": "rare disease", "symptoms": ["back pain", "anemia", "visual impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Short ulna-dysmorphism-hypotonia-intellectual disability syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "long philtrum", "micrognathia", "conductive hearing loss", "hypotonia", "intellectual disability mild", "global developmental delay", "abnormal facial shape"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 21 syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "global developmental delay", "abnormal facial shape", "microcephaly", "EEG abnormality", "delayed speech", "spasticity"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 20 syndrome", "category": "rare disease", "symptoms": ["EEG abnormality", "intellectual disability", "growth retardation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 19 syndrome", "category": "rare disease", "symptoms": ["microcephaly", "hypertelorism", "micrognathia", "hearing loss", "conductive hearing loss", "prominent nasal bridge", "autistic behavior", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 18 syndrome", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "global developmental delay", "short stature", "cognitive impairment", "seizures", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 17 syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "global developmental delay", "short stature", "microcephaly", "micrognathia", "wide nasal bridge", "delayed speech"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 14 syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "delayed speech", "hypotonia", "motor delay", "susceptibility to infections", "strabismus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Early-onset epileptic encephalopathy-cortical blindness-intellectual disability-facial dysmorphism syndrome", "category": "neurology", "symptoms": ["anteverted nares", "intellectual disability", "prominent nasal bridge", "hypotonia", "hypsarrhythmia", "basal ganglia abnormality"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Nasu-Hakola disease", "category": "rare disease", "symptoms": ["seizures", "spasticity", "cerebral cortical atrophy", "neurological speech impairment", "developmental regression", "joint pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Bruck syndrome", "category": "rare disease", "symptoms": ["respiratory insufficiency", "kyphosis", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital osteogenesis imperfecta-microcephaly-cataracts syndrome", "category": "orthopedics", "symptoms": ["microcephaly", "hypertelorism", "cryptorchidism", "ventricular septal defect"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Osteogenesis imperfecta-retinopathy-seizures-intellectual disability syndrome", "category": "orthopedics", "symptoms": ["optic atrophy", "intellectual disability", "seizures", "severe global developmental delay"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Ring chromosome 7 syndrome", "category": "rare disease", "symptoms": ["hypospadias", "microcephaly", "prominent nasal bridge", "wide nasal bridge", "downslanted palpebral fissures", "motor delay", "cerebral cortical atrophy", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Dihydropyrimidinuria", "category": "rare disease", "symptoms": ["intellectual disability", "seizures", "microcephaly", "hypotonia", "failure to thrive", "growth retardation", "attention deficit hyperactivity disorder", "developmental regression"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 6 syndrome", "category": "rare disease", "symptoms": ["microcephaly", "hypertelorism", "wide nasal bridge", "respiratory insufficiency", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Multicentric carpo-tarsal osteolysis with or without nephropathy", "category": "nephrology", "symptoms": ["proteinuria", "micrognathia", "wide nasal bridge", "proptosis", "osteolysis"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Autosomal recessive distal osteolysis syndrome", "category": "rare disease", "symptoms": ["proptosis", "intellectual disability mild", "osteolysis", "short stature", "midface retrusion"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 22 syndrome", "category": "rare disease", "symptoms": ["microcephaly", "autistic behavior", "delayed speech", "seizures", "global developmental delay", "hypotonia", "growth retardation", "developmental regression"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "7p22.1 microduplication syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "kidney abnormality", "hypertelorism", "delayed speech", "global developmental delay", "motor delay", "abnormal heart morphology", "abnormal facial shape"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked microcephaly-growth retardation-prognathism-cryptorchidism syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "microcephaly", "seizures", "hypotonia", "abnormal facial shape", "susceptibility to infections", "hypospadias", "sensorineural hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "COG2-CDG", "category": "rare disease", "symptoms": ["intellectual disability", "abnormal facial shape", "spastic tetraplegia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Familial osteodysplasia, Anderson type", "category": "rare disease", "symptoms": ["kyphosis", "growth delay", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Acute adrenal insufficiency", "category": "nephrology", "symptoms": ["muscle weakness", "failure to thrive", "fatigue", "renal insufficiency", "seizures", "hypotonia", "vertigo", "joint pain"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Ring chromosome 12 syndrome", "category": "rare disease", "symptoms": ["global developmental delay", "growth retardation", "abnormal facial shape", "microcephaly", "cryptorchidism", "conductive hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 10 syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "long philtrum", "micrognathia", "conductive hearing loss", "wide nasal bridge", "downslanted palpebral fissures", "intellectual disability", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ring chromosome 1 syndrome", "category": "rare disease", "symptoms": ["microcephaly", "long philtrum", "wide nasal bridge", "anteverted nares", "downslanted palpebral fissures", "ptosis", "growth delay", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Xq21 microdeletion syndrome", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "optic atrophy", "ataxia", "intellectual disability mild", "global developmental delay", "growth retardation", "delayed skeletal maturation", "bilateral sensorineural hearing impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Lethal recessive chondrodysplasia", "category": "orthopedics", "symptoms": ["short long bones", "gastrointestinal bleeding", "micrognathia"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Chondrodysplasia-difference of sex development syndrome", "category": "orthopedics", "symptoms": ["microcephaly", "intellectual disability", "severe short stature", "strabismus", "chorioretinal coloboma"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Semilobar holoprosencephaly", "category": "rare disease", "symptoms": ["failure to thrive", "growth retardation", "short stature", "microcephaly", "sensorineural hearing loss", "intellectual disability", "seizures", "spasticity"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Imperforate oropharynx-costovertebral anomalies syndrome", "category": "rare disease", "symptoms": ["conductive hearing loss", "wide nasal bridge", "downslanted palpebral fissures", "premature birth", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Staphylococcal necrotizing pneumonia", "category": "rare disease", "symptoms": ["thrombocytopenia", "pulmonary infiltrates", "immunodeficiency", "leukocytosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Orofaciodigital syndrome type 1", "category": "rare disease", "symptoms": ["hypertelorism", "wide nasal bridge", "downslanted palpebral fissures", "intellectual disability", "seizures", "ataxia", "cleft palate", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Orofaciodigital syndrome type 2", "category": "rare disease", "symptoms": ["wide nasal bridge", "short stature", "micrognathia", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive otospondylomegaepiphyseal dysplasia", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "anteverted nares", "myopathy", "midface retrusion", "micrognathia", "proptosis", "osteoarthritis", "strabismus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Bacterial toxic-shock syndrome", "category": "rare disease", "symptoms": ["renal insufficiency", "elevated creatine kinase", "myositis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Desbuquois syndrome", "category": "rare disease", "symptoms": ["proptosis", "intellectual disability", "severe short stature", "gastrointestinal bleeding", "ventricular septal defect", "growth delay", "small hand", "anteverted nares"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Orofaciodigital syndrome type 4", "category": "rare disease", "symptoms": ["microcephaly", "hypertelorism", "micrognathia", "conductive hearing loss", "intellectual disability", "global developmental delay", "severe short stature", "proptosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Orofaciodigital syndrome type 6", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "nystagmus", "intellectual disability", "ataxia", "hypotonia", "global developmental delay", "failure to thrive"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital alveolar capillary dysplasia", "category": "rare disease", "symptoms": ["hydronephrosis", "ventricular septal defect", "atrial septal defect", "patent ductus arteriosus", "bicuspid aortic valve", "pulmonary hypertension", "intestinal malrotation", "myopathy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "PAPA syndrome", "category": "rare disease", "symptoms": ["proteinuria", "joint pain", "fatigue", "myositis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Horizontal gaze palsy with progressive scoliosis", "category": "rare disease", "symptoms": ["kyphosis", "microcephaly", "strabismus", "visual impairment", "nystagmus", "hypotonia", "delayed skeletal maturation", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "SIX2-related frontonasal dysplasia", "category": "rare disease", "symptoms": ["ptosis", "hypertelorism", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Opitz GBBB syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "long philtrum", "wide nasal bridge", "hypospadias", "anteverted nares", "ptosis", "intellectual disability", "abnormal facial shape"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Opsismodysplasia", "category": "rare disease", "symptoms": ["hypotonia", "respiratory insufficiency", "hepatomegaly", "delayed skeletal maturation", "severe short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Camptodactyly syndrome, Guadalajara type 3", "category": "rare disease", "symptoms": ["intellectual disability mild", "delayed skeletal maturation", "small hand"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Diffuse cutaneous systemic sclerosis", "category": "rare disease", "symptoms": ["renal insufficiency", "muscle weakness", "dysphagia", "pulmonary hypertension", "pulmonary infiltrates", "osteolysis", "joint pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital ichthyosiform erythroderma", "category": "dermatology", "symptoms": ["hearing loss", "failure to thrive", "short stature"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Autosomal dominant generalized epidermolysis bullosa simplex, severe form", "category": "dermatology", "symptoms": ["growth retardation", "anemia", "global developmental delay", "failure to thrive", "susceptibility to infections"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Familial calcium pyrophosphate deposition", "category": "rare disease", "symptoms": ["joint pain", "osteoarthritis", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hardikar syndrome", "category": "rare disease", "symptoms": ["short stature", "hydronephrosis", "ventricular septal defect", "atrial septal defect", "patent ductus arteriosus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Andersen-Tawil syndrome", "category": "rare disease", "symptoms": ["micrognathia", "abnormal facial shape", "ventricular arrhythmia", "short stature", "hypertelorism", "conductive hearing loss", "wide nasal bridge", "seizures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ophthalmoplegia-intellectual disability-lingua scrotalis syndrome", "category": "rare disease", "symptoms": ["ptosis", "intellectual disability", "global developmental delay", "facial palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Severe combined immunodeficiency due to adenosine deaminase deficiency", "category": "immunology", "symptoms": ["failure to thrive", "inflammation of skin", "allergy"], "tests": ["immunoglobulin levels", "lymphocyte subset panel", "genetic testing"], "specialist": "immunologist", "relevance_score": 0},
    {"name": "Skeletal dysplasia-T-cell immunodeficiency-developmental delay syndrome", "category": "orthopedics", "symptoms": ["global developmental delay", "motor delay", "microcephaly", "intellectual disability", "hypotonia", "abnormal facial shape", "kyphosis", "limb joint contracture"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "T-B+ severe combined immunodeficiency due to gamma chain deficiency", "category": "immunology", "symptoms": ["recurrent infections", "failure to thrive", "hepatomegaly", "lymphoma"], "tests": ["immunoglobulin levels", "lymphocyte subset panel", "genetic testing"], "specialist": "immunologist", "relevance_score": 0},
    {"name": "Acquired von Willebrand syndrome", "category": "rare disease", "symptoms": ["progressive encephalopathy", "aortic insufficiency", "subcutaneous hemorrhage", "chronic pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Steinert myotonic dystrophy", "category": "rare disease", "symptoms": ["fatigue", "cognitive impairment", "muscle weakness", "autistic behavior", "intellectual disability mild", "global developmental delay", "respiratory insufficiency", "cerebral cortical atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital muscular dystrophy, Fukuyama type", "category": "neurology", "symptoms": ["visual impairment", "optic atrophy", "delayed speech", "seizures", "hypotonia", "global developmental delay", "EEG abnormality", "muscular dystrophy"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Severe combined immunodeficiency due to DCLRE1C deficiency", "category": "immunology", "symptoms": ["hemolysis", "bronchiectasis", "recurrent infections", "failure to thrive", "neoplasm"], "tests": ["immunoglobulin levels", "lymphocyte subset panel", "genetic testing"], "specialist": "immunologist", "relevance_score": 0},
    {"name": "Oculopharyngeal muscular dystrophy", "category": "neurology", "symptoms": ["cognitive impairment", "ptosis", "dysphagia", "spondylolisthesis", "fatigue", "axial muscle weakness", "elevated creatine kinase"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Craniometaphyseal dysplasia", "category": "orthopedics", "symptoms": ["hypertelorism", "sensorineural hearing loss", "wide nasal bridge", "visual impairment", "facial palsy"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Perrault syndrome", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "ataxia", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Craniofrontonasal dysplasia", "category": "orthopedics", "symptoms": ["hypertelorism", "wide nasal bridge", "microcephaly", "sensorineural hearing loss", "downslanted palpebral fissures", "intellectual disability", "hypotonia", "growth delay"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Craniofacial-deafness-hand syndrome", "category": "orthopedics", "symptoms": ["hypertelorism", "sensorineural hearing loss", "downslanted palpebral fissures"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Craniotelencephalic dysplasia", "category": "orthopedics", "symptoms": ["global developmental delay", "microcephaly", "visual impairment", "optic atrophy"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Alopecia-intellectual disability syndrome", "category": "rare disease", "symptoms": ["microcephaly", "hearing loss", "intellectual disability", "hypotonia", "delayed skeletal maturation", "seizures", "growth retardation", "EEG abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Facioscapulohumeral dystrophy", "category": "rare disease", "symptoms": ["skeletal muscle atrophy", "elevated creatine kinase", "muscle weakness progressive", "sensorineural hearing loss", "multiple fractures", "proximal muscle weakness", "seizures", "respiratory insufficiency"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cranio-osteoarthropathy", "category": "orthopedics", "symptoms": ["osteoarthritis", "joint pain", "cleft palate"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Dysferlin-related limb-girdle muscular dystrophy R2", "category": "neurology", "symptoms": ["elevated creatine kinase", "abnormal ECG", "abnormal lymphocyte morphology", "dysphagia", "basal ganglia abnormality", "spinal canal stenosis"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Congenital progressive bone marrow failure-B-cell immunodeficiency-skeletal dysplasia syndrome", "category": "orthopedics", "symptoms": ["anemia", "intellectual disability", "neutropenia", "thrombocytopenia", "abnormal facial shape", "myelodysplastic syndrome", "short stature", "hearing loss"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Calpain-3-related limb-girdle muscular dystrophy R1", "category": "neurology", "symptoms": ["elevated creatine kinase", "spinal canal stenosis", "muscular dystrophy"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Emery-Dreifuss muscular dystrophy", "category": "neurology", "symptoms": ["elevated creatine kinase", "elevated uric acid", "spinal canal stenosis", "back pain", "proximal muscle weakness", "ptosis", "hypotonia", "kyphosis"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "9p13 microdeletion syndrome", "category": "rare disease", "symptoms": ["wide nasal bridge", "anteverted nares", "short stature", "attention deficit hyperactivity disorder", "conductive hearing loss", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "46,XY difference of sex development due to isolated 17,20-lyase deficiency", "category": "rare disease", "symptoms": ["hypospadias", "delayed skeletal maturation", "cryptorchidism", "short stature", "failure to thrive"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Global developmental delay-neuro-ophthalmological abnormalities-seizures-intellectual disability syndrome", "category": "neurology", "symptoms": ["developmental regression", "basal ganglia abnormality", "global developmental delay", "hypotonia", "intellectual disability", "seizures", "failure to thrive", "growth retardation"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Intellectual disability-alacrima-achalasia syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "global developmental delay", "motor delay", "strabismus", "dysphagia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pericardial and diaphragmatic defect", "category": "rare disease", "symptoms": ["abnormal heart morphology", "atrial septal defect", "patent ductus arteriosus", "bicuspid aortic valve"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Perlman syndrome", "category": "rare disease", "symptoms": ["micrognathia", "wide nasal bridge", "intellectual disability", "hypotonia", "global developmental delay", "hepatomegaly", "cryptorchidism", "conductive hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "SPECC1L-related hypertelorism syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "hypertelorism", "long philtrum", "conductive hearing loss", "prominent nasal bridge", "wide nasal bridge", "strabismus", "downslanted palpebral fissures"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Aseptic abscess syndrome", "category": "rare disease", "symptoms": ["anemia", "elevated liver enzymes", "abnormal testis", "kidney abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cantú syndrome", "category": "rare disease", "symptoms": ["long philtrum", "wide nasal bridge", "anteverted nares", "intellectual disability mild", "patent ductus arteriosus", "delayed skeletal maturation", "gastrointestinal bleeding"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital lipoid adrenal hyperplasia due to STAR deficency", "category": "nephrology", "symptoms": ["seizures", "failure to thrive", "breast carcinoma"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Non-syndromic bilambdoid and sagittal craniosynostosis", "category": "orthopedics", "symptoms": ["hypertelorism", "micrognathia", "conductive hearing loss", "wide nasal bridge", "strabismus", "downslanted palpebral fissures", "nystagmus", "patent ductus arteriosus"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Congenital adrenal hyperplasia due to 3-beta-hydroxysteroid dehydrogenase deficiency", "category": "nephrology", "symptoms": ["hypospadias", "cryptorchidism", "global developmental delay", "failure to thrive"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Transketolase deficiency", "category": "rare disease", "symptoms": ["global developmental delay", "abnormal heart morphology", "delayed speech", "hypotonia", "intellectual disability mild", "ventricular septal defect", "atrial septal defect", "attention deficit hyperactivity disorder"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Cranioectodermal dysplasia", "category": "orthopedics", "symptoms": ["anteverted nares", "nystagmus", "growth delay"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Craniodigital-intellectual disability syndrome", "category": "orthopedics", "symptoms": ["micrognathia", "intellectual disability", "short stature"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Laminin subunit alpha 2-related congenital muscular dystrophy", "category": "neurology", "symptoms": ["muscular dystrophy", "hypotonia", "motor delay", "muscle weakness", "basal ganglia abnormality", "myositis", "intellectual disability", "seizures"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Congenital adrenal hyperplasia due to 17-alpha-hydroxylase deficiency", "category": "nephrology", "symptoms": ["hypospadias", "delayed skeletal maturation", "failure to thrive"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Craniodiaphyseal dysplasia", "category": "orthopedics", "symptoms": ["wide nasal bridge", "optic atrophy", "intellectual disability", "short stature"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Classic congenital adrenal hyperplasia due to 21-hydroxylase deficiency", "category": "nephrology", "symptoms": ["failure to thrive", "short stature", "gastrointestinal bleeding"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Crane-Heise syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "micrognathia", "anteverted nares", "cryptorchidism"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Penoscrotal transposition", "category": "rare disease", "symptoms": ["hypospadias", "micrognathia", "cerebral cortical atrophy", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ethylmalonic encephalopathy", "category": "neurology", "symptoms": ["intellectual disability", "seizures", "ataxia", "hypotonia", "failure to thrive", "developmental regression", "lactic acidosis"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Mitochondrial neurogastrointestinal encephalomyopathy", "category": "neurology", "symptoms": ["dysphagia", "abdominal distension", "sensorineural hearing loss", "ptosis", "abnormal cerebral white matter", "elevated liver enzymes", "lactic acidosis", "paresthesia"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Microform holoprosencephaly", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "premature birth", "short stature", "anteverted nares", "strabismus", "seizures", "asthma"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital total pulmonary venous return anomaly", "category": "pulmonology", "symptoms": ["atrial septal defect", "pulmonary hypertension", "fatigue", "ventricular septal defect", "patent ductus arteriosus", "hepatomegaly"], "tests": ["pulmonary function tests", "HRCT chest", "bronchoscopy"], "specialist": "pulmonologist", "relevance_score": 0},
    {"name": "Fetal cytomegalovirus syndrome", "category": "rare disease", "symptoms": ["sensorineural hearing loss", "intellectual disability", "anemia", "coagulation abnormality", "hepatomegaly", "elevated liver enzymes", "microcephaly", "optic atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Tick-borne encephalitis", "category": "rare disease", "symptoms": ["thrombocytopenia", "headache", "joint pain", "fatigue", "hearing loss", "visual impairment", "tremor", "leukocytosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Ollier disease", "category": "rare disease", "symptoms": ["osteolysis", "neoplasm", "osteosarcoma", "cranial nerve palsy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Distal duplication 5q syndrome", "category": "rare disease", "symptoms": ["microcephaly", "short stature", "cryptorchidism", "hypospadias", "hypertelorism", "long philtrum", "micrognathia", "conductive hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital varicella syndrome", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "global developmental delay", "cerebral cortical atrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital rubella syndrome", "category": "rare disease", "symptoms": ["microcephaly", "sensorineural hearing loss", "strabismus", "visual impairment", "nystagmus", "intellectual disability", "seizures", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital enterovirus infection", "category": "rare disease", "symptoms": ["premature birth", "neutropenia", "thrombocytopenia", "anemia", "leukocytosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "8p inverted duplication/deletion syndrome", "category": "rare disease", "symptoms": ["abnormal facial shape", "spastic tetraplegia", "intellectual disability severe", "severe global developmental delay", "urinary tract abnormality", "long philtrum", "wide nasal bridge", "autistic behavior"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Char syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "downslanted palpebral fissures", "ptosis", "patent ductus arteriosus", "growth delay", "hearing loss", "strabismus", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "RFT1-CDG", "category": "rare disease", "symptoms": ["seizures", "hypotonia", "global developmental delay", "hearing loss", "microcephaly", "visual impairment", "failure to thrive", "coagulation abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Stiff skin syndrome", "category": "dermatology", "symptoms": ["sensorineural hearing loss", "strabismus", "muscle weakness", "abnormal skeletal muscle", "elevated cholesterol", "short stature", "peripheral neuropathy", "midface retrusion"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Wrinkly skin syndrome", "category": "dermatology", "symptoms": ["cryptorchidism", "hypertelorism", "long philtrum", "conductive hearing loss", "downslanted palpebral fissures", "delayed speech", "global developmental delay", "failure to thrive"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "4H leukodystrophy", "category": "neurology", "symptoms": ["ataxia", "dystonia", "tremor", "dysphagia", "drooling", "short stature", "seizures", "optic atrophy"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Pectus excavatum-macrocephaly-dysplastic nails syndrome", "category": "rare disease", "symptoms": ["hypotonia", "global developmental delay", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "PEHO syndrome", "category": "rare disease", "symptoms": ["optic atrophy", "seizures", "global developmental delay", "cerebral cortical atrophy", "drowsiness", "EEG abnormality", "hypsarrhythmia", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Renal caliceal diverticuli-deafness syndrome", "category": "nephrology", "symptoms": ["kidney abnormality", "urinary tract abnormality", "sensorineural hearing loss", "hydronephrosis"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Pelvis-shoulder dysplasia", "category": "rare disease", "symptoms": ["hydronephrosis", "micrognathia", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Coxoauricular syndrome", "category": "rare disease", "symptoms": ["hearing loss", "hip dislocation", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Autosomal recessive Robinow syndrome", "category": "rare disease", "symptoms": ["hip abnormality", "hypertelorism", "wide nasal bridge", "anteverted nares", "growth delay", "midface retrusion", "cryptorchidism", "long philtrum"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Severe growth deficiency-strabismus-extensive dermal melanocytosis-intellectual disability syndrome", "category": "rare disease", "symptoms": ["seizures", "global developmental delay", "proteinuria", "cerebral cortical atrophy", "sensorineural hearing loss", "strabismus", "visual impairment", "nystagmus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Hereditary elliptocytosis", "category": "rare disease", "symptoms": ["abnormal erythrocyte", "hemolytic anemia", "reticulocytosis", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Classical Ehlers-Danlos syndrome", "category": "rheumatology", "symptoms": ["hypotonia", "muscle weakness", "fatigue", "motor delay", "premature birth", "osteoarthritis", "hip dislocation", "joint pain"], "tests": ["ANA panel", "RF/anti-CCP", "inflammatory markers (ESR/CRP)", "joint imaging"], "specialist": "rheumatologist", "relevance_score": 0},
    {"name": "Ellis Van Creveld syndrome", "category": "rare disease", "symptoms": ["failure to thrive", "osteosarcoma", "cryptorchidism", "hypospadias", "kidney abnormality", "strabismus", "ventricular septal defect", "atrial septal defect"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pelizaeus-Merzbacher disease, connatal form", "category": "rare disease", "symptoms": ["nystagmus", "intellectual disability severe", "ataxia", "hypotonia", "basal ganglia abnormality", "failure to thrive", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Alveolar echinococcosis", "category": "rare disease", "symptoms": ["eosinophilia", "anemia", "fatigue", "chronic pain", "renal cyst", "headache", "seizures", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Vascular Ehlers-Danlos syndrome", "category": "rheumatology", "symptoms": ["cryptorchidism", "hypertelorism", "global developmental delay", "short stature", "cognitive impairment", "proptosis", "premature birth", "respiratory insufficiency"], "tests": ["ANA panel", "RF/anti-CCP", "inflammatory markers (ESR/CRP)", "joint imaging"], "specialist": "rheumatologist", "relevance_score": 0},
    {"name": "Hypermobile Ehlers-Danlos syndrome", "category": "rheumatology", "symptoms": ["vertigo", "sleep disturbance", "hip dislocation", "joint pain", "fatigue", "osteoarthritis", "arrhythmia", "ptosis"], "tests": ["ANA panel", "RF/anti-CCP", "inflammatory markers (ESR/CRP)", "joint imaging"], "specialist": "rheumatologist", "relevance_score": 0},
    {"name": "Wolf-Hirschhorn syndrome", "category": "rare disease", "symptoms": ["downslanted palpebral fissures", "seizures", "ataxia", "hypotonia", "global developmental delay", "failure to thrive", "intellectual disability severe", "cryptorchidism"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Spastic paraplegia-facial-cutaneous lesions syndrome", "category": "rare disease", "symptoms": ["spasticity", "neurological speech impairment", "EEG abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Monosomy 5p syndrome", "category": "rare disease", "symptoms": ["microcephaly", "wide nasal bridge", "hypotonia", "intellectual disability severe", "severe global developmental delay", "hypertelorism", "downslanted palpebral fissures", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Early-onset epilepsy-intellectual disability-brain anomalies syndrome", "category": "neurology", "symptoms": ["seizures", "hypotonia", "severe global developmental delay", "delayed speech", "growth retardation", "abnormal facial shape", "hypertelorism", "autistic behavior"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Autosomal recessive spastic paraplegia type 11", "category": "rare disease", "symptoms": ["intellectual disability mild", "nystagmus", "ataxia", "dysphagia", "cerebral cortical atrophy", "basal ganglia abnormality", "strabismus", "visual impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Paraplegia-intellectual disability-hyperkeratosis syndrome", "category": "rare disease", "symptoms": ["intellectual disability", "spasticity", "micrognathia", "ptosis", "nystagmus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "TBCK-related encephalopathy-severe hypotonia-craniofacial dysmorphism syndrome", "category": "orthopedics", "symptoms": ["muscle weakness progressive", "delayed speech", "seizures", "abnormal facial shape", "respiratory insufficiency", "basal ganglia abnormality", "elevated cholesterol", "skeletal muscle atrophy"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Young-onset Parkinson disease", "category": "neurology", "symptoms": ["rigidity", "tremor", "postural instability", "spasticity", "dystonia", "cognitive impairment"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Spinocerebellar ataxia type 36", "category": "neurology", "symptoms": ["hearing loss", "ataxia", "skeletal muscle atrophy", "Babinski sign", "ptosis", "dysphagia", "vertigo", "attention deficit hyperactivity disorder"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Atrial septal defect, coronary sinus type", "category": "rare disease", "symptoms": ["fatigue", "pulmonary hypertension", "arrhythmia", "recurrent infections"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Atrial septal defect, ostium primum type", "category": "rare disease", "symptoms": ["left ventricular hypertrophy", "fatigue", "failure to thrive", "pulmonary hypertension"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Spastic paraplegia-nephritis-deafness syndrome", "category": "nephrology", "symptoms": ["proteinuria", "sensorineural hearing loss", "intellectual disability", "spasticity", "neurological speech impairment", "severe short stature", "growth delay"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "Duane retraction syndrome", "category": "rare disease", "symptoms": ["strabismus", "sensorineural hearing loss", "anteverted nares", "microcephaly", "micrognathia", "hearing loss", "wide nasal bridge", "ptosis"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Sickle cell anemia", "category": "hematology", "symptoms": ["hemolytic anemia", "susceptibility to infections", "reticulocytosis", "leukocytosis", "pulmonary hypertension", "iron deficiency anemia"], "tests": ["CBC with differential", "peripheral blood smear", "bone marrow biopsy", "flow cytometry"], "specialist": "hematologist", "relevance_score": 0},
    {"name": "Dubowitz syndrome", "category": "rare disease", "symptoms": ["micrognathia", "ptosis", "failure to thrive", "respiratory insufficiency", "susceptibility to infections", "delayed skeletal maturation", "growth delay", "attention deficit hyperactivity disorder"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Dubin-Johnson syndrome", "category": "rare disease", "symptoms": ["coagulation abnormality", "hepatomegaly", "fatigue"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Dopamine beta-hydroxylase deficiency", "category": "rare disease", "symptoms": ["anemia", "sleep disturbance", "elevated creatinine kinase", "fatigue", "hypotonia", "vertigo", "abnormal ECG"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Papilloma of choroid plexus", "category": "rare disease", "symptoms": ["visual impairment", "seizures", "neoplasm", "hemiplegia", "abnormal nervous system morphology", "cognitive impairment"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Parana hard skin syndrome", "category": "dermatology", "symptoms": ["growth retardation", "respiratory insufficiency", "short stature"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Spastic paraparesis-deafness syndrome", "category": "otolaryngology", "symptoms": ["sensorineural hearing loss", "visual impairment", "nystagmus", "ataxia", "short stature", "hemiplegia"], "tests": ["audiometry", "genetic testing", "vestibular testing"], "specialist": "otolaryngologist", "relevance_score": 0},
    {"name": "Pelizaeus-Merzbacher disease, classic form", "category": "rare disease", "symptoms": ["nystagmus", "delayed speech", "ataxia", "intellectual disability mild", "global developmental delay", "hypotonia", "spasticity", "dystonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Dihydropteridine reductase deficiency", "category": "rare disease", "symptoms": ["microcephaly", "intellectual disability", "global developmental delay", "dysphagia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Diphallia", "category": "rare disease", "symptoms": ["hypospadias", "cryptorchidism", "abnormal heart morphology", "atrial septal defect", "abnormality of the gastrointestinal tract"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Dermatomyositis", "category": "dermatology", "symptoms": ["respiratory insufficiency", "joint pain", "elevated liver enzymes", "elevated creatine kinase", "proximal muscle weakness", "fatigue", "myositis", "hypotonia"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "Arginine vasopressin resistance", "category": "rare disease", "symptoms": ["short stature", "global developmental delay", "failure to thrive", "renal insufficiency", "seizures", "growth retardation"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Null syndrome", "category": "rare disease", "symptoms": ["ataxia", "optic atrophy", "basal ganglia abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "8q22.1 microdeletion syndrome", "category": "rare disease", "symptoms": ["cryptorchidism", "long philtrum", "conductive hearing loss", "wide nasal bridge", "microcephaly", "global developmental delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Juvenile Paget disease", "category": "rare disease", "symptoms": ["hearing loss", "optic atrophy", "short stature"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked sideroblastic anemia and spinocerebellar ataxia", "category": "neurology", "symptoms": ["ataxia", "anemia", "neurological speech impairment", "strabismus", "nystagmus", "delayed speech", "postural instability", "delayed speech and language"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "W syndrome", "category": "rare disease", "symptoms": ["hypertelorism", "downslanted palpebral fissures", "spasticity", "global developmental delay", "lower limb pain"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Subacute sclerosing leukoencephalitis", "category": "rare disease", "symptoms": ["seizures", "dystonia", "sleep disturbance", "hypertension", "brain atrophy", "ataxia", "spasticity"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Pelizaeus-Merzbacher disease in female carriers", "category": "rare disease", "symptoms": ["hypertelorism", "nystagmus", "hypotonia", "growth retardation", "neurological speech impairment", "grand mal seizures", "developmental regression", "basal ganglia abnormality"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Isolated Dandy-Walker malformation", "category": "rare disease", "symptoms": ["intellectual disability", "motor delay", "nystagmus"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "X-linked acrogigantism", "category": "rare disease", "symptoms": ["seizures", "ataxia", "headache", "abdominal distension", "intellectual disability"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Trichothiodystrophy", "category": "rare disease", "symptoms": ["global developmental delay", "cryptorchidism", "microcephaly", "hypertelorism", "strabismus", "nystagmus", "spasticity", "hypotonia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Fibrous dysplasia of bone", "category": "orthopedics", "symptoms": ["osteolysis", "reduced bone mineral density", "short stature", "hearing loss", "paresthesia", "neoplasm of the breast"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Primary ciliary dyskinesia", "category": "dermatology", "symptoms": ["hearing loss", "delayed speech", "abnormal heart morphology", "bronchiectasis"], "tests": ["skin biopsy", "genetic testing", "dermoscopy"], "specialist": "dermatologist", "relevance_score": 0},
    {"name": "46,XX gonadal dysgenesis", "category": "rare disease", "symptoms": ["delayed skeletal maturation", "hearing loss", "short stature", "microcephaly", "ataxia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Postaxial acrofacial dysostosis", "category": "orthopedics", "symptoms": ["strabismus", "micrognathia", "downslanted palpebral fissures", "lower limb pain"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "Nager syndrome", "category": "rare disease", "symptoms": ["micrognathia", "hearing loss", "downslanted palpebral fissures", "delayed speech", "ptosis", "respiratory insufficiency"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Léri-Weill dyschondrosteosis", "category": "orthopedics", "symptoms": ["wide nasal bridge", "lower limb pain", "hip abnormality", "growth delay", "osteosarcoma"], "tests": ["X-ray/MRI skeleton", "bone density scan", "genetic testing"], "specialist": "orthopedic surgeon", "relevance_score": 0},
    {"name": "STXBP1-related encephalopathy", "category": "neurology", "symptoms": ["intellectual disability", "seizures", "global developmental delay", "autistic behavior", "delayed speech", "ataxia", "hypotonia", "tremor"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Pituitary carcinoma", "category": "oncology", "symptoms": ["headache", "hearing loss", "ataxia", "abnormal central motor function"], "tests": ["biopsy", "PET scan", "tumor markers", "genetic testing"], "specialist": "oncologist", "relevance_score": 0},
    {"name": "DPM3-CDG", "category": "rare disease", "symptoms": ["muscle weakness", "elevated liver enzymes", "Babinski sign", "muscular dystrophy"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Scleromyxedema", "category": "rare disease", "symptoms": ["dysphagia", "joint pain", "elevated creatine kinase", "kidney abnormality", "seizures", "abnormality of the gastrointestinal tract"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Primary non-essential cutis verticis gyrata", "category": "rare disease", "symptoms": ["seizures", "global developmental delay", "microcephaly", "ventricular septal defect", "atrial septal defect"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Reticular dysgenesis", "category": "rare disease", "symptoms": ["hearing loss", "failure to thrive", "thrombocytopenia", "anemia"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Trisomy 9p syndrome", "category": "rare disease", "symptoms": ["microcephaly", "hypertelorism", "downslanted palpebral fissures", "intellectual disability", "global developmental delay", "kyphosis", "growth delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Dyggve-Melchior-Clausen disease", "category": "rare disease", "symptoms": ["intellectual disability", "failure to thrive", "microcephaly", "motor delay", "short long bones", "hip abnormality", "severe short stature", "intellectual disability severe"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Robinow syndrome", "category": "rare disease", "symptoms": ["short stature", "hypertelorism", "cryptorchidism", "hydronephrosis", "long philtrum", "micrognathia", "conductive hearing loss", "anteverted nares"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Temple syndrome due to paternal 14q32.2 hypomethylation", "category": "rare disease", "symptoms": ["small hand", "micrognathia", "autistic behavior", "intellectual disability", "hypercholesterolemia", "delayed speech", "hypotonia", "motor delay"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Renal hypoplasia, bilateral", "category": "nephrology", "symptoms": ["proteinuria", "renal cyst", "cryptorchidism", "nystagmus", "failure to thrive", "growth retardation", "premature birth", "anemia"], "tests": ["urine protein/creatinine", "renal biopsy", "kidney ultrasound"], "specialist": "nephrologist", "relevance_score": 0},
    {"name": "COG5-CDG", "category": "rare disease", "symptoms": ["delayed speech", "motor delay", "microcephaly", "conductive hearing loss", "short stature", "intellectual disability severe", "cryptorchidism", "sensorineural hearing loss"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Kagami-Ogata syndrome due to maternal 14q32.2 hypermethylation", "category": "rare disease", "symptoms": ["intellectual disability mild", "global developmental delay", "hypotonia", "abnormal heart morphology", "ventricular septal defect", "delayed speech and language"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Alpha-B crystallin-related late-onset myopathy", "category": "rare disease", "symptoms": ["dysphagia", "axial muscle weakness", "proximal muscle weakness"], "tests": ["whole exome sequencing", "metabolic panel", "genetic consultation"], "specialist": "rare disease specialist", "relevance_score": 0},
    {"name": "Congenital muscular dystrophy with intellectual disability and severe epilepsy", "category": "neurology", "symptoms": ["micrognathia", "global developmental delay", "hypotonia", "failure to thrive", "abnormal facial shape", "generalized myoclonic seizures", "elevated creatine kinase", "strabismus"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
    {"name": "Caribbean parkinsonism", "category": "neurology", "symptoms": ["rigidity", "dystonia", "cerebral cortical atrophy", "postural instability", "sleep disturbance", "proximal muscle weakness"], "tests": ["MRI brain/spine", "nerve conduction studies", "EEG", "CSF analysis"], "specialist": "neurologist", "relevance_score": 0},
]

# ── Merge ORPHA diseases with original DISEASE_DB ─────────────────
existing_names = {d["name"].lower() for d in DISEASE_DB}

added = 0
for od in ORPHA_DISEASES:
    if od["name"].lower() not in existing_names:
        # Ensure required keys match DISEASE_DB format
        entry = {
            "name":            od["name"],
            "category":        od["category"],
            "symptoms":        od["symptoms"],
            "tests":           od.get("tests", ["genetic testing", "specialist evaluation"]),
            "specialist":      od.get("specialist", "rare disease specialist"),
            "misdiagnoses":    ["common conditions", "functional disorder"],
            "avg_delay_years": 5,
            "relevance_score": 0,
            "keywords":        od["symptoms"][:5],
        }
        DISEASE_DB.append(entry)
        existing_names.add(od["name"].lower())
        added += 1

print(f"✅ ORPHA expansion complete!")
print(f"   Added {added} new diseases from ORPHANET/HPO database")
print(f"   Total DISEASE_DB size: {len(DISEASE_DB)} rare diseases")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Semantic RAG with Sentence Transformers + FAISS
# Replaces keyword matching with true semantic understanding
# ════════════════════════════════════════════════════════════════

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

print("⏳ Loading semantic embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def disease_to_text(d):
    # Support both original format (key_symptoms/typical_misdiagnoses)
    # and ORPHA format (symptoms/misdiagnoses)
    syms = d.get("key_symptoms") or d.get("symptoms") or []
    misdx = d.get("typical_misdiagnoses") or d.get("misdiagnoses") or []
    return (
        f"{d['name']} {d.get('category', 'rare disease')} "
        f"symptoms: {' '.join(syms)} "
        f"misdiagnosed as: {' '.join(misdx)}"
    )

print(f"⏳ Encoding all {len(DISEASE_DB)} diseases into semantic vectors...")
disease_texts      = [disease_to_text(d) for d in DISEASE_DB]
disease_embeddings = embedder.encode(
    disease_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

DIM   = disease_embeddings.shape[1]
index = faiss.IndexFlatIP(DIM)   # cosine similarity via inner product
index.add(disease_embeddings.astype(np.float32))

def retrieve_diseases(patient_text, top_k=5):
    """Semantic search — understands meaning, not just keywords."""
    query_vec = embedder.encode([patient_text], normalize_embeddings=True)
    scores, indices = index.search(query_vec.astype(np.float32), top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        d = DISEASE_DB[idx].copy()
        d["relevance_score"] = round(float(score) * 100, 1)
        results.append(d)
    return results

# ── Smoke test ───────────────────────────────────────────────────
test_query = "fatigue joint pain heart racing when standing up fainting brain fog"
test_results = retrieve_diseases(test_query, top_k=3)

print(f"\n✅ Semantic RAG ready!")
print(f"   Embedding model : all-MiniLM-L6-v2 | Dimension: {DIM}")
print(f"   Diseases indexed: {len(DISEASE_DB)}")
print(f"\n🧪 Test query: '{test_query}'")
for d in test_results:
    print(f"   {d['relevance_score']:5.1f}%  →  {d['name']}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4b — Hybrid RAG: BM25 + FAISS + Cross-Encoder Reranking
# Run AFTER Cell 4. Upgrades retrieve_diseases to a 3-stage pipeline.
# Research shows this gives +17% better retrieval over FAISS alone.
# ════════════════════════════════════════════════════════════════

!pip install rank_bm25 -q

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import numpy as np, math

print("⏳ Building BM25 sparse index...")
tokenized_corpus = [text.lower().split() for text in disease_texts]
bm25_index = BM25Okapi(tokenized_corpus)

print("⏳ Loading cross-encoder reranker (ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

def retrieve_diseases_hybrid(patient_text, top_k=5, pool=20):
    """
    3-stage retrieval pipeline:
      Stage 1a — Dense (FAISS cosine similarity on sentence embeddings)
      Stage 1b — Sparse (BM25 keyword matching)
      Stage 2  — Reciprocal Rank Fusion to merge both rankings
      Stage 3  — Cross-encoder reranking for final precision
    """
    # ── Stage 1a: Dense FAISS ────────────────────────────────────
    q_vec = embedder.encode([patient_text], normalize_embeddings=True)
    dense_scores, dense_idxs = index.search(q_vec.astype(np.float32), pool)

    # ── Stage 1b: BM25 Sparse ────────────────────────────────────
    bm25_scores = bm25_index.get_scores(patient_text.lower().split())
    bm25_idxs   = np.argsort(bm25_scores)[::-1][:pool]

    # ── Stage 2: Reciprocal Rank Fusion ──────────────────────────
    K = 60  # standard RRF constant
    rrf = {}
    for rank, idx in enumerate(dense_idxs[0]):
        rrf[int(idx)] = rrf.get(int(idx), 0) + 1 / (K + rank + 1)
    for rank, idx in enumerate(bm25_idxs):
        rrf[int(idx)] = rrf.get(int(idx), 0) + 1 / (K + rank + 1)

    candidates = sorted(rrf, key=rrf.get, reverse=True)[:pool]

    # ── Stage 3: Cross-Encoder Reranking ─────────────────────────
    pairs     = [[patient_text, disease_texts[i]] for i in candidates]
    ce_scores = reranker.predict(pairs, show_progress_bar=False)
    ranked    = sorted(zip(candidates, ce_scores), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for idx, ce_score in ranked:
        d = DISEASE_DB[idx].copy()
        # Sigmoid-normalise cross-encoder logit → 0–100%
        d["relevance_score"] = round(100 / (1 + math.exp(-ce_score / 3)), 1)
        results.append(d)
    return results

# Override retrieve_diseases globally with the hybrid version
retrieve_diseases = retrieve_diseases_hybrid

# ── Smoke test ───────────────────────────────────────────────────
test = retrieve_diseases("heart racing when standing fainting brain fog extreme fatigue")
print(f"\n✅ Hybrid RAG ready! (BM25 + FAISS + Cross-Encoder)")
print(f"   Model: cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"   Test: 'heart racing when standing fainting brain fog'")
for d in test:
    print(f"   {d['relevance_score']:5.1f}%  →  {d['name']}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Prompt builder + Diagnostic engine
# ════════════════════════════════════════════════════════════════

import torch

def build_prompt(patient_data, retrieved_diseases):
    disease_context = ""
    for d in retrieved_diseases:
        score_str = f" (semantic match: {d.get('relevance_score', '?')}%)" if 'relevance_score' in d else ""
        disease_context += (
            f"\n• {d['name']} ({d['category']}){score_str} — avg delay: {d['avg_diagnosis_time']}\n"
            f"  Symptoms   : {', '.join(d['key_symptoms'])}\n"
            f"  Missed as  : {', '.join(d['typical_misdiagnoses'])}\n"
            f"  Key tests  : {', '.join(d['diagnostic_tests'])}\n"
            f"  See        : {d['specialist']}\n"
        )

    prompt = f"""You are a rare disease research assistant helping a patient who has been struggling to get answers. \
You do NOT diagnose — you help them understand what conditions may be worth investigating and how to advocate for themselves.

RARE DISEASE DATABASE — SEMANTICALLY MATCHED CONDITIONS:
{disease_context}

PATIENT HISTORY:
- Age: {patient_data.get('age', 'Not provided')} | Sex: {patient_data.get('sex', 'Not provided')}
- Symptom duration: {patient_data.get('duration', 'Not provided')}
- Symptoms: {patient_data.get('symptoms', 'See uploaded record')}
- Previous diagnoses: {patient_data.get('prev_diagnoses', 'Nothing conclusive')}
- Tests already done: {patient_data.get('tests_done', 'None mentioned')}
- Family history: {patient_data.get('family_history', 'None known')}
- Additional notes: {patient_data.get('notes', 'None')}

IMPORTANT: Start your response DIRECTLY with "1." — no preamble, no disclaimer, no introduction.

1. 🔍 TOP CONDITIONS WORTH INVESTIGATING
(List 3-4 conditions with clear reasoning based on their specific symptoms)

2. 🧪 SPECIFIC TESTS TO REQUEST FROM YOUR DOCTOR
(Give exact test names they should ask for)

3. 👨\u200d⚕️ SPECIALIST TO SEE NEXT
(Which type of specialist and why)

4. ❓ KEY QUESTIONS TO BRING TO YOUR APPOINTMENT
(List 5 specific questions to ask their doctor)

5. 📋 PATTERNS IN YOUR HISTORY THAT STAND OUT
(What is medically significant about this patient's case)"""

    return prompt


def run_diagnostic(patient_data):
    """Run rare disease analysis using Gemma 4 + Semantic RAG."""

    # ── Semantic RAG retrieval ────────────────────────────────────
    search_text = (
        f"{patient_data.get('symptoms', '')} "
        f"{patient_data.get('prev_diagnoses', '')} "
        f"{patient_data.get('notes', '')}"
    ).strip() or "rare disease chronic fatigue"

    retrieved = retrieve_diseases(search_text, top_k=5)
    prompt    = build_prompt(patient_data, retrieved)

    # ── Build messages (vision or text) ──────────────────────────
    image_b64  = patient_data.get("image_base64")
    image_mime = patient_data.get("image_mime")

    if image_b64 and image_mime:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:{image_mime};base64,{image_b64}"}},
                {"type": "text",
                 "text": f"Please read this medical record carefully, then:\n\n{prompt}"}
            ]
        }]
    else:
        messages = [{"role": "user", "content": prompt}]

    # ── Gemma 4 inference ─────────────────────────────────────────
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors        = "pt",
        return_dict           = True,
        add_generation_prompt = True,   # ← CRITICAL: without this model returns 1 token
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 900,
            temperature        = 0.7,
            do_sample          = True,
            top_p              = 0.9,
            repetition_penalty = 1.1,
        )

    input_len = inputs["input_ids"].shape[-1]
    response  = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )
    return response.strip()


print("✅ Diagnostic engine ready!")
print("   run_diagnostic(patient_data) — Semantic RAG + Gemma 4")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5b — Gemma 4 Native Function Calling (Agentic Loop)
# Run AFTER Cell 5. Overrides run_diagnostic with an agentic version
# where Gemma 4 decides which tools to call during reasoning.
# This uses Gemma 4's native function calling — a key hackathon feature.
# ════════════════════════════════════════════════════════════════

import json, re, torch

# ── Tool definitions ──────────────────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_rare_diseases",
            "description": "Search the rare disease database to find conditions matching the patient's symptoms. Always call this first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query":  {"type": "string",  "description": "Patient symptom description to search for"},
                    "top_k":  {"type": "integer", "description": "Number of diseases to return (default 5)"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_diagnostic_tests",
            "description": "Get the specific diagnostic tests for a named rare disease",
            "parameters": {
                "type": "object",
                "properties": {
                    "disease_name": {"type": "string", "description": "Disease name (partial match OK)"}
                },
                "required": ["disease_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_specialist_referral",
            "description": "Get the recommended specialist type for a named rare disease",
            "parameters": {
                "type": "object",
                "properties": {
                    "disease_name": {"type": "string", "description": "Disease name"}
                },
                "required": ["disease_name"]
            }
        }
    }
]

# ── Tool execution ─────────────────────────────────────────────
def execute_tool(name, args):
    if name == "search_rare_diseases":
        diseases = retrieve_diseases(args.get("query", ""), top_k=args.get("top_k", 5))
        return json.dumps([{
            "name":         d["name"],
            "category":     d["category"],
            "relevance_pct":d.get("relevance_score", 0),
            "key_symptoms": d["key_symptoms"],
            "missed_as":    d["typical_misdiagnoses"],
            "tests":        d["diagnostic_tests"],
            "specialist":   d["specialist"],
            "avg_delay":    d["avg_diagnosis_time"]
        } for d in diseases])

    elif name == "get_diagnostic_tests":
        q = args.get("disease_name", "").lower()
        for d in DISEASE_DB:
            if q in d["name"].lower():
                return json.dumps({"disease": d["name"], "tests": d["diagnostic_tests"]})
        return json.dumps({"error": "Not found"})

    elif name == "get_specialist_referral":
        q = args.get("disease_name", "").lower()
        for d in DISEASE_DB:
            if q in d["name"].lower():
                return json.dumps({"disease": d["name"], "specialist": d["specialist"]})
        return json.dumps({"error": "Not found"})

    return json.dumps({"error": f"Unknown tool: {name}"})

# ── Tool call parser ───────────────────────────────────────────
def parse_tool_calls(text):
    """Parse Gemma 4 function call output in multiple formats."""
    calls = []
    for pattern in [
        r'<tool_call>(.*?)</tool_call>',
        r'```(?:json)?\s*(\{[^`]*"name"[^`]*\})\s*```',
        r'\{[^{}]*"name"\s*:\s*"[^"]+"[^{}]*"arguments"\s*:\s*\{[^{}]+\}[^{}]*\}'
    ]:
        for m in re.findall(pattern, text, re.DOTALL):
            try:
                data = json.loads(m.strip())
                if "name" in data:
                    calls.append(data)
            except:
                pass
        if calls:
            break
    return calls

# ── Agentic run_diagnostic ─────────────────────────────────────
def run_diagnostic_agentic(patient_data):
    """
    Agentic diagnostic loop:
    1. Gemma 4 calls search_rare_diseases autonomously
    2. Results fed back as tool responses
    3. Gemma 4 calls get_diagnostic_tests for top matches
    4. Final structured analysis generated
    """
    symptoms       = patient_data.get("symptoms", "")
    age            = patient_data.get("age", "unknown")
    sex            = patient_data.get("sex", "unknown")
    duration       = patient_data.get("duration", "unknown")
    prev_diagnoses = patient_data.get("prev_diagnoses", "none")
    notes          = patient_data.get("notes", "")
    image_b64      = patient_data.get("image_base64")
    image_mime     = patient_data.get("image_mime")

    user_content = (
        f"Patient: {age}yo {sex}, symptoms for {duration}.\n"
        f"Symptoms: {symptoms}\n"
        f"Previously told: {prev_diagnoses}\n"
        f"Notes: {notes}\n\n"
        f"Search the rare disease database for conditions that match, "
        f"then get diagnostic tests for the top 2 matches, "
        f"then give a full structured analysis."
    )

    if image_b64 and image_mime:
        content = [
            {"type": "image_url", "image_url": {"url": f"data:{image_mime};base64,{image_b64}"}},
            {"type": "text", "text": f"Read this medical record, then:\n{user_content}"}
        ]
    else:
        content = user_content

    messages = [{"role": "user", "content": content}]
    tools_called = []

    for _ in range(5):  # max 5 agentic rounds
        try:
            inputs = tokenizer.apply_chat_template(
                messages,
                tools=TOOLS,
                return_tensors="pt",
                return_dict=True,
                add_generation_prompt=True,
            ).to(model.device)
        except Exception:
            # Fallback: some tokenizer versions don't support tools param
            inputs = tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                return_dict=True,
                add_generation_prompt=True,
            ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=700,
                temperature=0.4,
                do_sample=True,
                repetition_penalty=1.1,
            )

        input_len = inputs["input_ids"].shape[-1]
        reply     = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

        tool_calls = parse_tool_calls(reply)
        if not tool_calls:
            # No more tool calls — this is the final answer
            print(f"   🔧 Tools used: {tools_called}")
            return reply

        messages.append({"role": "assistant", "content": reply})
        for call in tool_calls:
            name   = call.get("name", "")
            args   = call.get("arguments", call.get("parameters", {}))
            result = execute_tool(name, args)
            tools_called.append(name)
            messages.append({"role": "tool", "name": name, "content": result})

    # All rounds used — force final answer
    messages.append({"role": "user", "content":
        "Based on the tool results, give your complete structured analysis now."})
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=900, temperature=0.7, do_sample=True)
    input_len = inputs["input_ids"].shape[-1]
    print(f"   🔧 Tools used: {tools_called}")
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

# Override run_diagnostic globally
run_diagnostic = run_diagnostic_agentic

print("✅ Gemma 4 Agentic Function Calling ready!")
print("   Tools: search_rare_diseases | get_diagnostic_tests | get_specialist_referral")
print("   Gemma 4 will autonomously decide which tools to call during reasoning")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — ⭐ UNSLOTH / PEFT FINE-TUNING ON GEMMA 4
# ════════════════════════════════════════════════════════════════

SKIP_FINE_TUNING = False   # ← Set True to skip

if not SKIP_FINE_TUNING:
    import gc, torch
    from datasets import Dataset
    from transformers import TrainingArguments
    from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
    from trl import SFTTrainer

    !pip install trl peft accelerate -q

    # ── Step 1: Prepare quantised model for training ──────────────
    # Gemma 4 uses Gemma4ClippableLinear wrappers around Linear4bit.
    # We unwrap them so PEFT can see the inner Linear4bit it supports.
    print("⚙ Unwrapping Gemma4ClippableLinear layers for PEFT compatibility...")

    def unwrap_gemma4_linears(m):
        for name, module in list(m.named_modules()):
            if type(module).__name__ == "Gemma4ClippableLinear" and hasattr(module, "linear"):
                parts = name.split(".")
                parent = m
                for part in parts[:-1]:
                    parent = parent[int(part)] if part.isdigit() else getattr(parent, part)
                setattr(parent, parts[-1], module.linear)
        return m

    ft_model = unwrap_gemma4_linears(model)
    ft_model = prepare_model_for_kbit_training(ft_model)
    print("✅ Layers unwrapped and model prepared for k-bit training")

    # ── Step 2: Apply LoRA adapters ───────────────────────────────
    lora_cfg = LoraConfig(
        r=16, lora_alpha=16, lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
    )
    ft_model     = get_peft_model(ft_model, lora_cfg)
    ft_tokenizer = tokenizer
    ft_model.print_trainable_parameters()
    print("✅ LoRA adapters applied to Gemma 4")

    # ── Step 3: Build training data ───────────────────────────────
    def make_training_data(db):
        examples, skipped = [], 0
        for d in db:
            syms   = d.get("key_symptoms") or d.get("symptoms") or []
            missed = d.get("typical_misdiagnoses") or d.get("misdiagnoses") or ["common conditions"]
            tests  = d.get("diagnostic_tests") or d.get("tests") or ["genetic testing"]
            spec   = d.get("specialist") or "rare disease specialist"
            name   = d.get("name", "Unknown")
            cat    = d.get("category", "rare disease")
            delay  = d.get("avg_diagnosis_time") or d.get("avg_delay_years") or "5+ years"
            if len(syms) < 3:
                skipped += 1
                continue
            sym_s = ", ".join(syms[:4])
            mis_s = ", ".join(missed[:2])
            tst_s = "\n".join([f"- {t}" for t in tests[:4]])
            for prompt in [
                f"Patient has {sym_s}. Previously told: {mis_s}. What rare condition fits?",
                f"I have {sym_s}. Doctors say {mis_s} but nothing helps. What is it really?",
            ]:
                response = (
                    f"1. TOP CONDITIONS\n{name} ({cat}) — avg delay {delay}.\n\n"
                    f"2. TESTS TO REQUEST\n{tst_s}\n\n"
                    f"3. SPECIALIST\n{spec}\n\n"
                    f"4. KEY QUESTIONS\n- Have you considered {name}?\n"
                    f"- Can we run {tests[0]}?\n\n"
                    f"5. PATTERN\nCombination of {sym_s} is characteristic of {name}."
                )
                text = (
                    f"<start_of_turn>user\n{prompt}<end_of_turn>\n"
                    f"<start_of_turn>model\n{response}<end_of_turn>"
                )
                examples.append({"text": text})
        print(f"   Used {len(db)-skipped} diseases | Skipped {skipped} (too few symptoms)")
        return examples

    print("⏳ Building training dataset...")
    train_data = make_training_data(DISEASE_DB)
    dataset    = Dataset.from_list(train_data)
    print(f"✅ {len(train_data)} training examples ready")

    # ── Step 4: Train ─────────────────────────────────────────────
    training_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps        = 5,
        num_train_epochs    = 3,
        learning_rate       = 2e-4,
        fp16                = not torch.cuda.is_bf16_supported(),
        bf16                = torch.cuda.is_bf16_supported(),
        logging_steps       = 10,
        optim               = "adamw_8bit",
        weight_decay        = 0.01,
        lr_scheduler_type   = "linear",
        output_dir          = "./rare_disease_gemma4",
        report_to           = "none",
    )
    try:
        trainer = SFTTrainer(
            model=ft_model, tokenizer=ft_tokenizer,
            train_dataset=dataset, dataset_text_field="text",
            max_seq_length=2048, dataset_num_proc=2, args=training_args,
        )
    except TypeError:
        trainer = SFTTrainer(
            model=ft_model, tokenizer=ft_tokenizer,
            train_dataset=dataset, max_seq_length=2048, args=training_args,
        )

    print("🚀 Fine-tuning Gemma 4 on rare disease data (~45 min on A100)...")
    stats = trainer.train()
    print(f"\n✅ Fine-tuning complete! Loss: {stats.training_loss:.4f}")
    print(f"   Time: {stats.metrics['train_runtime']:.0f}s")

    # ── Step 5: Save & switch to fine-tuned model ─────────────────
    ft_model.save_pretrained("rare_disease_gemma4_lora")
    ft_tokenizer.save_pretrained("rare_disease_gemma4_lora")
    print("✅ LoRA weights saved → rare_disease_gemma4_lora/")

    model     = ft_model
    tokenizer = ft_tokenizer
    print("✅ Now using fine-tuned Gemma 4 for all inference")

else:
    print("⏭  Fine-tuning skipped — using base Gemma 4")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — Write UI file (automatic — no upload needed)
# ════════════════════════════════════════════════════════════════

UI_HTML_CONTENT = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>The Diagnostic Odyssey Ender — Gemma 4</title>
  <style>
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
    body { background: #080b14; color: #e2e8f0; font-family: -apple-system, BlinkMacSystemFont, 'Inter', 'Segoe UI', sans-serif; }

    @keyframes spin    { to { transform: rotate(360deg); } }
    @keyframes fadeUp  { from { opacity:0; transform:translateY(20px); } to { opacity:1; transform:translateY(0); } }
    @keyframes pulse   { 0%,100%{opacity:1;} 50%{opacity:0.4;} }
    @keyframes gradMove{ 0%{background-position:0% 50%;} 50%{background-position:100% 50%;} 100%{background-position:0% 50%;} }
    @keyframes shimmer { from{left:-100%;} to{left:200%;} }
    @keyframes countUp { from{opacity:0;transform:scale(0.8);} to{opacity:1;transform:scale(1);} }

    .fade-up   { animation: fadeUp 0.5s ease forwards; }
    .fade-up-2 { animation: fadeUp 0.5s ease 0.1s forwards; opacity:0; }
    .fade-up-3 { animation: fadeUp 0.5s ease 0.2s forwards; opacity:0; }

    input, textarea, select {
      width:100%; background:rgba(255,255,255,0.04); border:1px solid rgba(255,255,255,0.08);
      border-radius:10px; padding:11px 14px; color:#e2e8f0; font-size:14px;
      outline:none; transition:all 0.2s; resize:vertical; font-family:inherit;
    }
    input:focus, textarea:focus, select:focus {
      border-color:#6c8ef5; background:rgba(108,142,245,0.06); box-shadow:0 0 0 3px rgba(108,142,245,0.12);
    }
    input::placeholder, textarea::placeholder { color:#4a5568; }
    select option { background:#1a1d2e; }
    label { display:block; font-size:12px; font-weight:600; color:#718096; text-transform:uppercase; letter-spacing:0.05em; margin-bottom:6px; }
    .field { margin-bottom:16px; }
    .hint  { font-size:11px; color:#4a5568; margin-top:4px; }

    .stat-card:hover { transform:translateY(-3px); transition:transform 0.2s; }
    .submit-btn { position:relative; overflow:hidden; }
    .submit-btn::after {
      content:''; position:absolute; top:0; left:-100%; width:60%;
      height:100%; background:linear-gradient(90deg,transparent,rgba(255,255,255,0.15),transparent);
      animation:shimmer 2s infinite; transform:skewX(-20deg);
    }
    .submit-btn:hover:not(:disabled) { transform:translateY(-2px); box-shadow:0 8px 25px rgba(108,142,245,0.4); }
    .submit-btn:disabled { opacity:0.5; cursor:not-allowed; }
    .submit-btn:disabled::after { display:none; }

    .result-section { animation:fadeUp 0.4s ease forwards; }
    .copy-btn:hover  { background:rgba(108,142,245,0.2) !important; }

    .upload-zone { border:2px dashed rgba(108,142,245,0.3); border-radius:14px; padding:28px 20px; text-align:center; cursor:pointer; transition:all 0.25s; background:rgba(108,142,245,0.03); }
    .upload-zone:hover, .upload-zone.drag-over { border-color:#6c8ef5; background:rgba(108,142,245,0.08); transform:scale(1.01); }
    .upload-zone.has-file { border-color:#34d399; background:rgba(52,211,153,0.06); }
    .file-preview { display:flex; align-items:center; gap:12px; padding:12px 16px; background:rgba(52,211,153,0.08); border:1px solid rgba(52,211,153,0.25); border-radius:10px; margin-top:10px; }

    .mesh-bg {
      position:absolute; inset:0; opacity:0.4;
      background-image:radial-gradient(circle at 20% 50%,rgba(108,142,245,0.15) 0%,transparent 50%),
                       radial-gradient(circle at 80% 20%,rgba(167,139,250,0.15) 0%,transparent 50%),
                       radial-gradient(circle at 50% 80%,rgba(52,211,153,0.08) 0%,transparent 40%);
    }
    .grid-bg {
      position:absolute; inset:0; opacity:0.06;
      background-image:linear-gradient(rgba(108,142,245,0.5) 1px,transparent 1px),
                       linear-gradient(90deg,rgba(108,142,245,0.5) 1px,transparent 1px);
      background-size:40px 40px;
    }

    @media(max-width:768px){
      .main-grid  { grid-template-columns:1fr !important; }
      .stats-grid { grid-template-columns:1fr 1fr !important; }
      .hero-title { font-size:32px !important; }
    }
  </style>
</head>
<body>

<div style="min-height:100vh; background:#080b14;">

  <!-- HERO -->
  <div style="position:relative; overflow:hidden; padding-bottom:60px;">
    <div class="mesh-bg"></div>
    <div class="grid-bg"></div>
    <div style="position:relative; max-width:900px; margin:0 auto; padding:60px 24px 0; text-align:center;">

      <!-- Badge -->
      <div class="fade-up" style="display:inline-flex; align-items:center; gap:8px; background:rgba(108,142,245,0.1); border:1px solid rgba(108,142,245,0.25); border-radius:24px; padding:6px 16px; font-size:12px; font-weight:700; color:#6c8ef5; letter-spacing:0.08em; margin-bottom:24px;">
        <span style="width:7px; height:7px; border-radius:50%; background:#6c8ef5; animation:pulse 1.5s infinite; display:inline-block;"></span>
        GEMMA 4 HACKATHON · HEALTH &amp; SCIENCES
      </div>

      <!-- Title -->
      <h1 class="hero-title fade-up-2" style="font-size:52px; font-weight:900; line-height:1.1; margin-bottom:20px; background:linear-gradient(135deg,#c7d2fe 0%,#a78bfa 40%,#6c8ef5 100%); -webkit-background-clip:text; -webkit-text-fill-color:transparent; letter-spacing:-0.02em;">
        The Diagnostic<br>Odyssey Ender
      </h1>

      <p class="fade-up-3" style="font-size:18px; color:#94a3b8; max-width:520px; margin:0 auto 32px; line-height:1.7;">
        For people who've seen doctor after doctor with no answers. Gemma 4 cross-references rare disease patterns to help you find your next step.
      </p>

      <!-- Disclaimer -->
      <div class="fade-up-3" style="display:inline-flex; align-items:center; gap:8px; background:rgba(251,191,36,0.07); border:1px solid rgba(251,191,36,0.2); border-radius:10px; padding:8px 16px; font-size:13px; color:#fbbf24;">
        ⚠ This tool helps you advocate for yourself — it does not replace a medical diagnosis
      </div>
    </div>

    <!-- Stats -->
    <div style="max-width:800px; margin:48px auto 0; padding:0 24px;">
      <div class="stats-grid" style="display:grid; grid-template-columns:repeat(4,1fr); gap:12px;">
        <div class="stat-card" style="text-align:center; padding:20px 16px; background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07); border-radius:14px;">
          <div style="font-size:26px; font-weight:800; color:#a78bfa; margin-bottom:4px; animation:countUp 0.6s ease;">300M+</div>
          <div style="font-size:12px; color:#718096; line-height:1.4;">People with rare diseases globally</div>
        </div>
        <div class="stat-card" style="text-align:center; padding:20px 16px; background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07); border-radius:14px;">
          <div style="font-size:26px; font-weight:800; color:#6c8ef5; margin-bottom:4px; animation:countUp 0.6s ease;">6 yrs</div>
          <div style="font-size:12px; color:#718096; line-height:1.4;">Average time to get a diagnosis</div>
        </div>
        <div class="stat-card" style="text-align:center; padding:20px 16px; background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07); border-radius:14px;">
          <div style="font-size:26px; font-weight:800; color:#34d399; margin-bottom:4px; animation:countUp 0.6s ease;">50+</div>
          <div style="font-size:12px; color:#718096; line-height:1.4;">Rare conditions in our database</div>
        </div>
        <div class="stat-card" style="text-align:center; padding:20px 16px; background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07); border-radius:14px;">
          <div style="font-size:26px; font-weight:800; color:#f87171; margin-bottom:4px; animation:countUp 0.6s ease;">7+</div>
          <div style="font-size:12px; color:#718096; line-height:1.4;">Doctors seen before diagnosis</div>
        </div>
      </div>
    </div>
  </div>

  <!-- MAIN CONTENT -->
  <div style="max-width:1200px; margin:0 auto; padding:40px 24px 80px;">
    <div class="main-grid" style="display:grid; grid-template-columns:1fr 1fr; gap:24px; align-items:start;">

      <!-- LEFT: Form -->
      <div style="background:rgba(255,255,255,0.02); border:1px solid rgba(255,255,255,0.07); border-radius:20px; padding:32px;">

        <div style="margin-bottom:28px;">
          <h2 style="font-size:18px; font-weight:700; color:#e2e8f0; margin-bottom:6px;">📋 Your Medical History</h2>
          <p style="font-size:13px; color:#718096;">The more detail you give, the better the analysis</p>
          <div style="margin-top:14px;">
            <div style="display:flex; justify-content:space-between; margin-bottom:6px;">
              <span style="font-size:11px; color:#718096;">Profile completeness</span>
              <span id="progress-label" style="font-size:11px; color:#6c8ef5; font-weight:600;">0%</span>
            </div>
            <div style="height:4px; background:rgba(255,255,255,0.06); border-radius:4px; overflow:hidden;">
              <div id="progress-bar" style="height:100%; width:0%; background:linear-gradient(90deg,#6c8ef5,#a78bfa); border-radius:4px; transition:width 0.4s ease;"></div>
            </div>
          </div>
        </div>

        <!-- Basic info -->
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:14px; margin-bottom:4px;">
          <div class="field">
            <label for="age">Age</label>
            <input type="text" id="age" placeholder="e.g. 32" oninput="updateProgress()" />
          </div>
          <div class="field">
            <label for="sex">Sex</label>
            <select id="sex" onchange="updateProgress()">
              <option>Female</option>
              <option>Male</option>
              <option>Non-binary / Other</option>
              <option>Prefer not to say</option>
            </select>
          </div>
        </div>

        <div class="field">
          <label for="duration">How long have symptoms lasted?</label>
          <input type="text" id="duration" placeholder="e.g. 3 years, since age 15, 8 months..." oninput="updateProgress()" />
        </div>

        <div class="field">
          <label for="symptoms">All your symptoms <span style="color:#6c8ef5;">*</span></label>
          <textarea id="symptoms" rows="4" placeholder="Be as detailed as possible — fatigue, joint pain, rashes, brain fog, GI issues, fainting, hair loss..." oninput="updateProgress(); updateSubmitBtn();"></textarea>
          <p class="hint">★ Most important field — describe every symptom even if they seem unrelated</p>
        </div>

        <div style="border-top:1px solid rgba(255,255,255,0.06); margin:20px 0; display:flex; align-items:center; gap:12px;">
          <span style="font-size:11px; color:#4a5568; white-space:nowrap; font-weight:600; letter-spacing:0.05em;">MEDICAL HISTORY</span>
          <div style="flex:1; height:1px; background:rgba(255,255,255,0.04);"></div>
        </div>

        <div class="field">
          <label for="prev_diagnoses">What doctors have told you</label>
          <textarea id="prev_diagnoses" rows="2" placeholder="e.g. anxiety, fibromyalgia, IBS, nothing found..." oninput="updateProgress()"></textarea>
        </div>
        <div class="field">
          <label for="tests_done">Tests already done &amp; results</label>
          <textarea id="tests_done" rows="2" placeholder="e.g. full blood count (normal), MRI brain (clear), thyroid (normal)..." oninput="updateProgress()"></textarea>
        </div>
        <div class="field">
          <label for="family_history">Family medical history</label>
          <input type="text" id="family_history" placeholder="e.g. mother has autoimmune disease, father had early heart problems..." oninput="updateProgress()" />
        </div>
        <div class="field">
          <label for="notes">Any other important notes?</label>
          <textarea id="notes" rows="2" placeholder="e.g. symptoms worse after exercise, began after a viral illness, only at night..." oninput="updateProgress()"></textarea>
        </div>

        <!-- File Upload -->
        <div style="border-top:1px solid rgba(255,255,255,0.06); margin:20px 0; display:flex; align-items:center; gap:12px;">
          <span style="font-size:11px; color:#4a5568; white-space:nowrap; font-weight:600; letter-spacing:0.05em;">UPLOAD RECORDS</span>
          <div style="flex:1; height:1px; background:rgba(255,255,255,0.04);"></div>
          <span style="font-size:10px; color:#6c8ef5; white-space:nowrap; font-weight:700; background:rgba(108,142,245,0.1); padding:2px 8px; border-radius:10px;">GEMMA 4 VISION</span>
        </div>

        <p style="font-size:12px; color:#718096; margin-bottom:12px; line-height:1.6;">
          Can't describe everything in words? Upload a photo or PDF of your medical records — lab reports, doctor letters, test results, prescriptions. Gemma 4 will read them directly.
        </p>

        <input type="file" id="file-input" accept="image/*,.pdf" style="display:none;" onchange="handleFileSelect(this.files[0])" />

        <div id="upload-zone" class="upload-zone" onclick="document.getElementById('file-input').click()"
             ondrop="handleDrop(event)" ondragover="handleDragOver(event)" ondragleave="handleDragLeave(event)">
          <div style="font-size:32px; margin-bottom:10px;">📁</div>
          <p style="font-size:14px; font-weight:600; color:#94a3b8; margin-bottom:4px;">Upload a medical record</p>
          <p style="font-size:12px; color:#4a5568; line-height:1.6;">Lab reports · Doctor letters · Discharge summaries · Test results</p>
          <p style="font-size:11px; color:#6c8ef5; margin-top:10px; font-weight:600;">JPG · PNG · PDF supported · Max 10MB</p>
          <div style="margin-top:14px; display:inline-block; padding:7px 18px; background:rgba(108,142,245,0.12); border:1px solid rgba(108,142,245,0.25); border-radius:8px; font-size:13px; color:#818cf8; font-weight:600;">
            Browse or drag &amp; drop
          </div>
        </div>

        <div id="file-preview" style="display:none;" class="file-preview">
          <span id="file-icon" style="font-size:28px;">📎</span>
          <div style="flex:1; min-width:0;">
            <p id="file-name" style="font-size:13px; font-weight:600; color:#34d399; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;"></p>
            <p id="file-meta" style="font-size:11px; color:#718096;"></p>
          </div>
          <button onclick="removeFile()" style="background:none; border:none; color:#718096; cursor:pointer; font-size:18px; padding:4px; transition:color 0.2s;" onmouseover="this.style.color='#f87171'" onmouseout="this.style.color='#718096'">✕</button>
        </div>

        <!-- Submit -->
        <button id="submit-btn" class="submit-btn" onclick="handleSubmit()"
          style="width:100%; margin-top:8px; padding:15px 24px; background:linear-gradient(135deg,#6c8ef5 0%,#a78bfa 100%); border:none; border-radius:12px; color:#fff; font-size:15px; font-weight:700; cursor:pointer; transition:all 0.2s; letter-spacing:0.02em;" disabled>
          🔍 Analyze My History
        </button>
      </div>

      <!-- RIGHT: Results -->
      <div id="results-panel">
        <!-- Empty state -->
        <div id="empty-state" style="background:rgba(255,255,255,0.02); border:1px dashed rgba(255,255,255,0.1); border-radius:20px; padding:48px; text-align:center;">
          <div style="font-size:52px; margin-bottom:20px;">🔬</div>
          <h3 style="font-size:18px; font-weight:700; color:#94a3b8; margin-bottom:10px;">Your analysis will appear here</h3>
          <p style="font-size:14px; color:#4a5568; max-width:320px; margin:0 auto; line-height:1.7;">Gemma 4 will cross-reference your symptoms against rare disease patterns — or read your uploaded medical records directly using its vision capability.</p>
          <div style="margin-top:32px; display:flex; flex-direction:column; gap:10px; text-align:left; max-width:280px; margin:32px auto 0;">
            <div style="display:flex; align-items:center; gap:10px; font-size:13px; color:#718096;"><div style="width:6px; height:6px; border-radius:50%; background:#6c8ef5; flex-shrink:0;"></div>Top conditions to investigate</div>
            <div style="display:flex; align-items:center; gap:10px; font-size:13px; color:#718096;"><div style="width:6px; height:6px; border-radius:50%; background:#6c8ef5; flex-shrink:0;"></div>Exact tests to request</div>
            <div style="display:flex; align-items:center; gap:10px; font-size:13px; color:#718096;"><div style="width:6px; height:6px; border-radius:50%; background:#6c8ef5; flex-shrink:0;"></div>Right specialist to see</div>
            <div style="display:flex; align-items:center; gap:10px; font-size:13px; color:#718096;"><div style="width:6px; height:6px; border-radius:50%; background:#6c8ef5; flex-shrink:0;"></div>Questions to ask your doctor</div>
          </div>
        </div>

        <!-- Loading state -->
        <div id="loading-state" style="display:none; background:rgba(255,255,255,0.02); border:1px solid rgba(255,255,255,0.07); border-radius:20px; padding:40px;">
          <div style="text-align:center; margin-bottom:40px;">
            <div style="width:56px; height:56px; border:3px solid rgba(108,142,245,0.2); border-top:3px solid #6c8ef5; border-radius:50%; animation:spin 0.8s linear infinite; margin:0 auto 20px;"></div>
            <h3 style="font-size:18px; font-weight:700; color:#e2e8f0; margin-bottom:8px;">Gemma 4 is working</h3>
            <p style="color:#718096; font-size:14px;">This takes 15–30 seconds</p>
          </div>
          <div id="loading-steps" style="display:flex; flex-direction:column; gap:12px;"></div>
        </div>

        <!-- Error state -->
        <div id="error-state" style="display:none; background:rgba(248,113,113,0.06); border:1px solid rgba(248,113,113,0.2); border-radius:20px; padding:28px;">
          <div style="font-size:28px; margin-bottom:12px;">⚠️</div>
          <h3 style="color:#f87171; font-weight:700; margin-bottom:8px;">Something went wrong</h3>
          <p id="error-msg" style="color:#fca5a5; font-size:14px; line-height:1.6;"></p>
          <button onclick="resetError()" style="margin-top:16px; padding:8px 18px; background:rgba(248,113,113,0.15); border:1px solid rgba(248,113,113,0.3); border-radius:8px; color:#f87171; font-size:13px; cursor:pointer; font-weight:600;">Try Again</button>
        </div>

        <!-- Results -->
        <div id="result-state" style="display:none;">
          <div style="display:flex; align-items:center; justify-content:space-between; margin-bottom:16px;">
            <div>
              <h2 style="font-size:18px; font-weight:700; color:#e2e8f0;">💡 Your Analysis</h2>
              <div style="display:flex; gap:8px; margin-top:4px; flex-wrap:wrap; align-items:center;">
                <p style="font-size:13px; color:#718096;">Generated by Gemma 4 · Not a medical diagnosis</p>
                <span id="vision-badge" style="display:none; font-size:11px; font-weight:700; color:#a78bfa; background:rgba(167,139,250,0.1); border:1px solid rgba(167,139,250,0.25); border-radius:10px; padding:1px 8px;">👁 Vision used</span>
              </div>
            </div>
            <button id="copy-btn" class="copy-btn" onclick="copyResult()" style="padding:8px 14px; background:rgba(255,255,255,0.05); border:1px solid rgba(255,255,255,0.1); border-radius:8px; color:#94a3b8; font-size:12px; font-weight:600; cursor:pointer; transition:all 0.2s; white-space:nowrap;">Copy Results</button>
          </div>
          <div id="result-sections" style="display:flex; flex-direction:column; gap:14px;"></div>
          <div id="disease-tags" style="margin-top:20px; display:none;">
            <p style="font-size:11px; color:#4a5568; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; margin-bottom:10px;">Diseases cross-referenced by Gemma 4</p>
            <div id="disease-list" style="display:flex; flex-wrap:wrap; gap:8px;"></div>
          </div>
          <div style="margin-top:20px; padding:14px 18px; background:rgba(251,191,36,0.05); border:1px solid rgba(251,191,36,0.15); border-radius:12px; font-size:12px; color:#d97706; line-height:1.6;">
            <strong>Important:</strong> This analysis is for educational purposes to help you have better conversations with your healthcare providers. Always consult a qualified medical professional for diagnosis and treatment.
          </div>
        </div>
      </div>
    </div>
  </div>

  <!-- FOOTER -->
  <div style="border-top:1px solid rgba(255,255,255,0.06); padding:28px 24px; text-align:center;">
    <div style="display:flex; align-items:center; justify-content:center; gap:16px; flex-wrap:wrap;">
      <span style="background:rgba(108,142,245,0.1); border:1px solid rgba(108,142,245,0.2); border-radius:20px; padding:4px 14px; font-size:12px; color:#6c8ef5; font-weight:600;">Gemma 4 Hackathon</span>
      <span style="color:#2d3748; font-size:13px;">·</span>
      <span style="font-size:13px; color:#4a5568;">Health &amp; Sciences Track</span>
      <span style="color:#2d3748; font-size:13px;">·</span>
      <span style="font-size:13px; color:#4a5568;">Powered by Google Gemma 4</span>
    </div>
  </div>

</div><!-- /app -->

<script>
// ── State ───────────────────────────────────────────────────────────────────
let uploadedFile = null;
let rawResult    = "";
let loadingTimer = null;

const LOADING_STEPS = [
  "Reading your symptom history",
  "Cross-referencing rare disease database",
  "Identifying diagnostic patterns",
  "Generating your personalized plan"
];

const SECTION_DEFS = [
  { key:"conditions", emoji:"🔍", label:"Conditions to Investigate",  color:"#a78bfa", bg:"rgba(167,139,250,0.08)", border:"rgba(167,139,250,0.2)",
    regex:/(?:#{0,3}\\s*1\\.?\\s*(?:🔍)?\\s*(?:TOP\\s*)?[\\d–-]*\\s*CONDITIONS?(?:\\s+WORTH\\s+INVESTIGATING)?[\\s\\S]*?)(?=(?:#{0,3}\\s*2\\.|$))/i },
  { key:"tests",      emoji:"🧪", label:"Specific Tests to Request",  color:"#6c8ef5", bg:"rgba(108,142,245,0.08)", border:"rgba(108,142,245,0.2)",
    regex:/(?:#{0,3}\\s*2\\.?\\s*(?:🧪)?\\s*SPECIFIC\\s*TESTS?[\\s\\S]*?)(?=(?:#{0,3}\\s*3\\.|$))/i },
  { key:"specialist", emoji:"👨‍⚕️", label:"Specialist to See Next",    color:"#34d399", bg:"rgba(52,211,153,0.08)",  border:"rgba(52,211,153,0.2)",
    regex:/(?:#{0,3}\\s*3\\.?\\s*(?:👨)?(?:‍⚕️)?\\s*(?:TYPE OF\\s*)?SPECIALIST[\\s\\S]*?)(?=(?:#{0,3}\\s*4\\.|$))/i },
  { key:"questions",  emoji:"❓", label:"Questions for Your Doctor",   color:"#fbbf24", bg:"rgba(251,191,36,0.08)",  border:"rgba(251,191,36,0.2)",
    regex:/(?:#{0,3}\\s*4\\.?\\s*(?:❓)?\\s*(?:5\\s*)?KEY\\s*QUESTIONS?[\\s\\S]*?)(?=(?:#{0,3}\\s*5\\.|$))/i },
  { key:"patterns",   emoji:"📋", label:"Patterns in Your History",   color:"#f87171", bg:"rgba(248,113,113,0.08)", border:"rgba(248,113,113,0.2)",
    regex:/(?:#{0,3}\\s*5\\.?\\s*(?:📋)?\\s*(?:IMPORTANT\\s*)?PATTERNS?[\\s\\S]*?)$/i },
];

// ── Output parsing ──────────────────────────────────────────────────────────
function cleanGemmaOutput(text) {
  let cleaned = text;
  const markerRegex = /\\*{0,3}\\s*\\(?\\s*start\\s+of\\s+response\\s*\\)?\\s*\\*{0,3}/i;
  const markerMatch = cleaned.match(markerRegex);
  if (markerMatch) {
    const markerEnd = cleaned.indexOf(markerMatch[0]) + markerMatch[0].length;
    const afterMarker = cleaned.slice(markerEnd).trim();
    if (afterMarker.length > 30) cleaned = afterMarker;
  }
  cleaned = cleaned.replace(/\\*{0,3}\\s*\\(?self[- ]correction[^)]*\\)?[\\s\\S]*?(?=\\*{2,3}|\\n\\n1\\.)/i, "").trim();
  cleaned = cleaned.replace(/^\\*{3,}\\s*$/gm, "").trim();
  cleaned = cleaned.replace(/\\*\\*(.*?)\\*\\*/g, "$1");
  if (cleaned.length < 30) {
    const m = text.match(/1\\.\\s*(?:🔍)?[\\s\\S]*/);
    if (m) cleaned = m[0].replace(/\\*\\*(.*?)\\*\\*/g, "$1");
  }
  return cleaned.trim() || text;
}

function parseResult(rawText) {
  const text = cleanGemmaOutput(rawText);
  const sections = [];
  let matched = false;
  SECTION_DEFS.forEach(p => {
    const m = text.match(p.regex);
    if (m && m[0].trim().length > 10) {
      sections.push({ ...p, content: m[0].trim() });
      matched = true;
    }
  });
  if (!matched) {
    sections.push({ key:"raw", emoji:"💬", label:"Analysis", color:"#6c8ef5",
      bg:"rgba(108,142,245,0.08)", border:"rgba(108,142,245,0.2)", content: text });
  }
  return sections;
}

// ── Progress bar ────────────────────────────────────────────────────────────
function updateProgress() {
  const ids = ["age","duration","symptoms","prev_diagnoses","tests_done","family_history","notes"];
  let filled = ids.filter(id => document.getElementById(id).value.trim()).length;
  if (uploadedFile) filled++;
  const pct = Math.round((filled / (ids.length + 1)) * 100);
  document.getElementById("progress-bar").style.width = pct + "%";
  document.getElementById("progress-label").textContent = pct + "%";
}

function updateSubmitBtn() {
  const sym = document.getElementById("symptoms").value.trim();
  const btn = document.getElementById("submit-btn");
  btn.disabled = !sym && !uploadedFile;
  btn.style.opacity = btn.disabled ? "0.5" : "1";
  btn.style.cursor  = btn.disabled ? "not-allowed" : "pointer";
  btn.textContent   = uploadedFile ? "🔍  Analyze My History + Records" : "🔍  Analyze My History";
}

// ── File upload ─────────────────────────────────────────────────────────────
function handleFileSelect(file) {
  if (!file) return;
  uploadedFile = file;
  showFilePreview(file);
  updateProgress();
  updateSubmitBtn();
}

function showFilePreview(file) {
  const isImg = file.type.startsWith("image/");
  const isPdf = file.type === "application/pdf";
  const icon  = isImg ? "🖼️" : isPdf ? "📄" : "📎";
  const meta  = `${formatSize(file.size)} · ${isImg ? "Gemma 4 will read this image directly 👁" : "Text will be extracted from PDF 📝"}`;
  document.getElementById("upload-zone").style.display = "none";
  document.getElementById("file-preview").style.display = "flex";
  document.getElementById("file-icon").textContent = icon;
  document.getElementById("file-name").textContent = file.name;
  document.getElementById("file-meta").textContent = meta;
}

function removeFile() {
  uploadedFile = null;
  document.getElementById("file-input").value = "";
  document.getElementById("upload-zone").style.display = "";
  document.getElementById("file-preview").style.display = "none";
  updateProgress();
  updateSubmitBtn();
}

function formatSize(bytes) {
  return bytes < 1024 * 1024 ? `${(bytes/1024).toFixed(0)} KB` : `${(bytes/(1024*1024)).toFixed(1)} MB`;
}

function handleDrop(e) {
  e.preventDefault();
  document.getElementById("upload-zone").classList.remove("drag-over");
  const file = e.dataTransfer.files[0];
  if (file) handleFileSelect(file);
}
function handleDragOver(e) { e.preventDefault(); document.getElementById("upload-zone").classList.add("drag-over"); }
function handleDragLeave()  { document.getElementById("upload-zone").classList.remove("drag-over"); }

// ── UI state helpers ────────────────────────────────────────────────────────
function showPanel(id) {
  ["empty-state","loading-state","error-state","result-state"].forEach(p => {
    document.getElementById(p).style.display = p === id ? "" : "none";
  });
}

function startLoadingAnimation() {
  let step = 0;
  const container = document.getElementById("loading-steps");
  function render() {
    container.innerHTML = LOADING_STEPS.map((s, i) => `
      <div style="display:flex; align-items:center; gap:14px; padding:12px 16px; border-radius:12px;
        background:${i===step?"rgba(108,142,245,0.1)":"rgba(255,255,255,0.02)"};
        border:1px solid ${i===step?"rgba(108,142,245,0.3)":"rgba(255,255,255,0.05)"};
        transition:all 0.4s;">
        <div style="width:28px; height:28px; border-radius:50%; display:flex; align-items:center; justify-content:center;
          background:${i<step?"#34d399":i===step?"rgba(108,142,245,0.3)":"rgba(255,255,255,0.05)"};
          font-size:12px; font-weight:700;
          color:${i<step?"#fff":i===step?"#6c8ef5":"#4a5568"}; flex-shrink:0; transition:all 0.4s;">
          ${i < step ? "✓" : i+1}
        </div>
        <span style="font-size:14px; font-weight:${i===step?600:400};
          color:${i===step?"#e2e8f0":i<step?"#34d399":"#4a5568"}; transition:all 0.4s;">
          ${s}${i===step?"..." : ""}
        </span>
      </div>`).join("");
  }
  render();
  loadingTimer = setInterval(() => { if (step < LOADING_STEPS.length - 1) { step++; render(); } }, 2000);
}

function stopLoadingAnimation() {
  clearInterval(loadingTimer);
}

// ── Submit ──────────────────────────────────────────────────────────────────
async function handleSubmit() {
  const symptoms = document.getElementById("symptoms").value.trim();
  if (!symptoms && !uploadedFile) return;

  showPanel("loading-state");
  startLoadingAnimation();
  document.getElementById("submit-btn").disabled = true;

  try {
    const fd = new FormData();
    fd.append("age",            document.getElementById("age").value);
    fd.append("sex",            document.getElementById("sex").value);
    fd.append("duration",       document.getElementById("duration").value);
    fd.append("symptoms",       symptoms);
    fd.append("prev_diagnoses", document.getElementById("prev_diagnoses").value);
    fd.append("tests_done",     document.getElementById("tests_done").value);
    fd.append("family_history", document.getElementById("family_history").value);
    fd.append("notes",          document.getElementById("notes").value);
    if (uploadedFile) fd.append("record_file", uploadedFile);

    const res  = await fetch("/diagnose", { method:"POST", body: fd });
    const data = await res.json();

    stopLoadingAnimation();

    if (!res.ok || data.error) {
      document.getElementById("error-msg").textContent = data.error || "Something went wrong.";
      showPanel("error-state");
    } else {
      rawResult = data.result || "";
      renderResults(data);
      showPanel("result-state");
    }
  } catch(err) {
    stopLoadingAnimation();
    document.getElementById("error-msg").textContent = "Network error — make sure the Colab server is running.";
    showPanel("error-state");
  } finally {
    document.getElementById("submit-btn").disabled = false;
    updateSubmitBtn();
  }
}

// ── Render results ──────────────────────────────────────────────────────────
function renderResults(data) {
  const sections = parseResult(data.result || "");
  const container = document.getElementById("result-sections");
  container.innerHTML = sections.map((sec, i) => `
    <div class="result-section" style="background:${sec.bg}; border:1px solid ${sec.border}; border-radius:16px; padding:20px; animation-delay:${i*0.08}s;">
      <div style="display:flex; align-items:center; gap:8px; margin-bottom:12px;">
        <span style="font-size:16px;">${sec.emoji}</span>
        <span style="font-size:12px; font-weight:700; color:${sec.color}; text-transform:uppercase; letter-spacing:0.07em;">${sec.label}</span>
      </div>
      <div style="font-size:14px; color:#cbd5e1; line-height:1.8; white-space:pre-wrap;">${escapeHTML(sec.content)}</div>
    </div>`).join("");

  // Vision badge
  document.getElementById("vision-badge").style.display = data.used_vision ? "inline" : "none";

  // Disease tags with semantic relevance scores
  const diseases = data.diseases_checked || [];
  if (diseases.length > 0) {
    document.getElementById("disease-tags").style.display = "";
    document.getElementById("disease-list").innerHTML = diseases.map(d => {
      const name      = typeof d === "object" ? d.name      : d;
      const relevance = typeof d === "object" ? d.relevance : null;
      const badge     = relevance != null
        ? `<span style="background:rgba(108,142,245,0.2);border-radius:10px;padding:1px 6px;font-size:10px;font-weight:700;color:#6c8ef5;margin-left:4px;">${relevance}%</span>`
        : "";
      return `<span style="display:inline-flex;align-items:center;background:rgba(108,142,245,0.08);color:#818cf8;border:1px solid rgba(108,142,245,0.2);border-radius:20px;padding:4px 12px;font-size:12px;font-weight:500;">${escapeHTML(name)}${badge}</span>`;
    }).join("");
  } else {
    document.getElementById("disease-tags").style.display = "none";
  }
}

function escapeHTML(str) {
  return str.replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");
}

function copyResult() {
  if (!rawResult) return;
  navigator.clipboard.writeText(rawResult).then(() => {
    const btn = document.getElementById("copy-btn");
    btn.textContent = "✓ Copied!";
    btn.style.color = "#34d399";
    btn.style.borderColor = "rgba(52,211,153,0.3)";
    setTimeout(() => {
      btn.textContent = "Copy Results";
      btn.style.color = "#94a3b8";
      btn.style.borderColor = "rgba(255,255,255,0.1)";
    }, 2000);
  });
}

function resetError() {
  showPanel("empty-state");
}
</script>
</body>
</html>
"""

with open("ui.html", "w", encoding="utf-8") as f:
    f.write(UI_HTML_CONTENT)

print("✅ UI file written automatically — no upload needed!")
print(f"   File size: {len(UI_HTML_CONTENT):,} characters")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — FastAPI Server + Cloudflare Tunnel
# This serves the full UI and handles all /diagnose requests
# ════════════════════════════════════════════════════════════════

!pip install fastapi uvicorn nest-asyncio python-multipart pdfminer.six -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import nest_asyncio, uvicorn, threading, subprocess, time, re, base64, io, os
from fastapi import FastAPI, Form, File, UploadFile
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from typing import Optional

nest_asyncio.apply()
os.system("fuser -k 8000/tcp 2>/dev/null")
time.sleep(1)

# ── Read UI HTML ──────────────────────────────────────────────
with open("ui.html", "r") as f:
    UI_HTML = f.read()

app = FastAPI(title="Diagnostic Odyssey Ender", version="1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

@app.get("/")
def serve_ui():
    return HTMLResponse(content=UI_HTML)

@app.get("/health")
def health():
    return {"status": "ok", "model": "gemma-4-E4B-it", "diseases": len(DISEASE_DB)}

@app.post("/diagnose")
async def diagnose(
    age:            str = Form(""),
    sex:            str = Form(""),
    duration:       str = Form(""),
    symptoms:       str = Form(""),
    prev_diagnoses: str = Form(""),
    tests_done:     str = Form(""),
    family_history: str = Form(""),
    notes:          str = Form(""),
    record_file:    Optional[UploadFile] = File(None)
):
    image_base64 = None
    image_mime   = None
    pdf_text     = ""

    # ── Handle file upload ────────────────────────────────────────
    if record_file and record_file.filename:
        content = await record_file.read()
        mime    = record_file.content_type or ""

        if mime.startswith("image/"):
            image_base64 = base64.b64encode(content).decode("utf-8")
            image_mime   = mime
        elif mime == "application/pdf":
            try:
                import pdfminer.high_level
                pdf_text = pdfminer.high_level.extract_text(io.BytesIO(content))[:4000]
            except:
                pdf_text = ""

    patient_data = {
        "age":            age,
        "sex":            sex,
        "duration":       duration,
        "symptoms":       symptoms,
        "prev_diagnoses": prev_diagnoses,
        "tests_done":     tests_done,
        "family_history": family_history,
        "notes":          f"{notes} {pdf_text}".strip(),
        "image_base64":   image_base64,
        "image_mime":     image_mime,
    }

    # ── Run Gemma 4 + Semantic RAG ────────────────────────────────
    result   = run_diagnostic(patient_data)
    search_t = f"{symptoms} {prev_diagnoses} {notes} {pdf_text}".strip()
    diseases = retrieve_diseases(search_t or "rare disease chronic", top_k=5)

    return {
        "result":     result,
        "diseases_checked": [
            {"name": d["name"], "relevance": d["relevance_score"]}
            for d in diseases
        ],
        "used_vision": image_base64 is not None,
        "used_pdf":    bool(pdf_text),
    }

# ── Start server ──────────────────────────────────────────────
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print("✅ Server running on port 8000")

# ── Cloudflare Tunnel ─────────────────────────────────────────
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print("⏳ Starting Cloudflare tunnel...")
time.sleep(3)

found = False
for _ in range(40):
    line = tunnel.stdout.readline().decode("utf-8", errors="ignore")
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f"""
╔══════════════════════════════════════════════════════════╗
  ✅  THE DIAGNOSTIC ODYSSEY ENDER IS LIVE!

  🌐  {url}

  Open this link to use the full app.
  Share it with anyone — no login needed.
╚══════════════════════════════════════════════════════════╝
""")
        found = True
        break

if not found:
    print("❌ Tunnel URL not found. Re-run this cell once.")